In [ ]:
!pip install transformers accelerate bitsandbytes sentence-transformers faiss-cpu datasets -q
!pip install torch==2.4.0 torchaudio==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121 -q

from huggingface_hub import login
login()
print("✅ Login HF OK")

In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import torch

print("Chargement Whisper fine-tuné (checkpoint-1000)...")

# Processor depuis le modèle de base (contient le tokenizer)
whisper_processor = WhisperProcessor.from_pretrained("openai/whisper-small")

# Poids fine-tunés depuis ton checkpoint
whisper_model = WhisperForConditionalGeneration.from_pretrained(
    "abdeldjalil-smaili/whisper-algerian-medical-final",
    subfolder="checkpoint-1000",
    token=login()
).to("cuda")
whisper_model.eval()
print("✅ Whisper chargé")

def transcribe(audio_path):
    import librosa
    audio, sr = librosa.load(audio_path, sr=16000)
    inputs = whisper_processor(
        audio, sampling_rate=16000, return_tensors="pt"
    ).input_features.to("cuda")
    with torch.no_grad():
        predicted_ids = whisper_model.generate(inputs, language="ar")
    return whisper_processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

print("✅ Fonction transcribe prête")

In [ ]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from datasets import load_dataset

print("Chargement dataset et construction FAISS...")
embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

dataset = load_dataset("abdeldjalil-smaili/algerian-medical-llm-v3", token=HF_TOKEN, split="train")

def get_question(s):
    try:
        return s['text'].split("### Input:")[1].split("### Response:")[0].strip()
    except:
        return s['text'][:200]

def get_answer(s):
    try:
        return s['text'].split("### Response:")[1].strip()
    except:
        return ""

questions = [get_question(s) for s in dataset]
answers   = [get_answer(s)   for s in dataset]

embeddings = embedder.encode(questions, batch_size=256, show_progress_bar=True, convert_to_numpy=True)
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings.astype(np.float32))
print(f"✅ FAISS prêt : {index.ntotal} documents")

def get_rag_context(question, k=3, threshold=8.0):
    q_emb = embedder.encode([question], convert_to_numpy=True)
    distances, indices = index.search(q_emb.astype(np.float32), k)
    contexts = []
    for dist, idx in zip(distances[0], indices[0]):
        if dist < threshold:
            contexts.append(f"Q: {questions[idx]}\nR: {answers[idx]}")
    return "\n\n".join(contexts)

In [ ]:
!pip uninstall torchcodec -y -q
!pip install sentence-transformers -q

In [ ]:
def generate_response(question, model, tokenizer, model_name="llama"):
    context = get_rag_context(question)

    if context:
        content = f"""أنت مساعد طبي يتكلم الدارجة الجزائرية فقط.
استخدم المعلومات التالية للإجابة:

{context}

السؤال: {question}
أجب مباشرة وبإيجاز بالدارجة الجزائرية فقط."""
    else:
        content = f"""أنت مساعد طبي يتكلم الدارجة الجزائرية فقط.
أجب مباشرة وبإيجاز:

{question}"""

    messages = [{"role": "user", "content": content}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.3
        )
    input_len = inputs["input_ids"].shape[1]
    return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()

In [ ]:
import os
print(os.path.exists('ref_voice.wav'))

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload new_test.mp4
!ffmpeg -i new_test.mp4 -ar 24000 -ac 1 ref_voice.wav -y

In [ ]:
os.makedirs('./outputs', exist_ok=True)

In [ ]:
import soundfile as sf
import IPython.display as ipd

ref_audio = 'ref_voice.wav'        # ton audio de référence uploadé avant
ref_text  = "نيفي راهو يسيل" # ← transcription de ton audio

def synthesize(text, filename="output.wav"):
    !habibi-tts_infer-cli \
        --ref_audio "{ref_audio}" \
        --ref_text "{ref_text}" \
        --gen_text "{text}" \
        --dialect ALG \
        --output_dir "./outputs"
    path = f"./outputs/infer_cli_example.wav"
    ipd.display(ipd.Audio(path, autoplay=True))
    return path

In [ ]:
!pip install tokenizers -q

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# ── AtlasChat 9B (zero-shot) ──
print("Chargement AtlasChat 9B...")
atlas_tokenizer = AutoTokenizer.from_pretrained("MBZUAI-Paris/Atlas-Chat-9B", token=HF_TOKEN)
atlas_model = AutoModelForCausalLM.from_pretrained(
    "MBZUAI-Paris/Atlas-Chat-9B",
    quantization_config=bnb_config,
    device_map="auto",
    token=login()
)
atlas_model.eval()
print("✅ AtlasChat chargé")


In [ ]:
print("Chargement LLaMA fine-tuné...")
llama_tokenizer = AutoTokenizer.from_pretrained(
    "meta-llama/Llama-3.2-3B-Instruct",
    token= login()
)
llama_model = AutoModelForCausalLM.from_pretrained(
    "abdeldjalil-smaili/llama3b-algerian-medical",
    quantization_config=bnb_config,
    device_map="auto",
    token=login()
)
llama_model.eval()
print("✅ LLaMA fine-tuné chargé")

In [ ]:
!pip install transformers==4.45.0 -q

In [ ]:
!pip install habibi-tts -q
!pip install torch==2.4.0 torchaudio==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121 -q

In [ ]:
import soundfile as sf
import IPython.display as ipd
import os

ref_audio = 'ref_voice.wav'
ref_text  = "نيفي راهو يسيل"

os.makedirs('./outputs', exist_ok=True)

def synthesize(text, filename="output.wav"):
    os.system(f'habibi-tts_infer-cli --ref_audio "{ref_audio}" --ref_text "{ref_text}" --gen_text "{text}" --dialect ALG --output_dir "./outputs"')
    path = './outputs/infer_cli_example.wav'
    if os.path.exists(path):
        ipd.display(ipd.Audio(path, autoplay=False))
    return path

In [ ]:
run_pipeline(text_input="كيفاش نحمي روحي من كوفيد؟")

In [ ]:
def generate_response(question, model, tokenizer, model_name="llama"):
    context = get_rag_context(question)

    if context:
        content = f"""أنت مساعد طبي يتكلم الدارجة الجزائرية فقط.
استخدم هاد المعلومات باش تجاوب:

{context}

السؤال: {question}
أجب بجملة واحدة أو جملتين فقط بالدارجة الجزائرية. ما تطولش."""
    else:
        content = f"""أنت مساعد طبي يتكلم الدارجة الجزائرية فقط.
أجب بجملة واحدة أو جملتين فقط. ما تطولش.

{question}"""

    messages = [{"role": "user", "content": content}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,   # réduit à 80 tokens
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.3
        )
    input_len = inputs["input_ids"].shape[1]
    return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()

In [ ]:
def run_pipeline(audio_path=None, text_input=None):
    # Étape 1 : ASR
    if audio_path:
        print("🎤 Transcription Whisper...")
        question = transcribe(audio_path)
    else:
        question = text_input
    print(f"Question : {question}\n")

    # Étape 2 : LLM x2
    print("🧠 AtlasChat (zero-shot)...")
    response_atlas = generate_response(question, atlas_model, atlas_tokenizer, "atlas")
    print(f"AtlasChat : {response_atlas}\n")

    print("🧠 LLaMA fine-tuné...")
    response_llama = generate_response(question, llama_model, llama_tokenizer, "llama")
    print(f"LLaMA     : {response_llama}\n")

    # Étape 3 : TTS sur la réponse du LLaMA fine-tuné uniquement
    print("🔊 Synthèse vocale...")
    synthesize(response_llama)

run_pipeline(text_input="وش هوما أعراض السكري؟")

In [ ]:
def run_pipeline(audio_path=None, text_input=None):
    # Étape 1 : ASR
    if audio_path:
        print("🎤 Transcription Whisper...")
        question = transcribe(audio_path)
    else:
        question = text_input
    print(f"Question : {question}\n")

    # Étape 2 : LLM x2
    print("🧠 AtlasChat (zero-shot)...")
    response_atlas = generate_response(question, atlas_model, atlas_tokenizer, "atlas")
    print(f"AtlasChat : {response_atlas}\n")

    print("🧠 LLaMA fine-tuné...")
    response_llama = generate_response(question, llama_model, llama_tokenizer, "llama")
    print(f"LLaMA     : {response_llama}\n")

    # Étape 3 : TTS les deux réponses
    print("🔊 Synthèse LLaMA fine-tuné...")
    os.system(f'habibi-tts_infer-cli --ref_audio "{ref_audio}" --ref_text "{ref_text}" --gen_text "{response_llama}" --dialect ALG --output_dir "./outputs_llama"')
    ipd.display(ipd.Audio('./outputs_llama/infer_cli_example.wav', autoplay=False))

    print("🔊 Synthèse AtlasChat (zero-shot)...")
    os.system(f'habibi-tts_infer-cli --ref_audio "{ref_audio}" --ref_text "{ref_text}" --gen_text "{response_atlas}" --dialect ALG --output_dir "./outputs_atlas"')
    ipd.display(ipd.Audio('./outputs_atlas/infer_cli_example.wav', autoplay=False))

os.makedirs('./outputs_llama', exist_ok=True)
os.makedirs('./outputs_atlas', exist_ok=True)

run_pipeline(text_input="وش هوما أعراض السكري؟")

In [ ]:
from IPython.display import Javascript, display
from google.colab import output
import base64
import numpy as np
import soundfile as sf

# Javascript pour enregistrer depuis le micro
record_js = """
async function recordAudio() {
    const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
    const recorder = new MediaRecorder(stream);
    const chunks = [];

    recorder.ondataavailable = e => chunks.push(e.data);
    recorder.start();

    document.getElementById('status').textContent = '🔴 Enregistrement en cours... (5 secondes)';

    await new Promise(r => setTimeout(r, 5000));  // 5 secondes
    recorder.stop();

    await new Promise(r => recorder.onstop = r);
    stream.getTracks().forEach(t => t.stop());

    const blob = new Blob(chunks, { type: 'audio/webm' });
    const reader = new FileReader();
    reader.readAsDataURL(blob);
    await new Promise(r => reader.onload = r);

    const base64 = reader.result.split(',')[1];
    document.getElementById('status').textContent = '✅ Enregistrement terminé !';

    google.colab.kernel.invokeFunction('save_audio', [base64], {});
}

recordAudio();
"""

display(Javascript("""
    const div = document.createElement('div');
    div.id = 'status';
    div.textContent = 'Initialisation...';
    document.body.appendChild(div);
"""))

# Callback Python pour sauvegarder l'audio
def save_audio(base64_audio):
    audio_bytes = base64.b64decode(base64_audio)
    with open('live_input.webm', 'wb') as f:
        f.write(audio_bytes)
    # Convertir webm en wav
    import subprocess
    subprocess.run(['ffmpeg', '-i', 'live_input.webm', '-ar', '16000', '-ac', '1', 'live_input.wav', '-y', '-q:a', '0'])
    print("✅ Audio sauvegardé → live_input.wav")
    # Lancer le pipeline
    run_pipeline(audio_path='live_input.wav')

output.register_callback('save_audio', save_audio)
display(Javascript(record_js))

In [ ]:
from IPython.display import Javascript, display
from google.colab import output
import base64, subprocess

RECORD_JS = """
(async () => {
  const stream = await navigator.mediaDevices.getUserMedia({audio: true});
  const rec = new MediaRecorder(stream);
  const chunks = [];
  rec.ondataavailable = e => chunks.push(e.data);
  rec.start();
  document.body.innerHTML += '<h3 id="msg">🔴 Parle maintenant... (5 sec)</h3>';
  await new Promise(r => setTimeout(r, 5000));
  rec.stop();
  await new Promise(r => rec.onstop = r);
  stream.getTracks().forEach(t => t.stop());
  const blob = new Blob(chunks);
  const buf = await blob.arrayBuffer();
  const b64 = btoa(String.fromCharCode(...new Uint8Array(buf)));
  document.getElementById('msg').textContent = '✅ Terminé !';
  await google.colab.kernel.invokeFunction('recv_audio', [b64], {});
})();
"""

def recv_audio(b64):
    with open('live_input.webm', 'wb') as f:
        f.write(base64.b64decode(b64))
    subprocess.run(['ffmpeg', '-i', 'live_input.webm', '-ar', '16000', '-ac', '1', 'live_input.wav', '-y'],
                   capture_output=True)
    print("✅ Audio reçu, lancement du pipeline...")
    run_pipeline(audio_path='live_input.wav')

output.register_callback('recv_audio', recv_audio)
display(Javascript(RECORD_JS))

In [ ]:
from IPython.display import Javascript, display
from google.colab import output
import base64, subprocess

RECORD_JS = """
(async () => {
  const stream = await navigator.mediaDevices.getUserMedia({audio: true});
  const rec = new MediaRecorder(stream);
  const chunks = [];
  rec.ondataavailable = e => chunks.push(e.data);
  rec.start();
  document.body.innerHTML += '<h3 id="msg">🔴 Parle maintenant... (5 sec)</h3>';
  await new Promise(r => setTimeout(r, 5000));
  rec.stop();
  await new Promise(r => rec.onstop = r);
  stream.getTracks().forEach(t => t.stop());
  const blob = new Blob(chunks);
  const buf = await blob.arrayBuffer();
  const b64 = btoa(String.fromCharCode(...new Uint8Array(buf)));
  document.getElementById('msg').textContent = '✅ Enregistrement terminé, pipeline en cours...';
  await google.colab.kernel.invokeFunction('recv_audio', [b64], {});
})();
"""

def recv_audio(b64):
    with open('live_input.webm', 'wb') as f:
        f.write(base64.b64decode(b64))
    subprocess.run(['ffmpeg', '-i', 'live_input.webm', '-ar', '16000', '-ac', '1', 'live_input.wav', '-y'], capture_output=True)
    if os.path.exists('live_input.wav'):
        run_pipeline(audio_path='live_input.wav')
    else:
        print("❌ Erreur conversion audio")

output.register_callback('recv_audio', recv_audio)
display(Javascript(RECORD_JS))

In [ ]:
# Vérifier que l'audio existe
if os.path.exists('live_input.webm'):
    print("✅ live_input.webm trouvé")
    subprocess.run(['ffmpeg', '-i', 'live_input.webm', '-ar', '16000', '-ac', '1', 'live_input.wav', '-y'], capture_output=True)
    print("✅ Converti en wav")

    # Écouter ce qui a été enregistré
    ipd.display(ipd.Audio('live_input.wav'))

    # Lancer le pipeline
    run_pipeline(audio_path='live_input.wav')
else:
    print("❌ Fichier non trouvé")

In [ ]:
def generate_response(question, model, tokenizer):
    context = get_rag_context(question)
    if context:
        content = f"""أنت مساعد طبي يتكلم الدارجة الجزائرية فقط.
استخدم هاد المعلومات باش تجاوب:
{context}

قواعد مهمة:
- أجب بجملة واحدة أو جملتين فقط
- استخدم كلام بسيط مفهوم
- ما تذكرش فحوصات معقدة إلا كان السؤال خطير
- ما تذكرش أدوية بالاسم
- ركز على النصيحة العملية البسيطة

السؤال: {question}
الجواب بالدارجة الجزائرية:"""
    else:
        content = f"""أنت مساعد طبي يتكلم الدارجة الجزائرية فقط.

قواعد مهمة:
- أجب بجملة واحدة أو جملتين فقط
- استخدم كلام بسيط مفهوم
- ما تذكرش فحوصات معقدة إلا كان السؤال خطير
- ما تذكرش أدوية بالاسم
- ركز على النصيحة العملية البسيطة

السؤال: {question}
الجواب بالدارجة الجزائرية:"""

    messages = [{"role": "user", "content": content}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=60,
            temperature=0.2,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.5
        )
    input_len = inputs["input_ids"].shape[1]
    return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()


def synthesize(text, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    os.system(f'habibi-tts_infer-cli --ref_audio "{ref_audio}" --ref_text "{ref_text}" --gen_text "{text}" --dialect ALG --output_dir "{output_dir}"')
    path = f'{output_dir}/infer_cli_example.wav'
    if os.path.exists(path):
        ipd.display(ipd.Audio(path, autoplay=False))
    return path


def run_pipeline(audio_path=None, text_input=None):
    # Étape 1 : ASR
    if audio_path:
        print("🎤 Transcription Whisper...")
        question = transcribe(audio_path)
    else:
        question = text_input
    print(f"📝 Question : {question}\n")

    # Étape 2 : LLM x2
    print("🧠 AtlasChat (zero-shot)...")
    response_atlas = generate_response(question, atlas_model, atlas_tokenizer)
    print(f"AtlasChat : {response_atlas}\n")

    print("🧠 LLaMA fine-tuné...")
    response_llama = generate_response(question, llama_model, llama_tokenizer)
    print(f"LLaMA     : {response_llama}\n")

    # Étape 3 : TTS
    print("🔊 Synthèse AtlasChat...")
    synthesize(response_atlas, './outputs_atlas')

    print("🔊 Synthèse LLaMA...")
    synthesize(response_llama, './outputs_llama')

print("✅ Fonctions prêtes")

In [ ]:
output.register_callback('recv_audio', recv_audio)
display(Javascript(RECORD_JS))

In [ ]:
if os.path.exists('live_input.webm'):
    subprocess.run(['ffmpeg', '-i', 'live_input.webm', '-ar', '16000', '-ac', '1', 'live_input.wav', '-y'], capture_output=True)
    run_pipeline(audio_path='live_input.wav')

In [ ]:
!pip install streamlit pyngrok -q
print("✅ Streamlit installé")

In [ ]:
from pyngrok import ngrok
import threading, os, time

def run_streamlit():
    os.system('streamlit run app.py --server.port 8501 --server.headless true 2>&1')

thread = threading.Thread(target=run_streamlit, daemon=True)
thread.start()
time.sleep(8)

url = ngrok.connect(8501)
print(f"🌐 Ouvre ce lien dans ton navigateur : {url}")

In [ ]:
from pyngrok import ngrok, conf
import threading, os, time

# Mettre ton token ngrok ici
conf.get_default().auth_token = "3EXbAhX6eMVMPfO6Og7iliM66Jj_67BvD1QbUD77p1Doa64WK"

def run_streamlit():
    os.system('streamlit run app.py --server.port 8501 --server.headless true 2>&1')

thread = threading.Thread(target=run_streamlit, daemon=True)
thread.start()
time.sleep(8)

url = ngrok.connect(8501)
print(f"🌐 Ouvre ce lien : {url}")

In [ ]:
import time
time.sleep(15)
!curl -s http://localhost:8501

In [ ]:
!ps aux | grep streamlit

In [ ]:
!curl -s http://localhost:8501/healthz

In [ ]:
# Tuer le tunnel actuel et relancer avec le bon header
ngrok.kill()
import time
time.sleep(2)

from pyngrok import ngrok, conf

url = ngrok.connect(8501, bind_tls=True)
public_url = url.public_url
print(f"🌐 Lien : {public_url}")

In [ ]:
# Lire le fichier
with open('app.py', 'r', encoding='utf-8') as f:
    content = f.read()

# Supprimer les dernières lignes problématiques
content = content.replace("""with open('app.py', 'w', encoding='utf-8') as f:
    f.write(app_code)

print("✅ app.py créé")""", "")

# Réécrire
with open('app.py', 'w', encoding='utf-8') as f:
    f.write(content)

print("✅ Corrigé")

In [ ]:
!tail -50 app.py

In [ ]:
!cat app.py

In [ ]:
import os
os.remove('app.py')
print("✅ app.py supprimé")

In [ ]:
# app.py
import streamlit as st
import torch
import os
import subprocess
import soundfile as sf
import numpy as np
import faiss
from pathlib import Path
from transformers import (
    WhisperProcessor, WhisperForConditionalGeneration,
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
)
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
from huggingface_hub import login
import librosa
import tempfile

# ─── CONFIG ───────────────────────────────────
HF_TOKEN   = "TON_TOKEN_ICI"
REF_AUDIO  = "ref_voice.wav"
REF_TEXT   = "نيفي راهو يسيل"

st.set_page_config(
    page_title="المساعد الطبي الجزائري",
    page_icon="🏥",
    layout="wide"
)

# ─── CSS ──────────────────────────────────────
st.markdown("""
<style>
    body { direction: rtl; }
    .main { background-color: #f0f4f8; }
    .stApp { background-color: #f0f4f8; }

    .header {
        background: linear-gradient(135deg, #1a6b8a, #0d4d6b);
        color: white;
        padding: 20px;
        border-radius: 12px;
        text-align: center;
        margin-bottom: 20px;
    }
    .header h1 { font-size: 2rem; margin: 0; }
    .header p  { font-size: 1rem; margin: 5px 0 0 0; opacity: 0.85; }

    .chat-container {
        background: white;
        border-radius: 12px;
        padding: 20px;
        min-height: 400px;
        max-height: 500px;
        overflow-y: auto;
        box-shadow: 0 2px 8px rgba(0,0,0,0.08);
        margin-bottom: 16px;
    }

    .msg-user {
        background: #1a6b8a;
        color: white;
        padding: 10px 16px;
        border-radius: 18px 18px 4px 18px;
        margin: 8px 0;
        max-width: 75%;
        margin-left: auto;
        text-align: right;
        direction: rtl;
    }
    .msg-atlas {
        background: #e8f4f8;
        color: #1a1a1a;
        padding: 10px 16px;
        border-radius: 18px 18px 18px 4px;
        margin: 8px 0;
        max-width: 75%;
        border-left: 4px solid #1a6b8a;
        direction: rtl;
    }
    .msg-llama {
        background: #e8f8ee;
        color: #1a1a1a;
        padding: 10px 16px;
        border-radius: 18px 18px 18px 4px;
        margin: 8px 0;
        max-width: 75%;
        border-left: 4px solid #2ecc71;
        direction: rtl;
    }
    .label {
        font-size: 0.75rem;
        font-weight: bold;
        margin-bottom: 4px;
        opacity: 0.7;
    }

    .transcription-box {
        background: #fff8e1;
        border: 1px solid #ffc107;
        border-radius: 8px;
        padding: 12px;
        margin: 10px 0;
        direction: rtl;
        font-size: 1.1rem;
    }

    .status-box {
        background: #e3f2fd;
        border-radius: 8px;
        padding: 10px;
        text-align: center;
        margin: 8px 0;
        font-weight: bold;
        color: #1a6b8a;
    }
</style>
""", unsafe_allow_html=True)


# ─── CHARGEMENT MODÈLES (cache) ───────────────
@st.cache_resource
def load_all_models():
    login(token=HF_TOKEN)

    # Whisper
    whisper_processor = WhisperProcessor.from_pretrained("openai/whisper-small")
    whisper_model = WhisperForConditionalGeneration.from_pretrained(
        "abdeldjalil-smaili/whisper-algerian-medical-final",
        subfolder="checkpoint-1000",
        token=HF_TOKEN
    ).to("cuda" if torch.cuda.is_available() else "cpu")
    whisper_model.eval()

    # FAISS
    embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
    dataset  = load_dataset("abdeldjalil-smaili/algerian-medical-llm-v3", token=HF_TOKEN, split="train")

    def get_q(s):
        try: return s['text'].split("### Input:")[1].split("### Response:")[0].strip()
        except: return s['text'][:200]

    def get_a(s):
        try: return s['text'].split("### Response:")[1].strip()
        except: return ""

    questions = [get_q(s) for s in dataset]
    answers   = [get_a(s) for s in dataset]
    embeddings = embedder.encode(questions, batch_size=256, show_progress_bar=False, convert_to_numpy=True)
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings.astype(np.float32))

    # LLM
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )

    atlas_tok = AutoTokenizer.from_pretrained("MBZUAI-Paris/Atlas-Chat-9B", token=HF_TOKEN)
    atlas_mdl = AutoModelForCausalLM.from_pretrained(
        "MBZUAI-Paris/Atlas-Chat-9B", quantization_config=bnb,
        device_map="auto", token=HF_TOKEN
    )
    atlas_mdl.eval()

    llama_tok = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B-Instruct", token=HF_TOKEN)
    llama_mdl = AutoModelForCausalLM.from_pretrained(
        "abdeldjalil-smaili/llama3b-algerian-medical", quantization_config=bnb,
        device_map="auto", token=HF_TOKEN
    )
    llama_mdl.eval()

    return (whisper_processor, whisper_model,
            embedder, index, questions, answers,
            atlas_tok, atlas_mdl, llama_tok, llama_mdl)


# ─── FONCTIONS ────────────────────────────────
def transcribe(audio_path, processor, model):
    audio, _ = librosa.load(audio_path, sr=16000)
    inputs = processor(audio, sampling_rate=16000, return_tensors="pt").input_features
    inputs = inputs.to("cuda" if torch.cuda.is_available() else "cpu")
    with torch.no_grad():
        ids = model.generate(inputs, language="ar")
    return processor.batch_decode(ids, skip_special_tokens=True)[0]


def get_rag_context(question, embedder, index, questions, answers, k=3, threshold=8.0):
    q_emb = embedder.encode([question], convert_to_numpy=True)
    distances, indices = index.search(q_emb.astype(np.float32), k)
    contexts = []
    for dist, idx in zip(distances[0], indices[0]):
        if dist < threshold:
            contexts.append(f"Q: {questions[idx]}\nR: {answers[idx]}")
    return "\n\n".join(contexts)


def generate_response(question, model, tokenizer, embedder, index, questions, answers):
    context = get_rag_context(question, embedder, index, questions, answers)
    if context:
        content = f"""أنت مساعد طبي يتكلم الدارجة الجزائرية فقط.
استخدم هاد المعلومات باش تجاوب:
{context}

قواعد مهمة:
- أجب بجملة واحدة أو جملتين فقط
- استخدم كلام بسيط مفهوم
- ما تذكرش فحوصات معقدة إلا كان السؤال خطير
- ما تذكرش أدوية بالاسم
- ركز على النصيحة العملية البسيطة

السؤال: {question}
الجواب بالدارجة الجزائرية:"""
    else:
        content = f"""أنت مساعد طبي يتكلم الدارجة الجزائرية فقط.

قواعد مهمة:
- أجب بجملة واحدة أو جملتين فقط
- استخدم كلام بسيط مفهوم
- ما تذكرش فحوصات معقدة إلا كان السؤال خطير
- ما تذكرش أدوية بالاسم
- ركز على النصيحة العملية البسيطة

السؤال: {question}
الجواب بالدارجة الجزائرية:"""

    messages = [{"role": "user", "content": content}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    inputs = {k: v.to("cuda" if torch.cuda.is_available() else "cpu") for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=60, temperature=0.2,
            do_sample=True, pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.5
        )
    input_len = inputs["input_ids"].shape[1]
    return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()


def synthesize_tts(text, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    os.system(f'habibi-tts_infer-cli --ref_audio "{REF_AUDIO}" --ref_text "{REF_TEXT}" --gen_text "{text}" --dialect ALG --output_dir "{output_dir}"')
    return f"{output_dir}/infer_cli_example.wav"


# ─── UI ───────────────────────────────────────
st.markdown("""
<div class="header">
    <h1>🏥 المساعد الطبي الجزائري</h1>
    <p>نظام ذكاء اصطناعي طبي بالدارجة الجزائرية</p>
</div>
""", unsafe_allow_html=True)

# Initialiser historique
if "history" not in st.session_state:
    st.session_state.history = []

# Charger modèles
with st.spinner("⏳ جاري تحميل النماذج... (قد يأخذ بعض الوقت)"):
    (w_proc, w_mdl, embedder, index, questions, answers,
     atlas_tok, atlas_mdl, llama_tok, llama_mdl) = load_all_models()
st.success("✅ النماذج جاهزة!")

col1, col2 = st.columns([2, 1])

with col1:
    st.markdown("### 💬 المحادثة")

    # Afficher historique
    chat_html = '<div class="chat-container">'
    if not st.session_state.history:
        chat_html += '<p style="text-align:center; color:#999; margin-top:150px;">اسأل سؤالك الطبي بالدارجة الجزائرية 🎤</p>'
    for msg in st.session_state.history:
        chat_html += f'<div class="label" style="text-align:right;">👤 أنت</div>'
        chat_html += f'<div class="msg-user">{msg["question"]}</div>'
        chat_html += f'<div class="label">🤖 Atlas-Chat (Zero-Shot)</div>'
        chat_html += f'<div class="msg-atlas">{msg["atlas"]}</div>'
        chat_html += f'<div class="label">🧠 LLaMA (Fine-Tuné)</div>'
        chat_html += f'<div class="msg-llama">{msg["llama"]}</div>'
        chat_html += '<hr style="border:none;border-top:1px solid #eee;margin:12px 0">'
    chat_html += '</div>'
    st.markdown(chat_html, unsafe_allow_html=True)

    # Zone transcription
    if "transcription" in st.session_state and st.session_state.transcription:
        st.markdown(f"""
        <div class="transcription-box">
            📝 <b>التفريغ:</b> {st.session_state.transcription}
        </div>
        """, unsafe_allow_html=True)

with col2:
    st.markdown("### 🎤 تسجيل السؤال")

    # Upload audio
    audio_file = st.file_uploader(
        "ارفع تسجيلك الصوتي", type=["wav", "mp3", "webm", "m4a"],
        help="سجل سؤالك بالدارجة الجزائرية وارفعه هنا"
    )

    if audio_file:
        st.audio(audio_file)

        if st.button("🚀 ابدأ التحليل", type="primary", use_container_width=True):
            # Sauvegarder audio
            with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
                tmp_path = tmp.name

            with open(tmp_path, 'wb') as f:
                f.write(audio_file.read())

            # Convertir en wav 16kHz
            wav_path = tmp_path.replace(".wav", "_16k.wav")
            subprocess.run(['ffmpeg', '-i', tmp_path, '-ar', '16000', '-ac', '1', wav_path, '-y'],
                           capture_output=True)

            # Transcription
            with st.spinner("🎤 جاري التفريغ..."):
                question = transcribe(wav_path, w_proc, w_mdl)
                st.session_state.transcription = question

            st.markdown(f"""
            <div class="transcription-box">
                📝 <b>التفريغ:</b> {question}
            </div>
            """, unsafe_allow_html=True)

            # Génération réponses
            with st.spinner("🧠 Atlas-Chat يجاوب..."):
                response_atlas = generate_response(
                    question, atlas_mdl, atlas_tok, embedder, index, questions, answers
                )

            with st.spinner("🧠 LLaMA يجاوب..."):
                response_llama = generate_response(
                    question, llama_mdl, llama_tok, embedder, index, questions, answers
                )

            # TTS
            with st.spinner("🔊 جاري التوليد الصوتي..."):
                atlas_audio = synthesize_tts(response_atlas, "./outputs_atlas")
                llama_audio = synthesize_tts(response_llama, "./outputs_llama")

            # Ajouter à l'historique
            st.session_state.history.append({
                "question": question,
                "atlas": response_atlas,
                "llama": response_llama,
                "atlas_audio": atlas_audio,
                "llama_audio": llama_audio
            })

            st.rerun()

    # Afficher audios du dernier message
    if st.session_state.history:
        last = st.session_state.history[-1]
        st.markdown("### 🔊 الردود الصوتية")

        st.markdown("**Atlas-Chat:**")
        if os.path.exists(last["atlas_audio"]):
            st.audio(last["atlas_audio"])

        st.markdown("**LLaMA fine-tuné:**")
        if os.path.exists(last["llama_audio"]):
            st.audio(last["llama_audio"])

    # Bouton effacer historique
    if st.session_state.history:
        st.markdown("---")
        if st.button("🗑️ مسح المحادثة", use_container_width=True):
            st.session_state.history = []
            st.session_state.transcription = ""
            st.rerun()

In [ ]:
def generate_response(question, model, tokenizer):
    context = get_rag_context(question)
    if context:
        content = f"""أنت مساعد طبي يتكلم الدارجة الجزائرية فقط.
استخدم هاد المعلومات باش تجاوب:
{context}

قواعد مهمة:
- أجب بجملة واحدة أو جملتين فقط
- استخدم كلام بسيط مفهوم
- ما تذكرش فحوصات معقدة إلا كان السؤال خطير
- ما تذكرش أدوية بالاسم
- ركز على النصيحة العملية البسيطة

السؤال: {question}
الجواب بالدارجة الجزائرية:"""
    else:
        content = f"""أنت مساعد طبي يتكلم الدارجة الجزائرية فقط.

قواعد مهمة:
- أجب بجملة واحدة أو جملتين فقط
- استخدم كلام بسيط مفهوم
- ما تذكرش فحوصات معقدة إلا كان السؤال خطير
- ما تذكرش أدوية بالاسم
- ركز على النصيحة العملية البسيطة

السؤال: {question}
الجواب بالدارجة الجزائرية:"""

    messages = [{"role": "user", "content": content}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=60,
            temperature=0.2,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.5
        )
    input_len = inputs["input_ids"].shape[1]
    return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()

In [ ]:
%%writefile app.py
import os
os.system("pip uninstall torchcodec -y -q")

import streamlit as st
import torch
import subprocess
import numpy as np
import faiss
from transformers import (
    WhisperProcessor, WhisperForConditionalGeneration,
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
)
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
from huggingface_hub import login
import librosa
import tempfile

from huggingface_hub import login
login()
REF_AUDIO = "ref_voice.wav"
REF_TEXT  = "نيفي راهو يسيل"

st.set_page_config(page_title="المساعد الطبي الجزائري", page_icon="🏥", layout="wide")

st.markdown("""
<style>
    .header {
        background: linear-gradient(135deg, #1a6b8a, #0d4d6b);
        color: white; padding: 20px; border-radius: 12px;
        text-align: center; margin-bottom: 20px;
    }
    .header h1 { font-size: 2rem; margin: 0; }
    .header p  { font-size: 1rem; margin: 5px 0 0 0; opacity: 0.85; }
    .chat-container {
        background: white; border-radius: 12px; padding: 20px;
        min-height: 400px; max-height: 500px; overflow-y: auto;
        box-shadow: 0 2px 8px rgba(0,0,0,0.08); margin-bottom: 16px;
    }
    .msg-user {
        background: #1a6b8a; color: white; padding: 10px 16px;
        border-radius: 18px 18px 4px 18px; margin: 8px 0;
        max-width: 75%; margin-left: auto; text-align: right; direction: rtl;
    }
    .msg-atlas {
        background: #e8f4f8; color: #1a1a1a; padding: 10px 16px;
        border-radius: 18px 18px 18px 4px; margin: 8px 0;
        max-width: 75%; border-left: 4px solid #1a6b8a; direction: rtl;
    }
    .msg-llama {
        background: #e8f8ee; color: #1a1a1a; padding: 10px 16px;
        border-radius: 18px 18px 18px 4px; margin: 8px 0;
        max-width: 75%; border-left: 4px solid #2ecc71; direction: rtl;
    }
    .transcription-box {
        background: #fff8e1; border: 1px solid #ffc107;
        border-radius: 8px; padding: 12px; margin: 10px 0;
        direction: rtl; font-size: 1.1rem;
    }
</style>
""", unsafe_allow_html=True)

st.markdown("""
<div class="header">
    <h1>🏥 المساعد الطبي الجزائري</h1>
    <p>نظام ذكاء اصطناعي طبي بالدارجة الجزائرية</p>
</div>
""", unsafe_allow_html=True)

if "history" not in st.session_state:
    st.session_state.history = []
if "transcription" not in st.session_state:
    st.session_state.transcription = ""

@st.cache_resource
def load_all_models():
    login(token=HF_TOKEN)
    whisper_processor = WhisperProcessor.from_pretrained("openai/whisper-small")
    whisper_model = WhisperForConditionalGeneration.from_pretrained(
        "abdeldjalil-smaili/whisper-algerian-medical-final",
        subfolder="checkpoint-1000", token=HF_TOKEN
    ).to("cuda" if torch.cuda.is_available() else "cpu")
    whisper_model.eval()

    embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
    dataset  = load_dataset("abdeldjalil-smaili/algerian-medical-llm-v3", token=HF_TOKEN, split="train")

    def get_q(s):
        try: return s['text'].split("### Input:")[1].split("### Response:")[0].strip()
        except: return s['text'][:200]
    def get_a(s):
        try: return s['text'].split("### Response:")[1].strip()
        except: return ""

    questions  = [get_q(s) for s in dataset]
    answers    = [get_a(s) for s in dataset]
    embeddings = embedder.encode(questions, batch_size=256, show_progress_bar=False, convert_to_numpy=True)
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings.astype(np.float32))

    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16)

    atlas_tok = AutoTokenizer.from_pretrained("MBZUAI-Paris/Atlas-Chat-9B", token=HF_TOKEN)
    atlas_mdl = AutoModelForCausalLM.from_pretrained(
        "MBZUAI-Paris/Atlas-Chat-9B", quantization_config=bnb, device_map="auto", token=HF_TOKEN
    )
    atlas_mdl.eval()

    llama_tok = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B-Instruct", token=HF_TOKEN)
    llama_mdl = AutoModelForCausalLM.from_pretrained(
        "abdeldjalil-smaili/llama3b-algerian-medical", quantization_config=bnb, device_map="auto", token=HF_TOKEN
    )
    llama_mdl.eval()

    return (whisper_processor, whisper_model, embedder, index, questions, answers,
            atlas_tok, atlas_mdl, llama_tok, llama_mdl)


def transcribe(audio_path, processor, model):
    audio, _ = librosa.load(audio_path, sr=16000)
    inputs = processor(audio, sampling_rate=16000, return_tensors="pt").input_features
    inputs = inputs.to("cuda" if torch.cuda.is_available() else "cpu")
    with torch.no_grad():
        ids = model.generate(inputs, language="ar")
    return processor.batch_decode(ids, skip_special_tokens=True)[0]


def get_rag_context(question, embedder, index, questions, answers, k=3, threshold=8.0):
    q_emb = embedder.encode([question], convert_to_numpy=True)
    distances, indices = index.search(q_emb.astype(np.float32), k)
    contexts = []
    for dist, idx in zip(distances[0], indices[0]):
        if dist < threshold:
            contexts.append(f"Q: {questions[idx]}\nR: {answers[idx]}")
    return "\n\n".join(contexts)


def generate_response(question, model, tokenizer, embedder, index, questions, answers):
    context = get_rag_context(question, embedder, index, questions, answers)
    content = f"""أنت مساعد طبي يتكلم الدارجة الجزائرية فقط.
{f"استخدم هاد المعلومات باش تجاوب:{chr(10)}{context}{chr(10)}{chr(10)}" if context else ""}
قواعد مهمة:
- أجب بجملة واحدة أو جملتين فقط
- استخدم كلام بسيط مفهوم
- ما تذكرش فحوصات معقدة إلا كان السؤال خطير
- ركز على النصيحة العملية البسيطة

السؤال: {question}
الجواب بالدارجة الجزائرية:"""

    messages = [{"role": "user", "content": content}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    inputs = {k: v.to("cuda" if torch.cuda.is_available() else "cpu") for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=60, temperature=0.2,
            do_sample=True, pad_token_id=tokenizer.eos_token_id, repetition_penalty=1.5
        )
    input_len = inputs["input_ids"].shape[1]
    return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()


def synthesize_tts(text, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    os.system(f'habibi-tts_infer-cli --ref_audio "{REF_AUDIO}" --ref_text "{REF_TEXT}" --gen_text "{text}" --dialect ALG --output_dir "{output_dir}"')
    return f"{output_dir}/infer_cli_example.wav"


with st.spinner("⏳ جاري تحميل النماذج..."):
    (w_proc, w_mdl, embedder, index, questions, answers,
     atlas_tok, atlas_mdl, llama_tok, llama_mdl) = load_all_models()
st.success("✅ النماذج جاهزة!")

col1, col2 = st.columns([2, 1])

with col1:
    st.markdown("### 💬 المحادثة")
    chat_html = '<div class="chat-container">'
    if not st.session_state.history:
        chat_html += '<p style="text-align:center;color:#999;margin-top:150px;">اسأل سؤالك الطبي بالدارجة الجزائرية 🎤</p>'
    for msg in st.session_state.history:
        chat_html += f'<div style="text-align:right;font-size:0.75rem;opacity:0.7;">👤 أنت</div>'
        chat_html += f'<div class="msg-user">{msg["question"]}</div>'
        chat_html += f'<div style="font-size:0.75rem;opacity:0.7;">🤖 Atlas-Chat</div>'
        chat_html += f'<div class="msg-atlas">{msg["atlas"]}</div>'
        chat_html += f'<div style="font-size:0.75rem;opacity:0.7;">🧠 LLaMA Fine-Tuné</div>'
        chat_html += f'<div class="msg-llama">{msg["llama"]}</div>'
        chat_html += '<hr style="border:none;border-top:1px solid #eee;margin:12px 0">'
    chat_html += '</div>'
    st.markdown(chat_html, unsafe_allow_html=True)

    if st.session_state.transcription:
        st.markdown(f'<div class="transcription-box">📝 <b>التفريغ:</b> {st.session_state.transcription}</div>',
                    unsafe_allow_html=True)

with col2:
    st.markdown("### 🎤 تسجيل السؤال")
    audio_file = st.file_uploader("ارفع تسجيلك الصوتي", type=["wav", "mp3", "webm", "m4a"])

    if audio_file:
        st.audio(audio_file)
        if st.button("🚀 ابدأ التحليل", type="primary", use_container_width=True):
            with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
                tmp.write(audio_file.read())
                tmp_path = tmp.name

            wav_path = tmp_path.replace(".wav", "_16k.wav")
            subprocess.run(['ffmpeg', '-i', tmp_path, '-ar', '16000', '-ac', '1', wav_path, '-y'], capture_output=True)

            with st.spinner("🎤 جاري التفريغ..."):
                question = transcribe(wav_path, w_proc, w_mdl)
                st.session_state.transcription = question

            with st.spinner("🧠 Atlas-Chat يجاوب..."):
                response_atlas = generate_response(question, atlas_mdl, atlas_tok, embedder, index, questions, answers)

            with st.spinner("🧠 LLaMA يجاوب..."):
                response_llama = generate_response(question, llama_mdl, llama_tok, embedder, index, questions, answers)

            with st.spinner("🔊 جاري التوليد الصوتي..."):
                atlas_audio = synthesize_tts(response_atlas, "./outputs_atlas")
                llama_audio = synthesize_tts(response_llama, "./outputs_llama")

            st.session_state.history.append({
                "question": question,
                "atlas": response_atlas,
                "llama": response_llama,
                "atlas_audio": atlas_audio,
                "llama_audio": llama_audio
            })
            st.rerun()

    if st.session_state.history:
        last = st.session_state.history[-1]
        st.markdown("### 🔊 الردود الصوتية")
        st.markdown("**Atlas-Chat:**")
        if os.path.exists(last["atlas_audio"]):
            st.audio(last["atlas_audio"])
        st.markdown("**LLaMA Fine-Tuné:**")
        if os.path.exists(last["llama_audio"]):
            st.audio(last["llama_audio"])
        st.markdown("---")
        if st.button("🗑️ مسح المحادثة", use_container_width=True):
            st.session_state.history = []
            st.session_state.transcription = ""
            st.rerun()

In [ ]:
import os, threading, time
from pyngrok import ngrok, conf

conf.get_default().auth_token = "3EXbAhX6eMVMPfO6Og7iliM66Jj_67BvD1QbUD77p1Doa64WK"

os.system("pkill -f streamlit")
time.sleep(2)

def run():
    os.system("streamlit run app.py --server.port 8501 --server.headless true")

threading.Thread(target=run, daemon=True).start()
time.sleep(8)

url = ngrok.connect(8501)
print(f"🌐 {url.public_url}?ngrok-skip-browser-warning=true")

In [ ]:
%%writefile app.py
import os
os.system("pip uninstall torchcodec -y -q")

import streamlit as st
import torch
import subprocess
import numpy as np
import faiss
from transformers import (
    WhisperProcessor, WhisperForConditionalGeneration,
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
)
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
from huggingface_hub import login
import librosa
import tempfile

from huggingface_hub import login
login()
REF_AUDIO = "ref_voice.wav"
REF_TEXT  = "نيفي راهو يسيل"

st.set_page_config(page_title="المساعد الطبي الجزائري", page_icon="🏥", layout="wide")

st.markdown("""
<style>
    .header {
        background: linear-gradient(135deg, #1a6b8a, #0d4d6b);
        color: white; padding: 20px; border-radius: 12px;
        text-align: center; margin-bottom: 20px;
    }
    .header h1 { font-size: 2rem; margin: 0; }
    .header p  { font-size: 1rem; margin: 5px 0 0 0; opacity: 0.85; }
    .chat-container {
        background: white; border-radius: 12px; padding: 20px;
        min-height: 400px; max-height: 500px; overflow-y: auto;
        box-shadow: 0 2px 8px rgba(0,0,0,0.08); margin-bottom: 16px;
    }
    .msg-user {
        background: #1a6b8a; color: white; padding: 10px 16px;
        border-radius: 18px 18px 4px 18px; margin: 8px 0;
        max-width: 75%; margin-left: auto; text-align: right; direction: rtl;
    }
    .msg-atlas {
        background: #e8f4f8; color: #1a1a1a; padding: 10px 16px;
        border-radius: 18px 18px 18px 4px; margin: 8px 0;
        max-width: 75%; border-left: 4px solid #1a6b8a; direction: rtl;
    }
    .msg-llama {
        background: #e8f8ee; color: #1a1a1a; padding: 10px 16px;
        border-radius: 18px 18px 18px 4px; margin: 8px 0;
        max-width: 75%; border-left: 4px solid #2ecc71; direction: rtl;
    }
    .transcription-box {
        background: #fff8e1; border: 1px solid #ffc107;
        border-radius: 8px; padding: 12px; margin: 10px 0;
        direction: rtl; font-size: 1.1rem;
    }
    .last-question-box {
        background: #e3f2fd; border: 1px solid #1a6b8a;
        border-radius: 8px; padding: 12px; margin: 10px 0;
        direction: rtl; font-size: 1.1rem;
    }
</style>
""", unsafe_allow_html=True)

st.markdown("""
<div class="header">
    <h1>🏥 المساعد الطبي الجزائري</h1>
    <p>نظام ذكاء اصطناعي طبي بالدارجة الجزائرية</p>
</div>
""", unsafe_allow_html=True)

if "history" not in st.session_state:
    st.session_state.history = []
if "transcription" not in st.session_state:
    st.session_state.transcription = ""

@st.cache_resource
def load_all_models():
    login(token=HF_TOKEN)

    whisper_processor = WhisperProcessor.from_pretrained("openai/whisper-small")
    whisper_model = WhisperForConditionalGeneration.from_pretrained(
        "abdeldjalil-smaili/whisper-algerian-medical-final",
        subfolder="checkpoint-1000", token=HF_TOKEN
    ).to("cuda" if torch.cuda.is_available() else "cpu")
    whisper_model.eval()

    embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
    dataset  = load_dataset("abdeldjalil-smaili/algerian-medical-llm-v3", token=HF_TOKEN, split="train")

    def get_q(s):
        try: return s['text'].split("### Input:")[1].split("### Response:")[0].strip()
        except: return s['text'][:200]
    def get_a(s):
        try: return s['text'].split("### Response:")[1].strip()
        except: return ""

    questions  = [get_q(s) for s in dataset]
    answers    = [get_a(s) for s in dataset]
    embeddings = embedder.encode(questions, batch_size=256, show_progress_bar=False, convert_to_numpy=True)
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings.astype(np.float32))

    bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )

    atlas_tok = AutoTokenizer.from_pretrained("MBZUAI-Paris/Atlas-Chat-9B", token=HF_TOKEN)
    atlas_mdl = AutoModelForCausalLM.from_pretrained(
        "MBZUAI-Paris/Atlas-Chat-9B", quantization_config=bnb,
        device_map="auto", token=HF_TOKEN
    )
    atlas_mdl.eval()

    llama_tok = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B-Instruct", token=HF_TOKEN)
    llama_mdl = AutoModelForCausalLM.from_pretrained(
        "abdeldjalil-smaili/llama3b-algerian-medical", quantization_config=bnb,
        device_map="auto", token=HF_TOKEN
    )
    llama_mdl.eval()

    return (whisper_processor, whisper_model, embedder, index, questions, answers,
            atlas_tok, atlas_mdl, llama_tok, llama_mdl)


def transcribe(audio_path, processor, model):
    audio, _ = librosa.load(audio_path, sr=16000)
    inputs = processor(audio, sampling_rate=16000, return_tensors="pt").input_features
    inputs = inputs.to("cuda" if torch.cuda.is_available() else "cpu")
    with torch.no_grad():
        ids = model.generate(inputs, language="ar")
    return processor.batch_decode(ids, skip_special_tokens=True)[0]


def get_rag_context(question, embedder, index, questions, answers, k=3, threshold=8.0):
    q_emb = embedder.encode([question], convert_to_numpy=True)
    distances, indices = index.search(q_emb.astype(np.float32), k)
    contexts = []
    for dist, idx in zip(distances[0], indices[0]):
        if dist < threshold:
            contexts.append(f"Q: {questions[idx]}\nR: {answers[idx]}")
    return "\n\n".join(contexts)


def generate_response(question, model, tokenizer, embedder, index, questions, answers):
    context = get_rag_context(question, embedder, index, questions, answers)
    content = f"""أنت مساعد طبي يتكلم الدارجة الجزائرية فقط.
{f"استخدم هاد المعلومات باش تجاوب:{chr(10)}{context}{chr(10)}{chr(10)}" if context else ""}
قواعد مهمة:
- أجب بجملة واحدة أو جملتين فقط
- استخدم كلام بسيط مفهوم
- ما تذكرش فحوصات معقدة إلا كان السؤال خطير
- ركز على النصيحة العملية البسيطة

السؤال: {question}
الجواب بالدارجة الجزائرية:"""

    messages = [{"role": "user", "content": content}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    inputs = {k: v.to("cuda" if torch.cuda.is_available() else "cpu") for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=60, temperature=0.2,
            do_sample=True, pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.5
        )
    input_len = inputs["input_ids"].shape[1]
    return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()


def synthesize_tts(text, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    os.system(f'habibi-tts_infer-cli --ref_audio "{REF_AUDIO}" --ref_text "{REF_TEXT}" --gen_text "{text}" --dialect ALG --output_dir "{output_dir}"')
    return f"{output_dir}/infer_cli_example.wav"


with st.spinner("⏳ جاري تحميل النماذج..."):
    (w_proc, w_mdl, embedder, index, questions, answers,
     atlas_tok, atlas_mdl, llama_tok, llama_mdl) = load_all_models()
st.success("✅ النماذج جاهزة!")

col1, col2 = st.columns([2, 1])

with col1:
    st.markdown("### 💬 المحادثة")

    chat_html = '<div class="chat-container">'
    if not st.session_state.history:
        chat_html += '<p style="text-align:center;color:#999;margin-top:150px;">اسأل سؤالك الطبي بالدارجة الجزائرية 🎤</p>'
    for msg in st.session_state.history:
        chat_html += f'<div style="text-align:right;font-size:0.75rem;opacity:0.7;margin-top:10px;">👤 أنت</div>'
        chat_html += f'<div class="msg-user">{msg["question"]}</div>'
        chat_html += f'<div style="font-size:0.75rem;opacity:0.7;">🤖 Atlas-Chat (Zero-Shot)</div>'
        chat_html += f'<div class="msg-atlas">{msg["atlas"]}</div>'
        chat_html += f'<div style="font-size:0.75rem;opacity:0.7;">🧠 LLaMA (Fine-Tuné)</div>'
        chat_html += f'<div class="msg-llama">{msg["llama"]}</div>'
        chat_html += '<hr style="border:none;border-top:1px solid #eee;margin:12px 0">'
    chat_html += '</div>'
    st.markdown(chat_html, unsafe_allow_html=True)

    if st.session_state.transcription:
        st.markdown(f"""
        <div class="last-question-box">
            📝 <b>آخر سؤال مفرَّغ:</b> {st.session_state.transcription}
        </div>
        """, unsafe_allow_html=True)


with col2:
    st.markdown("### 🎤 تسجيل السؤال")

    wav_audio_data = st_audiorec()

    transcription_placeholder = st.empty()

    if wav_audio_data is not None:
        st.audio(wav_audio_data, format='audio/wav')

        if st.button("🚀 ابدأ التحليل", type="primary", use_container_width=True):

            with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
                tmp.write(wav_audio_data)
                tmp_path = tmp.name

            wav_path = tmp_path.replace(".wav", "_16k.wav")
            subprocess.run(
                ['ffmpeg', '-i', tmp_path, '-ar', '16000', '-ac', '1', wav_path, '-y'],
                capture_output=True
            )

            with st.spinner("🎤 جاري التفريغ..."):
                question = transcribe(wav_path, w_proc, w_mdl)
                st.session_state.transcription = question

            transcription_placeholder.markdown(f"""
            <div class="transcription-box">
                🎤 <b>سؤالك:</b> {question}
            </div>
            """, unsafe_allow_html=True)

            with st.spinner("🧠 Atlas-Chat يجاوب..."):
                response_atlas = generate_response(
                    question, atlas_mdl, atlas_tok, embedder, index, questions, answers
                )

            with st.spinner("🧠 LLaMA يجاوب..."):
                response_llama = generate_response(
                    question, llama_mdl, llama_tok, embedder, index, questions, answers
                )

            with st.spinner("🔊 جاري التوليد الصوتي..."):
                atlas_audio = synthesize_tts(response_atlas, "./outputs_atlas")
                llama_audio = synthesize_tts(response_llama, "./outputs_llama")

            st.session_state.history.append({
                "question": question,
                "atlas": response_atlas,
                "llama": response_llama,
                "atlas_audio": atlas_audio,
                "llama_audio": llama_audio
            })
            st.rerun()

    if st.session_state.history:
        last = st.session_state.history[-1]
        st.markdown("### 🔊 الردود الصوتية")
        st.markdown("**Atlas-Chat:**")
        if os.path.exists(last["atlas_audio"]):
            st.audio(last["atlas_audio"])
        st.markdown("**LLaMA Fine-Tuné:**")
        if os.path.exists(last["llama_audio"]):
            st.audio(last["llama_audio"])
        st.markdown("---")
        if st.button("🗑️ مسح المحادثة", use_container_width=True):
            st.session_state.history = []
            st.session_state.transcription = ""
            st.rerun()

In [ ]:
os.system("pkill -f streamlit")
import time
time.sleep(2)

import threading
from pyngrok import ngrok, conf

conf.get_default().auth_token = "3EXbAhX6eMVMPfO6Og7iliM66Jj_67BvD1QbUD77p1Doa64WK"

def run():
    os.system("streamlit run app.py --server.port 8501 --server.headless true")

threading.Thread(target=run, daemon=True).start()
time.sleep(8)

url = ngrok.connect(8501)
print(f"🌐 {url.public_url}?ngrok-skip-browser-warning=true")

In [ ]:
!pip install streamlit-audiorec -q

In [ ]:
!pip install transformers==4.45.0 accelerate bitsandbytes sentence-transformers faiss-cpu datasets tokenizers soundfile librosa -q
!pip install torch==2.4.0 torchaudio==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121 -q
!pip install habibi-tts streamlit pyngrok streamlit-audiorec -q
!pip uninstall torchcodec -y -q
!apt-get install -y ffmpeg -q
print("✅ Installation terminée")

In [ ]:
from huggingface_hub import login
import os

from huggingface_hub import login
login()

ref_audio = 'ref_voice.wav'
ref_text  = "نيفي راهو يسيل"

os.makedirs('./outputs_llama', exist_ok=True)
os.makedirs('./outputs_atlas', exist_ok=True)
print("✅ Config OK")

In [ ]:
from google.colab import files
uploaded = files.upload()  # uploade new_test.mp4
os.system('ffmpeg -i new_test.mp4 -ar 24000 -ac 1 ref_voice.wav -y')
print("✅ ref_voice.wav prêt")

In [ ]:
%%writefile app.py
import os
os.system("pip uninstall torchcodec -y -q")
import streamlit as st
import torch
import os
import subprocess
import soundfile as sf
import numpy as np
import faiss
from pathlib import Path
from transformers import (
    WhisperProcessor, WhisperForConditionalGeneration,
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
)
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
from huggingface_hub import login
import librosa
import tempfile

# ─── CONFIG ───────────────────────────────────
from huggingface_hub import login
login()
REF_AUDIO  = "ref_voice.wav"
REF_TEXT   = "نيفي راهو يسيل"

st.set_page_config(
    page_title="المساعد الطبي الجزائري",
    page_icon="🏥",
    layout="wide"
)

# ─── CSS ──────────────────────────────────────
st.markdown("""
<style>
    body { direction: rtl; }
    .main { background-color: #f0f4f8; }
    .stApp { background-color: #f0f4f8; }

    .header {
        background: linear-gradient(135deg, #1a6b8a, #0d4d6b);
        color: white;
        padding: 20px;
        border-radius: 12px;
        text-align: center;
        margin-bottom: 20px;
    }
    .header h1 { font-size: 2rem; margin: 0; }
    .header p  { font-size: 1rem; margin: 5px 0 0 0; opacity: 0.85; }

    .chat-container {
        background: white;
        border-radius: 12px;
        padding: 20px;
        min-height: 400px;
        max-height: 500px;
        overflow-y: auto;
        box-shadow: 0 2px 8px rgba(0,0,0,0.08);
        margin-bottom: 16px;
    }

    .msg-user {
        background: #1a6b8a;
        color: white;
        padding: 10px 16px;
        border-radius: 18px 18px 4px 18px;
        margin: 8px 0;
        max-width: 75%;
        margin-left: auto;
        text-align: right;
        direction: rtl;
    }
    .msg-atlas {
        background: #e8f4f8;
        color: #1a1a1a;
        padding: 10px 16px;
        border-radius: 18px 18px 18px 4px;
        margin: 8px 0;
        max-width: 75%;
        border-left: 4px solid #1a6b8a;
        direction: rtl;
    }
    .msg-llama {
        background: #e8f8ee;
        color: #1a1a1a;
        padding: 10px 16px;
        border-radius: 18px 18px 18px 4px;
        margin: 8px 0;
        max-width: 75%;
        border-left: 4px solid #2ecc71;
        direction: rtl;
    }
    .label {
        font-size: 0.75rem;
        font-weight: bold;
        margin-bottom: 4px;
        opacity: 0.7;
    }

    .transcription-box {
        background: #fff8e1;
        border: 1px solid #ffc107;
        border-radius: 8px;
        padding: 12px;
        margin: 10px 0;
        direction: rtl;
        font-size: 1.1rem;
    }

    .status-box {
        background: #e3f2fd;
        border-radius: 8px;
        padding: 10px;
        text-align: center;
        margin: 8px 0;
        font-weight: bold;
        color: #1a6b8a;
    }
</style>
""", unsafe_allow_html=True)


# ─── CHARGEMENT MODÈLES (cache) ───────────────
@st.cache_resource
def load_all_models():
    login(token=HF_TOKEN)

    # Whisper
    whisper_processor = WhisperProcessor.from_pretrained("openai/whisper-small")
    whisper_model = WhisperForConditionalGeneration.from_pretrained(
        "abdeldjalil-smaili/whisper-algerian-medical-final",
        subfolder="checkpoint-1000",
        token=HF_TOKEN
    ).to("cuda" if torch.cuda.is_available() else "cpu")
    whisper_model.eval()

    # FAISS
    embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
    dataset  = load_dataset("abdeldjalil-smaili/algerian-medical-llm-v3", token=HF_TOKEN, split="train")

    def get_q(s):
        try: return s['text'].split("### Input:")[1].split("### Response:")[0].strip()
        except: return s['text'][:200]

    def get_a(s):
        try: return s['text'].split("### Response:")[1].strip()
        except: return ""

    questions = [get_q(s) for s in dataset]
    answers   = [get_a(s) for s in dataset]
    embeddings = embedder.encode(questions, batch_size=256, show_progress_bar=False, convert_to_numpy=True)
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings.astype(np.float32))

    # LLM
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )

    atlas_tok = AutoTokenizer.from_pretrained("MBZUAI-Paris/Atlas-Chat-9B", token=HF_TOKEN)
    atlas_mdl = AutoModelForCausalLM.from_pretrained(
        "MBZUAI-Paris/Atlas-Chat-9B", quantization_config=bnb,
        device_map="auto", token=HF_TOKEN
    )
    atlas_mdl.eval()

    llama_tok = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B-Instruct", token=HF_TOKEN)
    llama_mdl = AutoModelForCausalLM.from_pretrained(
        "abdeldjalil-smaili/llama3b-algerian-medical", quantization_config=bnb,
        device_map="auto", token=HF_TOKEN
    )
    llama_mdl.eval()

    return (whisper_processor, whisper_model,
            embedder, index, questions, answers,
            atlas_tok, atlas_mdl, llama_tok, llama_mdl)


# ─── FONCTIONS ────────────────────────────────
def transcribe(audio_path, processor, model):
    audio, _ = librosa.load(audio_path, sr=16000)
    inputs = processor(audio, sampling_rate=16000, return_tensors="pt").input_features
    inputs = inputs.to("cuda" if torch.cuda.is_available() else "cpu")
    with torch.no_grad():
        ids = model.generate(inputs, language="ar")
    return processor.batch_decode(ids, skip_special_tokens=True)[0]


def get_rag_context(question, embedder, index, questions, answers, k=3, threshold=8.0):
    q_emb = embedder.encode([question], convert_to_numpy=True)
    distances, indices = index.search(q_emb.astype(np.float32), k)
    contexts = []
    for dist, idx in zip(distances[0], indices[0]):
        if dist < threshold:
            contexts.append(f"Q: {questions[idx]}\nR: {answers[idx]}")
    return "\n\n".join(contexts)


def generate_response(question, model, tokenizer, embedder, index, questions, answers):
    context = get_rag_context(question, embedder, index, questions, answers)
    if context:
        content = f"""أنت مساعد طبي يتكلم الدارجة الجزائرية فقط.
استخدم هاد المعلومات باش تجاوب:
{context}

قواعد مهمة:
- أجب بجملة واحدة أو جملتين فقط
- استخدم كلام بسيط مفهوم
- ما تذكرش فحوصات معقدة إلا كان السؤال خطير
- ما تذكرش أدوية بالاسم
- ركز على النصيحة العملية البسيطة

السؤال: {question}
الجواب بالدارجة الجزائرية:"""
    else:
        content = f"""أنت مساعد طبي يتكلم الدارجة الجزائرية فقط.

قواعد مهمة:
- أجب بجملة واحدة أو جملتين فقط
- استخدم كلام بسيط مفهوم
- ما تذكرش فحوصات معقدة إلا كان السؤال خطير
- ما تذكرش أدوية بالاسم
- ركز على النصيحة العملية البسيطة

السؤال: {question}
الجواب بالدارجة الجزائرية:"""

    messages = [{"role": "user", "content": content}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    inputs = {k: v.to("cuda" if torch.cuda.is_available() else "cpu") for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=60, temperature=0.2,
            do_sample=True, pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.5
        )
    input_len = inputs["input_ids"].shape[1]
    return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()


def synthesize_tts(text, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    os.system(f'habibi-tts_infer-cli --ref_audio "{REF_AUDIO}" --ref_text "{REF_TEXT}" --gen_text "{text}" --dialect ALG --output_dir "{output_dir}"')
    return f"{output_dir}/infer_cli_example.wav"


# ─── UI ───────────────────────────────────────
st.markdown("""
<div class="header">
    <h1>🏥 المساعد الطبي الجزائري</h1>
    <p>نظام ذكاء اصطناعي طبي بالدارجة الجزائرية</p>
</div>
""", unsafe_allow_html=True)

# Initialiser historique
if "history" not in st.session_state:
    st.session_state.history = []

# Charger modèles
with st.spinner("⏳ استنا راهم يطلعو"):
    (w_proc, w_mdl, embedder, index, questions, answers,
     atlas_tok, atlas_mdl, llama_tok, llama_mdl) = load_all_models()
st.success("✅ راهم واجدين")

col1, col2 = st.columns([2, 1])

with col1:
    st.markdown("### 💬 المحادثة")

    # Afficher historique
    chat_html = '<div class="chat-container">'
    if not st.session_state.history:
        chat_html += '<p style="text-align:center; color:#999; margin-top:150px;"> سقسي بالدزيري🎤</p>'
    for msg in st.session_state.history:
        chat_html += f'<div class="label" style="text-align:right;">👤 أنت</div>'
        chat_html += f'<div class="msg-user">{msg["question"]}</div>'
        chat_html += f'<div class="label">🤖 Atlas-Chat (Zero-Shot)</div>'
        chat_html += f'<div class="msg-atlas">{msg["atlas"]}</div>'
        chat_html += f'<div class="label">🧠 LLaMA (Fine-Tuné)</div>'
        chat_html += f'<div class="msg-llama">{msg["llama"]}</div>'
        chat_html += '<hr style="border:none;border-top:1px solid #eee;margin:12px 0">'
    chat_html += '</div>'
    st.markdown(chat_html, unsafe_allow_html=True)

    # Zone transcription
    if "transcription" in st.session_state and st.session_state.transcription:
        st.markdown(f"""
        <div class="transcription-box">
            📝 <b>التفريغ:</b> {st.session_state.transcription}
        </div>
        """, unsafe_allow_html=True)

with col2:
    st.markdown("### 🎤 تسجيل السؤال")

    # Upload audio
    audio_file = st.file_uploader(
        "ارفع تسجيلك الصوتي", type=["wav", "mp3", "webm", "m4a"],
        help="سجل سؤالك بالدارجة الجزائرية وارفعه هنا"
    )

    if audio_file:
        st.audio(audio_file)

        if st.button("🚀 ابدأ التحليل", type="primary", use_container_width=True):
            # Sauvegarder audio
            with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
                tmp_path = tmp.name

            with open(tmp_path, 'wb') as f:
                f.write(audio_file.read())

            # Convertir en wav 16kHz
            wav_path = tmp_path.replace(".wav", "_16k.wav")
            subprocess.run(['ffmpeg', '-i', tmp_path, '-ar', '16000', '-ac', '1', wav_path, '-y'],
                           capture_output=True)

            # Transcription
            with st.spinner("🎤 جاري التفريغ..."):
                question = transcribe(wav_path, w_proc, w_mdl)
                st.session_state.transcription = question

            st.markdown(f"""
            <div class="transcription-box">
                📝 <b>التفريغ:</b> {question}
            </div>
            """, unsafe_allow_html=True)

            # Génération réponses
            with st.spinner("🧠 Atlas-Chat يجاوب..."):
                response_atlas = generate_response(
                    question, atlas_mdl, atlas_tok, embedder, index, questions, answers
                )

            with st.spinner("🧠 LLaMA يجاوب..."):
                response_llama = generate_response(
                    question, llama_mdl, llama_tok, embedder, index, questions, answers
                )

            # TTS
            with st.spinner("🔊 جاري التوليد الصوتي..."):
                atlas_audio = synthesize_tts(response_atlas, "./outputs_atlas")
                llama_audio = synthesize_tts(response_llama, "./outputs_llama")

            # Ajouter à l'historique
            st.session_state.history.append({
                "question": question,
                "atlas": response_atlas,
                "llama": response_llama,
                "atlas_audio": atlas_audio,
                "llama_audio": llama_audio
            })

            st.rerun()

    # Afficher audios du dernier message
    if st.session_state.history:
        last = st.session_state.history[-1]
        st.markdown("### 🔊 الردود الصوتية")

        st.markdown("**Atlas-Chat:**")
        if os.path.exists(last["atlas_audio"]):
            st.audio(last["atlas_audio"])

        st.markdown("**LLaMA fine-tuné:**")
        if os.path.exists(last["llama_audio"]):
            st.audio(last["llama_audio"])

    # Bouton effacer historique
    if st.session_state.history:
        st.markdown("---")
        if st.button("🗑️ مسح المحادثة", use_container_width=True):
            st.session_state.history = []
            st.session_state.transcription = ""
            st.rerun()

with open('app.py', 'w', encoding='utf-8') as f:
    f.write(app_code)

print("✅ app.py créé")

In [ ]:
%%writefile app.py
import os
os.system("pip uninstall torchcodec -y -q")

import streamlit as st
import torch
import subprocess
import numpy as np
import faiss
from transformers import (
   WhisperProcessor, WhisperForConditionalGeneration,
   AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
)
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
from huggingface_hub import login
import librosa
import tempfile

from huggingface_hub import login
login()
REF_AUDIO = "ref_voice.wav"
REF_TEXT  = "نيفي راهو يسيل"

st.set_page_config(page_title="المساعد الطبي الجزائري", page_icon="🏥", layout="wide")

st.markdown("""
<style>
   .header {
       background: linear-gradient(135deg, #1a6b8a, #0d4d6b);
       color: white; padding: 20px; border-radius: 12px;
       text-align: center; margin-bottom: 20px;
   }
   .header h1 { font-size: 2rem; margin: 0; }
   .header p  { font-size: 1rem; margin: 5px 0 0 0; opacity: 0.85; }
   .chat-container {
       background: white; border-radius: 12px; padding: 20px;
       min-height: 400px; max-height: 500px; overflow-y: auto;
       box-shadow: 0 2px 8px rgba(0,0,0,0.08); margin-bottom: 16px;
   }
   .msg-user {
       background: #1a6b8a; color: white; padding: 10px 16px;
       border-radius: 18px 18px 4px 18px; margin: 8px 0;
       max-width: 75%; margin-left: auto; text-align: right; direction: rtl;
   }
   .msg-atlas {
       background: #e8f4f8; color: #1a1a1a; padding: 10px 16px;
       border-radius: 18px 18px 18px 4px; margin: 8px 0;
       max-width: 75%; border-left: 4px solid #1a6b8a; direction: rtl;
   }
   .msg-llama {
       background: #e8f8ee; color: #1a1a1a; padding: 10px 16px;
       border-radius: 18px 18px 18px 4px; margin: 8px 0;
       max-width: 75%; border-left: 4px solid #2ecc71; direction: rtl;
   }
   .transcription-box {
       background: #fff8e1; border: 1px solid #ffc107;
       border-radius: 8px; padding: 12px; margin: 10px 0;
       direction: rtl; font-size: 1.1rem;
   }
   .last-question-box {
       background: #e3f2fd; border: 1px solid #1a6b8a;
       border-radius: 8px; padding: 12px; margin: 10px 0;
       direction: rtl; font-size: 1.1rem;
   }
</style>
""", unsafe_allow_html=True)

st.markdown("""
<div class="header">
   <h1>🏥 المساعد الطبي الجزائري</h1>
   <p>نظام ذكاء اصطناعي طبي بالدارجة الجزائرية</p>
</div>
""", unsafe_allow_html=True)

if "history" not in st.session_state:
   st.session_state.history = []
if "transcription" not in st.session_state:
   st.session_state.transcription = ""
if "last_audio" not in st.session_state:
   st.session_state.last_audio = None

@st.cache_resource
def load_all_models():
   login(token=HF_TOKEN)

   whisper_processor = WhisperProcessor.from_pretrained("openai/whisper-small")
   whisper_model = WhisperForConditionalGeneration.from_pretrained(
       "abdeldjalil-smaili/whisper-algerian-medical-final",
       subfolder="checkpoint-1000", token=HF_TOKEN
   ).to("cuda" if torch.cuda.is_available() else "cpu")
   whisper_model.eval()

   embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
   dataset  = load_dataset("abdeldjalil-smaili/algerian-medical-llm-v3", token=HF_TOKEN, split="train")

   def get_q(s):
       try: return s['text'].split("### Input:")[1].split("### Response:")[0].strip()
       except: return s['text'][:200]
   def get_a(s):
       try: return s['text'].split("### Response:")[1].strip()
       except: return ""

   questions  = [get_q(s) for s in dataset]
   answers    = [get_a(s) for s in dataset]
   embeddings = embedder.encode(questions, batch_size=256,
                                 show_progress_bar=False, convert_to_numpy=True)
   index = faiss.IndexFlatL2(embeddings.shape[1])
   index.add(embeddings.astype(np.float32))

   bnb = BitsAndBytesConfig(
       load_in_4bit=True, bnb_4bit_quant_type="nf4",
       bnb_4bit_compute_dtype=torch.bfloat16
   )

   atlas_tok = AutoTokenizer.from_pretrained("MBZUAI-Paris/Atlas-Chat-9B", token=HF_TOKEN)
   atlas_mdl = AutoModelForCausalLM.from_pretrained(
       "MBZUAI-Paris/Atlas-Chat-9B", quantization_config=bnb,
       device_map="auto", token=HF_TOKEN
   )
   atlas_mdl.eval()

   llama_tok = AutoTokenizer.from_pretrained(
       "meta-llama/Llama-3.2-3B-Instruct", token=HF_TOKEN)
   llama_mdl = AutoModelForCausalLM.from_pretrained(
       "abdeldjalil-smaili/llama3b-algerian-medical", quantization_config=bnb,
       device_map="auto", token=HF_TOKEN
   )
   llama_mdl.eval()

   return (whisper_processor, whisper_model, embedder, index, questions, answers,
           atlas_tok, atlas_mdl, llama_tok, llama_mdl)


def transcribe(audio_path, processor, model):
   audio, _ = librosa.load(audio_path, sr=16000)
   inputs = processor(audio, sampling_rate=16000, return_tensors="pt").input_features
   inputs = inputs.to("cuda" if torch.cuda.is_available() else "cpu")
   with torch.no_grad():
       ids = model.generate(inputs, language="ar")
   return processor.batch_decode(ids, skip_special_tokens=True)[0]


def get_rag_context(question, embedder, index, questions, answers, k=3, threshold=8.0):
   q_emb = embedder.encode([question], convert_to_numpy=True)
   distances, indices = index.search(q_emb.astype(np.float32), k)
   contexts = []
   for dist, idx in zip(distances[0], indices[0]):
       if dist < threshold:
           contexts.append(f"Q: {questions[idx]}\nR: {answers[idx]}")
   return "\n\n".join(contexts)


def generate_response(question, model, tokenizer, embedder, index, questions, answers):
   context = get_rag_context(question, embedder, index, questions, answers)
   content = f"""أنت مساعد طبي يتكلم الدارجة الجزائرية فقط.
{f"استخدم هاد المعلومات باش تجاوب:{chr(10)}{context}{chr(10)}{chr(10)}" if context else ""}
قواعد مهمة:
- أجب بجملة واحدة أو جملتين فقط
- استخدم كلام بسيط مفهوم
- ما تذكرش فحوصات معقدة إلا كان السؤال خطير
- ركز على النصيحة العملية البسيطة

السؤال: {question}
الجواب بالدارجة الجزائرية:"""

   messages = [{"role": "user", "content": content}]
   prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
   inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
   inputs = {k: v.to("cuda" if torch.cuda.is_available() else "cpu") for k, v in inputs.items()}
   with torch.no_grad():
       outputs = model.generate(
           **inputs, max_new_tokens=60, temperature=0.2,
           do_sample=True, pad_token_id=tokenizer.eos_token_id,
           repetition_penalty=1.5
       )
   input_len = inputs["input_ids"].shape[1]
   return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()


def synthesize_tts(text, output_dir):
   os.makedirs(output_dir, exist_ok=True)
   os.system(f'habibi-tts_infer-cli --ref_audio "{REF_AUDIO}" --ref_text "{REF_TEXT}" --gen_text "{text}" --dialect ALG --output_dir "{output_dir}"')
   return f"{output_dir}/infer_cli_example.wav"


with st.spinner("⏳ جاري تحميل النماذج..."):
   (w_proc, w_mdl, embedder, index, questions, answers,
    atlas_tok, atlas_mdl, llama_tok, llama_mdl) = load_all_models()
st.success("✅ النماذج جاهزة!")

col1, col2 = st.columns([2, 1])

with col1:
   st.markdown("### 💬 المحادثة")

   chat_html = '<div class="chat-container">'
   if not st.session_state.history:
       chat_html += '<p style="text-align:center;color:#999;margin-top:150px;">اسأل سؤالك الطبي بالدارجة الجزائرية 🎤</p>'
   for msg in st.session_state.history:
       chat_html += f'<div style="text-align:right;font-size:0.75rem;opacity:0.7;margin-top:10px;">👤 أنت</div>'
       chat_html += f'<div class="msg-user">{msg["question"]}</div>'
       chat_html += f'<div style="font-size:0.75rem;opacity:0.7;">🤖 Atlas-Chat (Zero-Shot)</div>'
       chat_html += f'<div class="msg-atlas">{msg["atlas"]}</div>'
       chat_html += f'<div style="font-size:0.75rem;opacity:0.7;">🧠 LLaMA (Fine-Tuné)</div>'
       chat_html += f'<div class="msg-llama">{msg["llama"]}</div>'
       chat_html += '<hr style="border:none;border-top:1px solid #eee;margin:12px 0">'
   chat_html += '</div>'
   st.markdown(chat_html, unsafe_allow_html=True)

   if st.session_state.transcription:
       st.markdown(f"""
       <div class="last-question-box">
           📝 <b>آخر سؤال مفرَّغ:</b> {st.session_state.transcription}
       </div>
       """, unsafe_allow_html=True)


with col2:
   st.markdown("### 🎤 تسجيل السؤال")

   # Composant natif Streamlit 1.58 — plus stable que st_audiorec
   audio_input = st.audio_input("🎤 سجل سؤالك هنا")

   if audio_input is not None:
       st.session_state.last_audio = audio_input.read()

   transcription_placeholder = st.empty()

   if st.session_state.last_audio is not None:
       st.audio(st.session_state.last_audio, format='audio/wav')

       if st.button("🚀 ابدأ التحليل", type="primary", use_container_width=True):

           with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
               tmp.write(st.session_state.last_audio)
               tmp_path = tmp.name

           wav_path = tmp_path.replace(".wav", "_16k.wav")
           subprocess.run(
               ['ffmpeg', '-i', tmp_path, '-ar', '16000', '-ac', '1', wav_path, '-y'],
               capture_output=True
           )

           with st.spinner("🎤 جاري التفريغ..."):
               question = transcribe(wav_path, w_proc, w_mdl)
               st.session_state.transcription = question

           transcription_placeholder.markdown(f"""
           <div class="transcription-box">
               🎤 <b>سؤالك:</b> {question}
           </div>
           """, unsafe_allow_html=True)

           with st.spinner("🧠 Atlas-Chat يجاوب..."):
               response_atlas = generate_response(
                   question, atlas_mdl, atlas_tok, embedder, index, questions, answers
               )

           with st.spinner("🧠 LLaMA يجاوب..."):
               response_llama = generate_response(
                   question, llama_mdl, llama_tok, embedder, index, questions, answers
               )

           with st.spinner("🔊 جاري التوليد الصوتي..."):
               atlas_audio = synthesize_tts(response_atlas, "./outputs_atlas")
               llama_audio = synthesize_tts(response_llama, "./outputs_llama")

           st.session_state.history.append({
               "question": question,
               "atlas": response_atlas,
               "llama": response_llama,
               "atlas_audio": atlas_audio,
               "llama_audio": llama_audio
           })
           st.session_state.last_audio = None
           st.rerun()

   if st.session_state.history:
       last = st.session_state.history[-1]
       st.markdown("### 🔊 الردود الصوتية")
       st.markdown("**Atlas-Chat:**")
       if os.path.exists(last["atlas_audio"]):
           st.audio(last["atlas_audio"])
       st.markdown("**LLaMA Fine-Tuné:**")
       if os.path.exists(last["llama_audio"]):
           st.audio(last["llama_audio"])
       st.markdown("---")
       if st.button("🗑️ مسح المحادثة", use_container_width=True):
           st.session_state.history = []
           st.session_state.transcription = ""
           st.session_state.last_audio = None
           st.rerun()

In [ ]:
from pyngrok import ngrok

# Fermer tous les tunnels existants
tunnels = ngrok.get_tunnels()
for tunnel in tunnels:
    ngrok.disconnect(tunnel.public_url)
    print(f"Fermé: {tunnel.public_url}")

ngrok.kill()
print("✅ Tous les tunnels fermés")

In [ ]:
%%writefile app.py
import os
os.system("pip uninstall torchcodec -y -q")

import streamlit as st
import torch
import subprocess
import numpy as np
import faiss
from transformers import (
   WhisperProcessor, WhisperForConditionalGeneration,
   AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
)
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
from huggingface_hub import login
import librosa
import tempfile

from huggingface_hub import login login()
REF_AUDIO = "ref_voice.wav"
REF_TEXT  = "نيفي راهو يسيل"

st.set_page_config(page_title="المساعد الطبي الجزائري", page_icon="🏥", layout="wide")

st.markdown("""
<style>
   .header {
       background: linear-gradient(135deg, #1a6b8a, #0d4d6b);
       color: white; padding: 20px; border-radius: 12px;
       text-align: center; margin-bottom: 20px;
   }
   .header h1 { font-size: 2rem; margin: 0; }
   .header p  { font-size: 1rem; margin: 5px 0 0 0; opacity: 0.85; }
   .chat-container {
       background: white; border-radius: 12px; padding: 20px;
       min-height: 400px; max-height: 500px; overflow-y: auto;
       box-shadow: 0 2px 8px rgba(0,0,0,0.08); margin-bottom: 16px;
   }
   .msg-user {
       background: #1a6b8a; color: white; padding: 10px 16px;
       border-radius: 18px 18px 4px 18px; margin: 8px 0;
       max-width: 75%; margin-left: auto; text-align: right; direction: rtl;
   }
   .msg-atlas {
       background: #e8f4f8; color: #1a1a1a; padding: 10px 16px;
       border-radius: 18px 18px 18px 4px; margin: 8px 0;
       max-width: 75%; border-left: 4px solid #1a6b8a; direction: rtl;
   }
   .msg-llama {
       background: #e8f8ee; color: #1a1a1a; padding: 10px 16px;
       border-radius: 18px 18px 18px 4px; margin: 8px 0;
       max-width: 75%; border-left: 4px solid #2ecc71; direction: rtl;
   }
   .transcription-box {
       background: #fff8e1; border: 1px solid #ffc107;
       border-radius: 8px; padding: 12px; margin: 10px 0;
       direction: rtl; font-size: 1.1rem;
   }
   .last-question-box {
       background: #e3f2fd; border: 1px solid #1a6b8a;
       border-radius: 8px; padding: 12px; margin: 10px 0;
       direction: rtl; font-size: 1.1rem;
   }
</style>
""", unsafe_allow_html=True)

st.markdown("""
<div class="header">
   <h1>🏥 المساعد الطبي الجزائري</h1>
   <p>نظام ذكاء اصطناعي طبي بالدارجة الجزائرية</p>
</div>
""", unsafe_allow_html=True)

if "history" not in st.session_state:
   st.session_state.history = []
if "transcription" not in st.session_state:
   st.session_state.transcription = ""

@st.cache_resource
def load_all_models():
   login(token=HF_TOKEN)

   whisper_processor = WhisperProcessor.from_pretrained("openai/whisper-small")
   whisper_model = WhisperForConditionalGeneration.from_pretrained(
       "abdeldjalil-smaili/whisper-algerian-medical-final",
       subfolder="checkpoint-1000", token=HF_TOKEN
   ).to("cuda" if torch.cuda.is_available() else "cpu")
   whisper_model.eval()

   embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
   dataset  = load_dataset("abdeldjalil-smaili/algerian-medical-llm-v3", token=HF_TOKEN, split="train")

   def get_q(s):
       try: return s['text'].split("### Input:")[1].split("### Response:")[0].strip()
       except: return s['text'][:200]
   def get_a(s):
       try: return s['text'].split("### Response:")[1].strip()
       except: return ""

   questions  = [get_q(s) for s in dataset]
   answers    = [get_a(s) for s in dataset]
   embeddings = embedder.encode(questions, batch_size=256,
                                 show_progress_bar=False, convert_to_numpy=True)
   index = faiss.IndexFlatL2(embeddings.shape[1])
   index.add(embeddings.astype(np.float32))

   bnb = BitsAndBytesConfig(
       load_in_4bit=True, bnb_4bit_quant_type="nf4",
       bnb_4bit_compute_dtype=torch.bfloat16
   )

   atlas_tok = AutoTokenizer.from_pretrained("MBZUAI-Paris/Atlas-Chat-9B", token=HF_TOKEN)
   atlas_mdl = AutoModelForCausalLM.from_pretrained(
       "MBZUAI-Paris/Atlas-Chat-9B", quantization_config=bnb,
       device_map="auto", token=HF_TOKEN
   )
   atlas_mdl.eval()

   llama_tok = AutoTokenizer.from_pretrained(
       "meta-llama/Llama-3.2-3B-Instruct", token=HF_TOKEN)
   llama_mdl = AutoModelForCausalLM.from_pretrained(
       "abdeldjalil-smaili/llama3b-algerian-medical", quantization_config=bnb,
       device_map="auto", token=HF_TOKEN
   )
   llama_mdl.eval()

   return (whisper_processor, whisper_model, embedder, index, questions, answers,
           atlas_tok, atlas_mdl, llama_tok, llama_mdl)


def transcribe(audio_path, processor, model):
   audio, _ = librosa.load(audio_path, sr=16000)
   inputs = processor(audio, sampling_rate=16000, return_tensors="pt").input_features
   inputs = inputs.to("cuda" if torch.cuda.is_available() else "cpu")
   with torch.no_grad():
       ids = model.generate(inputs, language="ar")
   return processor.batch_decode(ids, skip_special_tokens=True)[0]


def get_rag_context(question, embedder, index, questions, answers, k=3, threshold=8.0):
   q_emb = embedder.encode([question], convert_to_numpy=True)
   distances, indices = index.search(q_emb.astype(np.float32), k)
   contexts = []
   for dist, idx in zip(distances[0], indices[0]):
       if dist < threshold:
           contexts.append(f"Q: {questions[idx]}\nR: {answers[idx]}")
   return "\n\n".join(contexts)


def generate_response(question, model, tokenizer, embedder, index, questions, answers):
   context = get_rag_context(question, embedder, index, questions, answers)
   content = f"""أنت مساعد طبي يتكلم الدارجة الجزائرية فقط.
{f"استخدم هاد المعلومات باش تجاوب:{chr(10)}{context}{chr(10)}{chr(10)}" if context else ""}
قواعد مهمة:
- أجب بجملة واحدة أو جملتين فقط
- استخدم كلام بسيط مفهوم
- ما تذكرش فحوصات معقدة إلا كان السؤال خطير
- ركز على النصيحة العملية البسيطة

السؤال: {question}
الجواب بالدارجة الجزائرية:"""

   messages = [{"role": "user", "content": content}]
   prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
   inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
   inputs = {k: v.to("cuda" if torch.cuda.is_available() else "cpu") for k, v in inputs.items()}
   with torch.no_grad():
       outputs = model.generate(
           **inputs, max_new_tokens=60, temperature=0.2,
           do_sample=True, pad_token_id=tokenizer.eos_token_id,
           repetition_penalty=1.5
       )
   input_len = inputs["input_ids"].shape[1]
   return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()


def synthesize_tts(text, output_dir):
   os.makedirs(output_dir, exist_ok=True)
   os.system(f'habibi-tts_infer-cli --ref_audio "{REF_AUDIO}" --ref_text "{REF_TEXT}" --gen_text "{text}" --dialect ALG --output_dir "{output_dir}"')
   return f"{output_dir}/infer_cli_example.wav"


with st.spinner("⏳ جاري تحميل النماذج..."):
   (w_proc, w_mdl, embedder, index, questions, answers,
    atlas_tok, atlas_mdl, llama_tok, llama_mdl) = load_all_models()
st.success("✅ النماذج جاهزة!")

col1, col2 = st.columns([2, 1])

with col1:
   st.markdown("### 💬 المحادثة")

   chat_html = '<div class="chat-container">'
   if not st.session_state.history:
       chat_html += '<p style="text-align:center;color:#999;margin-top:150px;">اسأل سؤالك الطبي بالدارجة الجزائرية 🎤</p>'
   for msg in st.session_state.history:
       chat_html += f'<div style="text-align:right;font-size:0.75rem;opacity:0.7;margin-top:10px;">👤 أنت</div>'
       chat_html += f'<div class="msg-user">{msg["question"]}</div>'
       chat_html += f'<div style="font-size:0.75rem;opacity:0.7;">🤖 Atlas-Chat (Zero-Shot)</div>'
       chat_html += f'<div class="msg-atlas">{msg["atlas"]}</div>'
       chat_html += f'<div style="font-size:0.75rem;opacity:0.7;">🧠 LLaMA (Fine-Tuné)</div>'
       chat_html += f'<div class="msg-llama">{msg["llama"]}</div>'
       chat_html += '<hr style="border:none;border-top:1px solid #eee;margin:12px 0">'
   chat_html += '</div>'
   st.markdown(chat_html, unsafe_allow_html=True)

   if st.session_state.transcription:
       st.markdown(f"""
       <div class="last-question-box">
           📝 <b>آخر سؤال مفرَّغ:</b> {st.session_state.transcription}
       </div>
       """, unsafe_allow_html=True)

with col2:
   st.markdown("### 🎤 تسجيل السؤال")

   audio_input = st.audio_input("🎤 سجل سؤالك هنا")

   transcription_placeholder = st.empty()

   if audio_input is not None:
       audio_bytes = audio_input.read()
       st.audio(audio_bytes, format='audio/wav')

       if st.button("🚀 ابدأ التحليل", type="primary", use_container_width=True):

           with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
               tmp.write(audio_bytes)
               tmp_path = tmp.name

           wav_path = tmp_path.replace(".wav", "_16k.wav")
           subprocess.run(
               ['ffmpeg', '-i', tmp_path, '-ar', '16000', '-ac', '1', wav_path, '-y'],
               capture_output=True
           )

           with st.spinner("🎤 جاري التفريغ..."):
               question = transcribe(wav_path, w_proc, w_mdl)
               st.session_state.transcription = question

           transcription_placeholder.markdown(f"""
           <div class="transcription-box">
               🎤 <b>سؤالك:</b> {question}
           </div>
           """, unsafe_allow_html=True)

           with st.spinner("🧠 Atlas-Chat يجاوب..."):
               response_atlas = generate_response(
                   question, atlas_mdl, atlas_tok, embedder, index, questions, answers
               )

           with st.spinner("🧠 LLaMA يجاوب..."):
               response_llama = generate_response(
                   question, llama_mdl, llama_tok, embedder, index, questions, answers
               )

           with st.spinner("🔊 جاري التوليد الصوتي..."):
               atlas_audio = synthesize_tts(response_atlas, "./outputs_atlas")
               llama_audio = synthesize_tts(response_llama, "./outputs_llama")

           st.session_state.history.append({
               "question": question,
               "atlas": response_atlas,
               "llama": response_llama,
               "atlas_audio": atlas_audio,
               "llama_audio": llama_audio
           })
           st.rerun()

   if st.session_state.history:
       last = st.session_state.history[-1]
       st.markdown("### 🔊 الردود الصوتية")
       st.markdown("**Atlas-Chat:**")
       if os.path.exists(last["atlas_audio"]):
           st.audio(last["atlas_audio"])
       st.markdown("**LLaMA Fine-Tuné:**")
       if os.path.exists(last["llama_audio"]):
           st.audio(last["llama_audio"])
       st.markdown("---")
       if st.button("🗑️ مسح المحادثة", use_container_width=True):
           st.session_state.history = []
           st.session_state.transcription = ""
           st.rerun()

In [ ]:
%%writefile app.py
import os
os.system("pip install habibi-tts -q")
os.system("pip uninstall torchcodec -y -q")

import streamlit as st
import torch
import subprocess
import numpy as np
import faiss
import time
from transformers import (
   WhisperProcessor, WhisperForConditionalGeneration,
   AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
)
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
from huggingface_hub import login
import librosa
import tempfile

from huggingface_hub import login
login()
REF_AUDIO = "ref_voice.wav"
REF_TEXT  = "نيفي راهو يسيل"

st.set_page_config(page_title="المساعد الطبي الجزائري", page_icon="🏥", layout="wide")

st.markdown("""
<style>
   .header {
       background: linear-gradient(135deg, #1a6b8a, #0d4d6b);
       color: white; padding: 20px; border-radius: 12px;
       text-align: center; margin-bottom: 20px;
   }
   .header h1 { font-size: 2rem; margin: 0; }
   .header p  { font-size: 1rem; margin: 5px 0 0 0; opacity: 0.85; }
   .chat-container {
       background: white; border-radius: 12px; padding: 20px;
       min-height: 400px; max-height: 500px; overflow-y: auto;
       box-shadow: 0 2px 8px rgba(0,0,0,0.08); margin-bottom: 16px;
   }
   .msg-user {
       background: #1a6b8a; color: white; padding: 10px 16px;
       border-radius: 18px 18px 4px 18px; margin: 8px 0;
       max-width: 75%; margin-left: auto; text-align: right; direction: rtl;
   }
   .msg-atlas {
       background: #e8f4f8; color: #1a1a1a; padding: 10px 16px;
       border-radius: 18px 18px 18px 4px; margin: 8px 0;
       max-width: 75%; border-left: 4px solid #1a6b8a; direction: rtl;
   }
   .msg-llama-darja {
       background: #e8f8ee; color: #1a1a1a; padding: 10px 16px;
       border-radius: 18px 18px 18px 4px; margin: 8px 0;
       max-width: 75%; border-left: 4px solid #2ecc71; direction: rtl;
   }
   .msg-llama-maqa {
       background: #fef9e7; color: #1a1a1a; padding: 10px 16px;
       border-radius: 18px 18px 18px 4px; margin: 8px 0;
       max-width: 75%; border-left: 4px solid #f39c12; direction: rtl;
   }
   .transcription-box {
       background: #fff8e1; border: 1px solid #ffc107;
       border-radius: 8px; padding: 12px; margin: 10px 0;
       direction: rtl; font-size: 1.1rem;
   }
   .last-question-box {
       background: #e3f2fd; border: 1px solid #1a6b8a;
       border-radius: 8px; padding: 12px; margin: 10px 0;
       direction: rtl; font-size: 1.1rem;
   }
</style>
""", unsafe_allow_html=True)

st.markdown("""
<div class="header">
   <h1>🏥 المساعد الطبي الجزائري</h1>
   <p>نظام ذكاء اصطناعي طبي بالدارجة الجزائرية</p>
</div>
""", unsafe_allow_html=True)

if "history" not in st.session_state:
   st.session_state.history = []
if "transcription" not in st.session_state:
   st.session_state.transcription = ""

@st.cache_resource
def load_all_models():
   login(token=HF_TOKEN)

   # ── Whisper ──────────────────────────────────────────
   whisper_processor = WhisperProcessor.from_pretrained("openai/whisper-small")
   whisper_model = WhisperForConditionalGeneration.from_pretrained(
       "abdeldjalil-smaili/whisper-algerian-medical-final",
       subfolder="checkpoint-1000", token=HF_TOKEN
   ).to("cuda" if torch.cuda.is_available() else "cpu")
   whisper_model.eval()

   # ── RAG (dataset Darja) ───────────────────────────────
   embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
   dataset  = load_dataset("abdeldjalil-smaili/algerian-medical-llm-v3", token=HF_TOKEN, split="train")

   def get_q(s):
       try: return s['text'].split("### Input:")[1].split("### Response:")[0].strip()
       except: return s['text'][:200]
   def get_a(s):
       try: return s['text'].split("### Response:")[1].strip()
       except: return ""

   questions  = [get_q(s) for s in dataset]
   answers    = [get_a(s) for s in dataset]
   embeddings = embedder.encode(questions, batch_size=256,
                                show_progress_bar=False, convert_to_numpy=True)
   index = faiss.IndexFlatL2(embeddings.shape[1])
   index.add(embeddings.astype(np.float32))

   bnb = BitsAndBytesConfig(
       load_in_4bit=True, bnb_4bit_quant_type="nf4",
       bnb_4bit_compute_dtype=torch.bfloat16
   )

   # ── Atlas-Chat 9B (Zero-Shot) ─────────────────────────
   atlas_tok = AutoTokenizer.from_pretrained("MBZUAI-Paris/Atlas-Chat-9B", token=HF_TOKEN)
   atlas_mdl = AutoModelForCausalLM.from_pretrained(
       "MBZUAI-Paris/Atlas-Chat-9B", quantization_config=bnb,
       device_map="auto", token=HF_TOKEN
   )
   atlas_mdl.eval()

   # ── LLaMA Darja (fine-tuné dialecte algérien) ─────────
   llama_darja_tok = AutoTokenizer.from_pretrained(
       "meta-llama/Llama-3.2-3B-Instruct", token=HF_TOKEN)
   llama_darja_mdl = AutoModelForCausalLM.from_pretrained(
       "abdeldjalil-smaili/llama3b-algerian-medical", quantization_config=bnb,
       device_map="auto", token=HF_TOKEN
   )
   llama_darja_mdl.eval()

   # ── LLaMA MAQA (fine-tuné arabe médical MSA) ──────────
   llama_maqa_tok = AutoTokenizer.from_pretrained(
       "unsloth/Llama-3.2-3B-Instruct",
       token=HF_TOKEN,
       clean_up_tokenization_spaces=False
   )
   llama_maqa_mdl = AutoModelForCausalLM.from_pretrained(
       "abdeldjalil-smaili/llama3-maqa-medical",
       quantization_config=bnb,
       device_map="auto",
       token=HF_TOKEN,
       torch_dtype=torch.bfloat16,
   )
   llama_maqa_mdl.eval()

   return (whisper_processor, whisper_model, embedder, index, questions, answers,
           atlas_tok, atlas_mdl,
           llama_darja_tok, llama_darja_mdl,
           llama_maqa_tok, llama_maqa_mdl)


def transcribe(audio_path, processor, model):
   audio, _ = librosa.load(audio_path, sr=16000)
   inputs = processor(audio, sampling_rate=16000, return_tensors="pt").input_features
   inputs = inputs.to("cuda" if torch.cuda.is_available() else "cpu")
   with torch.no_grad():
       ids = model.generate(inputs, language="ar")
   return processor.batch_decode(ids, skip_special_tokens=True)[0]


def get_rag_context(question, embedder, index, questions, answers, k=3, threshold=8.0):
   q_emb = embedder.encode([question], convert_to_numpy=True)
   distances, indices = index.search(q_emb.astype(np.float32), k)
   contexts = []
   for dist, idx in zip(distances[0], indices[0]):
       if dist < threshold:
           contexts.append(f"Q: {questions[idx]}\nR: {answers[idx]}")
   return "\n\n".join(contexts)


def generate_response_darja(question, model, tokenizer, embedder, index, questions, answers):
   context = get_rag_context(question, embedder, index, questions, answers)
   content = f"""أنت مساعد طبي يتكلم الدارجة الجزائرية فقط.
{f"استخدم هاد المعلومات باش تجاوب:{chr(10)}{context}{chr(10)}{chr(10)}" if context else ""}
قواعد مهمة:
- أجب بجملة واحدة أو جملتين فقط
- استخدم كلام بسيط مفهوم
- ما تذكرش فحوصات معقدة إلا كان السؤال خطير
- ركز على النصيحة العملية البسيطة

السؤال: {question}
الجواب بالدارجة الجزائرية:"""

   messages = [{"role": "user", "content": content}]
   prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
   inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
   inputs = {k: v.to("cuda" if torch.cuda.is_available() else "cpu") for k, v in inputs.items()}
   with torch.no_grad():
       outputs = model.generate(
           **inputs, max_new_tokens=60, temperature=0.2,
           do_sample=True, pad_token_id=tokenizer.eos_token_id,
           repetition_penalty=1.5
       )
   input_len = inputs["input_ids"].shape[1]
   return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()


def generate_response_maqa(question, model, tokenizer):
   content = f"""أنت مساعد طبي متخصص. أجب على الأسئلة الطبية بدقة وباللغة العربية الفصحى.
قواعد مهمة:
- أجب بشكل مباشر ومختصر
- استخدم المصطلحات الطبية الصحيحة
- انصح بمراجعة الطبيب عند الضرورة

السؤال: {question}
الجواب:"""

   messages = [{"role": "user", "content": content}]
   prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
   inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
   inputs = {k: v.to("cuda" if torch.cuda.is_available() else "cpu") for k, v in inputs.items()}
   with torch.no_grad():
       outputs = model.generate(
           **inputs, max_new_tokens=150, temperature=0.7,
           do_sample=True, pad_token_id=tokenizer.eos_token_id,
           repetition_penalty=1.3
       )
   input_len = inputs["input_ids"].shape[1]
   return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()


def synthesize_tts(text, output_path):
   """Génère un fichier audio TTS avec chemin unique via --output_file."""
   output_dir  = os.path.dirname(output_path)
   output_file = os.path.basename(output_path)
   os.makedirs(output_dir, exist_ok=True)
   os.system(
       f'habibi-tts_infer-cli '
       f'--ref_audio "{REF_AUDIO}" '
       f'--ref_text "{REF_TEXT}" '
       f'--gen_text "{text}" '
       f'--dialect ALG '
       f'--output_dir "{output_dir}" '
       f'--output_file "{output_file}"'
   )
   return output_path


with st.spinner("⏳ جاري تحميل النماذج..."):
   (w_proc, w_mdl, embedder, index, questions, answers,
    atlas_tok, atlas_mdl,
    llama_darja_tok, llama_darja_mdl,
    llama_maqa_tok, llama_maqa_mdl) = load_all_models()
st.success("✅ النماذج جاهزة!")

col1, col2 = st.columns([2, 1])

with col1:
   st.markdown("### 💬 المحادثة")

   chat_html = '<div class="chat-container">'
   if not st.session_state.history:
       chat_html += '<p style="text-align:center;color:#999;margin-top:150px;">اسأل سؤالك الطبي بالدارجة الجزائرية 🎤</p>'
   for msg in st.session_state.history:
       chat_html += f'<div style="text-align:right;font-size:0.75rem;opacity:0.7;margin-top:10px;">👤 أنت</div>'
       chat_html += f'<div class="msg-user">{msg["question"]}</div>'
       chat_html += f'<div style="font-size:0.75rem;opacity:0.7;">🔵 Atlas-Chat (Zero-Shot)</div>'
       chat_html += f'<div class="msg-atlas">{msg["atlas"]}</div>'
       chat_html += f'<div style="font-size:0.75rem;opacity:0.7;">🟢 LLaMA Darja (Fine-Tuné)</div>'
       chat_html += f'<div class="msg-llama-darja">{msg["llama_darja"]}</div>'
       chat_html += f'<div style="font-size:0.75rem;opacity:0.7;">🟡 LLaMA MAQA (Fine-Tuné MSA)</div>'
       chat_html += f'<div class="msg-llama-maqa">{msg["llama_maqa"]}</div>'
       chat_html += '<hr style="border:none;border-top:1px solid #eee;margin:12px 0">'
   chat_html += '</div>'
   st.markdown(chat_html, unsafe_allow_html=True)

   if st.session_state.transcription:
       st.markdown(f"""
       <div class="last-question-box">
           📝 <b>آخر سؤال مفرَّغ:</b> {st.session_state.transcription}
       </div>
       """, unsafe_allow_html=True)

with col2:
   st.markdown("### 🎤 تسجيل السؤال")

   audio_input = st.audio_input("🎤 سجل سؤالك هنا")
   transcription_placeholder = st.empty()

   if audio_input is not None:
       audio_bytes = audio_input.read()
       st.audio(audio_bytes, format='audio/wav')

       if st.button("🚀 ابدأ التحليل", type="primary", use_container_width=True):

           with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
               tmp.write(audio_bytes)
               tmp_path = tmp.name

           wav_path = tmp_path.replace(".wav", "_16k.wav")
           subprocess.run(
               ['ffmpeg', '-i', tmp_path, '-ar', '16000', '-ac', '1', wav_path, '-y'],
               capture_output=True
           )

           with st.spinner("🎤 جاري التفريغ..."):
               question = transcribe(wav_path, w_proc, w_mdl)
               st.session_state.transcription = question

           transcription_placeholder.markdown(f"""
           <div class="transcription-box">
               🎤 <b>سؤالك:</b> {question}
           </div>
           """, unsafe_allow_html=True)

           with st.spinner("🔵 Atlas-Chat يجاوب..."):
               response_atlas = generate_response_darja(
                   question, atlas_mdl, atlas_tok, embedder, index, questions, answers
               )

           with st.spinner("🟢 LLaMA Darja يجاوب..."):
               response_llama_darja = generate_response_darja(
                   question, llama_darja_mdl, llama_darja_tok, embedder, index, questions, answers
               )

           with st.spinner("🟡 LLaMA MAQA يجاوب..."):
               response_llama_maqa = generate_response_maqa(
                   question, llama_maqa_mdl, llama_maqa_tok
               )

           ts = int(time.time())
           with st.spinner("🔊 جاري التوليد الصوتي..."):
               atlas_audio       = synthesize_tts(response_atlas,       f"./outputs/atlas_{ts}.wav")
               llama_darja_audio = synthesize_tts(response_llama_darja, f"./outputs/llama_darja_{ts}.wav")
               llama_maqa_audio  = synthesize_tts(response_llama_maqa,  f"./outputs/llama_maqa_{ts}.wav")

           st.session_state.history.append({
               "question"         : question,
               "atlas"            : response_atlas,
               "llama_darja"      : response_llama_darja,
               "llama_maqa"       : response_llama_maqa,
               "atlas_audio"      : atlas_audio,
               "llama_darja_audio": llama_darja_audio,
               "llama_maqa_audio" : llama_maqa_audio,
           })
           st.rerun()

   if st.session_state.history:
       last = st.session_state.history[-1]
       st.markdown("### 🔊 الردود الصوتية")
       st.markdown("**🔵 Atlas-Chat:**")
       if os.path.exists(last["atlas_audio"]):
           st.audio(last["atlas_audio"])
       else:
           st.warning("الصوت غير متاح")
       st.markdown("**🟢 LLaMA Darja:**")
       if os.path.exists(last["llama_darja_audio"]):
           st.audio(last["llama_darja_audio"])
       else:
           st.warning("الصوت غير متاح")
       st.markdown("**🟡 LLaMA MAQA:**")
       if os.path.exists(last["llama_maqa_audio"]):
           st.audio(last["llama_maqa_audio"])
       else:
           st.warning("الصوت غير متاح")
       st.markdown("---")
       if st.button("🗑️ مسح المحادثة", use_container_width=True):
           st.session_state.history = []
           st.session_state.transcription = ""
           st.rerun()

In [ ]:
%%writefile app.py
import os
os.system("pip uninstall torchcodec -y -q")

import streamlit as st
import torch
import subprocess
import numpy as np
import faiss
from transformers import (
    WhisperProcessor, WhisperForConditionalGeneration,
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
)
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
from huggingface_hub import login
import librosa
import tempfile

from huggingface_hub import login login()
REF_AUDIO = "ref_voice.wav"
REF_TEXT  = "نيفي راهو يسيل"

st.set_page_config(page_title="المساعد الطبي الجزائري", page_icon="🏥", layout="wide")

st.markdown("""
<style>
    .header {
        background: linear-gradient(135deg, #1a6b8a, #0d4d6b);
        color: white; padding: 20px; border-radius: 12px;
        text-align: center; margin-bottom: 20px;
    }
    .header h1 { font-size: 2rem; margin: 0; }
    .header p  { font-size: 1rem; margin: 5px 0 0 0; opacity: 0.85; }
    .chat-container {
        background: white; border-radius: 12px; padding: 20px;
        min-height: 400px; max-height: 600px; overflow-y: auto;
        box-shadow: 0 2px 8px rgba(0,0,0,0.08); margin-bottom: 16px;
    }
    .msg-user {
        background: #1a6b8a; color: white; padding: 10px 16px;
        border-radius: 18px 18px 4px 18px; margin: 8px 0;
        max-width: 75%; margin-left: auto; text-align: right; direction: rtl;
    }
    .msg-atlas {
        background: #e8f4f8; color: #1a1a1a; padding: 10px 16px;
        border-radius: 18px 18px 18px 4px; margin: 8px 0;
        max-width: 75%; border-left: 4px solid #1a6b8a; direction: rtl;
    }
    .msg-llama {
        background: #e8f8ee; color: #1a1a1a; padding: 10px 16px;
        border-radius: 18px 18px 18px 4px; margin: 8px 0;
        max-width: 75%; border-left: 4px solid #2ecc71; direction: rtl;
    }
    .msg-maqa {
        background: #f3e8ff; color: #1a1a1a; padding: 10px 16px;
        border-radius: 18px 18px 18px 4px; margin: 8px 0;
        max-width: 75%; border-left: 4px solid #8e44ad; direction: rtl;
    }
    .transcription-box {
        background: #fff8e1; border: 1px solid #ffc107;
        border-radius: 8px; padding: 12px; margin: 10px 0;
        direction: rtl; font-size: 1.1rem;
    }
    .last-question-box {
        background: #e3f2fd; border: 1px solid #1a6b8a;
        border-radius: 8px; padding: 12px; margin: 10px 0;
        direction: rtl; font-size: 1.1rem;
    }
</style>
""", unsafe_allow_html=True)

st.markdown("""
<div class="header">
    <h1>🏥 المساعد الطبي الجزائري</h1>
    <p>نظام ذكاء اصطناعي طبي بالدارجة الجزائرية</p>
</div>
""", unsafe_allow_html=True)

if "history" not in st.session_state:
    st.session_state.history = []
if "transcription" not in st.session_state:
    st.session_state.transcription = ""


@st.cache_resource
def load_all_models():
    login(token=HF_TOKEN)

    # ── Whisper ──────────────────────────────────────────────────────────────
    whisper_processor = WhisperProcessor.from_pretrained("openai/whisper-small")
    whisper_model = WhisperForConditionalGeneration.from_pretrained(
        "abdeldjalil-smaili/whisper-algerian-medical-final",
        subfolder="checkpoint-1000", token=HF_TOKEN
    ).to("cuda" if torch.cuda.is_available() else "cpu")
    whisper_model.eval()

    # ── RAG (FAISS) ──────────────────────────────────────────────────────────
    embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
    dataset  = load_dataset("abdeldjalil-smaili/algerian-medical-llm-v3", token=HF_TOKEN, split="train")

    def get_q(s):
        try:    return s['text'].split("### Input:")[1].split("### Response:")[0].strip()
        except: return s['text'][:200]

    def get_a(s):
        try:    return s['text'].split("### Response:")[1].strip()
        except: return ""

    questions  = [get_q(s) for s in dataset]
    answers    = [get_a(s) for s in dataset]
    embeddings = embedder.encode(questions, batch_size=256,
                                 show_progress_bar=False, convert_to_numpy=True)
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings.astype(np.float32))

    # ── 4-bit config partagée ────────────────────────────────────────────────
    bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )

    # ── Modèle 1 : Atlas-Chat 9B (zero-shot) ────────────────────────────────
    atlas_tok = AutoTokenizer.from_pretrained("MBZUAI-Paris/Atlas-Chat-9B", token=HF_TOKEN)
    atlas_mdl = AutoModelForCausalLM.from_pretrained(
        "MBZUAI-Paris/Atlas-Chat-9B", quantization_config=bnb,
        device_map="auto", token=HF_TOKEN
    )
    atlas_mdl.eval()

    # ── Modèle 2 : LLaMA Darja fine-tuné ────────────────────────────────────
    llama_darja_tok = AutoTokenizer.from_pretrained(
        "meta-llama/Llama-3.2-3B-Instruct", token=HF_TOKEN)
    llama_darja_mdl = AutoModelForCausalLM.from_pretrained(
        "abdeldjalil-smaili/llama3b-algerian-medical", quantization_config=bnb,
        device_map="auto", token=HF_TOKEN
    )
    llama_darja_mdl.eval()

    # ── Modèle 3 : LLaMA MAQA fine-tuné (merged fp16) ───────────────────────
    llama_maqa_tok = AutoTokenizer.from_pretrained(
        "abdeldjalil-smaili/llama3-maqa-medical", token=HF_TOKEN
    )
    llama_maqa_mdl = AutoModelForCausalLM.from_pretrained(
        "abdeldjalil-smaili/llama3-maqa-medical", quantization_config=bnb,
        device_map="auto", token=HF_TOKEN
    )
    llama_maqa_mdl.eval()

    return (whisper_processor, whisper_model,
            embedder, index, questions, answers,
            atlas_tok, atlas_mdl,
            llama_darja_tok, llama_darja_mdl,
            llama_maqa_tok, llama_maqa_mdl)


# ── Transcription Whisper ────────────────────────────────────────────────────
def transcribe(audio_path, processor, model):
    audio, _ = librosa.load(audio_path, sr=16000)
    inputs = processor(audio, sampling_rate=16000, return_tensors="pt").input_features
    inputs = inputs.to("cuda" if torch.cuda.is_available() else "cpu")
    with torch.no_grad():
        ids = model.generate(inputs, language="ar")
    return processor.batch_decode(ids, skip_special_tokens=True)[0]


# ── RAG context ──────────────────────────────────────────────────────────────
def get_rag_context(question, embedder, index, questions, answers, k=3, threshold=8.0):
    q_emb = embedder.encode([question], convert_to_numpy=True)
    distances, indices = index.search(q_emb.astype(np.float32), k)
    contexts = []
    for dist, idx in zip(distances[0], indices[0]):
        if dist < threshold:
            contexts.append(f"Q: {questions[idx]}\nR: {answers[idx]}")
    return "\n\n".join(contexts)


# ── Inférence Atlas-Chat & LLaMA Darja (prompt Darja, params courts) ─────────
def generate_response_darja(question, model, tokenizer, embedder, index, questions, answers):
    context = get_rag_context(question, embedder, index, questions, answers)
    content = f"""أنت مساعد طبي يتكلم الدارجة الجزائرية فقط.
{f"استخدم هاد المعلومات باش تجاوب:{chr(10)}{context}{chr(10)}{chr(10)}" if context else ""}
قواعد مهمة:
- أجب بجملة واحدة أو جملتين فقط
- استخدم كلام بسيط مفهوم
- ما تذكرش فحوصات معقدة إلا كان السؤال خطير
- ركز على النصيحة العملية البسيطة

السؤال: {question}
الجواب بالدارجة الجزائرية:"""

    messages = [{"role": "user", "content": content}]
    prompt   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs   = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    inputs   = {k: v.to("cuda" if torch.cuda.is_available() else "cpu") for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens    = 60,
            temperature       = 0.2,
            do_sample         = True,
            pad_token_id      = tokenizer.eos_token_id,
            repetition_penalty= 1.5,
        )
    input_len = inputs["input_ids"].shape[1]
    return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()


# ── Inférence LLaMA MAQA — fidèle au notebook ────────────────────────────────
MAQA_SYSTEM_PROMPT = (
    "أنت مساعد طبي متخصص. أجب على الأسئلة الطبية "
    "بدقة وباللغة العربية الفصحى."
)

def generate_response_maqa(question, model, tokenizer):
    """
    Inférence identique au notebook de fine-tuning :
    - system prompt MSA
    - max_new_tokens=200, temperature=0.7, top_p=0.9, repetition_penalty=1.1
    - extraction du texte via .split("assistant")[-1].strip()
    Pas de RAG injection ici (le modèle a été entraîné sans RAG).
    """
    messages = [
        {"role": "system", "content": MAQA_SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    inputs = {k: v.to("cuda" if torch.cuda.is_available() else "cpu") for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens     = 200,
            temperature        = 0.7,
            top_p              = 0.9,
            repetition_penalty = 1.1,
            do_sample          = True,
            pad_token_id       = tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded.split("assistant")[-1].strip()


# ── TTS ──────────────────────────────────────────────────────────────────────
def synthesize_tts(text, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    os.system(
        f'habibi-tts_infer-cli --ref_audio "{REF_AUDIO}" --ref_text "{REF_TEXT}" '
        f'--gen_text "{text}" --dialect ALG --output_dir "{output_dir}"'
    )
    return f"{output_dir}/infer_cli_example.wav"


# ── Chargement des modèles ────────────────────────────────────────────────────
with st.spinner("⏳ جاري تحميل النماذج..."):
    (w_proc, w_mdl,
     embedder, index, questions, answers,
     atlas_tok, atlas_mdl,
     llama_darja_tok, llama_darja_mdl,
     llama_maqa_tok, llama_maqa_mdl) = load_all_models()
st.success("✅ النماذج جاهزة!")


# ── UI ────────────────────────────────────────────────────────────────────────
col1, col2 = st.columns([2, 1])

with col1:
    st.markdown("### 💬 المحادثة")

    chat_html = '<div class="chat-container">'
    if not st.session_state.history:
        chat_html += '<p style="text-align:center;color:#999;margin-top:150px;">اسأل سؤالك الطبي بالدارجة الجزائرية 🎤</p>'
    for msg in st.session_state.history:
        chat_html += '<div style="text-align:right;font-size:0.75rem;opacity:0.7;margin-top:10px;">👤 أنت</div>'
        chat_html += f'<div class="msg-user">{msg["question"]}</div>'

        chat_html += '<div style="font-size:0.75rem;opacity:0.7;">🤖 Atlas-Chat (Zero-Shot + RAG)</div>'
        chat_html += f'<div class="msg-atlas">{msg["atlas"]}</div>'

        chat_html += '<div style="font-size:0.75rem;opacity:0.7;">🟢 LLaMA Darja (Fine-Tuné + RAG)</div>'
        chat_html += f'<div class="msg-llama">{msg["llama"]}</div>'

        chat_html += '<div style="font-size:0.75rem;opacity:0.7;">🟣 LLaMA MAQA (Fine-Tuné MSA)</div>'
        chat_html += f'<div class="msg-maqa">{msg["maqa"]}</div>'

        chat_html += '<hr style="border:none;border-top:1px solid #eee;margin:12px 0">'
    chat_html += '</div>'
    st.markdown(chat_html, unsafe_allow_html=True)

    if st.session_state.transcription:
        st.markdown(f"""
        <div class="last-question-box">
            📝 <b>آخر سؤال مفرَّغ:</b> {st.session_state.transcription}
        </div>
        """, unsafe_allow_html=True)

with col2:
    st.markdown("### 🎤 تسجيل السؤال")

    audio_input = st.audio_input("🎤 سجل سؤالك هنا")
    transcription_placeholder = st.empty()

    if audio_input is not None:
        audio_bytes = audio_input.read()
        st.audio(audio_bytes, format='audio/wav')

        if st.button("🚀 ابدأ التحليل", type="primary", use_container_width=True):

            # Sauvegarde audio temporaire
            with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
                tmp.write(audio_bytes)
                tmp_path = tmp.name

            wav_path = tmp_path.replace(".wav", "_16k.wav")
            subprocess.run(
                ['ffmpeg', '-i', tmp_path, '-ar', '16000', '-ac', '1', wav_path, '-y'],
                capture_output=True
            )

            # Transcription
            with st.spinner("🎤 جاري التفريغ..."):
                question = transcribe(wav_path, w_proc, w_mdl)
                st.session_state.transcription = question

            transcription_placeholder.markdown(f"""
            <div class="transcription-box">
                🎤 <b>سؤالك:</b> {question}
            </div>
            """, unsafe_allow_html=True)

            # Génération des 3 réponses
            with st.spinner("🤖 Atlas-Chat يجاوب..."):
                response_atlas = generate_response_darja(
                    question, atlas_mdl, atlas_tok, embedder, index, questions, answers
                )

            with st.spinner("🟢 LLaMA Darja يجاوب..."):
                response_llama = generate_response_darja(
                    question, llama_darja_mdl, llama_darja_tok, embedder, index, questions, answers
                )

            with st.spinner("🟣 LLaMA MAQA يجاوب..."):
                response_maqa = generate_response_maqa(
                    question, llama_maqa_mdl, llama_maqa_tok
                )

            # TTS pour les 3 réponses
            with st.spinner("🔊 جاري التوليد الصوتي..."):
                atlas_audio = synthesize_tts(response_atlas, "./outputs_atlas")
                llama_audio = synthesize_tts(response_llama, "./outputs_llama")
                maqa_audio  = synthesize_tts(response_maqa,  "./outputs_maqa")

            st.session_state.history.append({
                "question"   : question,
                "atlas"      : response_atlas,
                "llama"      : response_llama,
                "maqa"       : response_maqa,
                "atlas_audio": atlas_audio,
                "llama_audio": llama_audio,
                "maqa_audio" : maqa_audio,
            })
            st.rerun()

    if st.session_state.history:
        last = st.session_state.history[-1]
        st.markdown("### 🔊 الردود الصوتية")

        st.markdown("**🤖 Atlas-Chat:**")
        if os.path.exists(last["atlas_audio"]):
            st.audio(last["atlas_audio"])

        st.markdown("**🟢 LLaMA Darja:**")
        if os.path.exists(last["llama_audio"]):
            st.audio(last["llama_audio"])

        st.markdown("**🟣 LLaMA MAQA:**")
        if os.path.exists(last["maqa_audio"]):
            st.audio(last["maqa_audio"])

        st.markdown("---")
        if st.button("🗑️ مسح المحادثة", use_container_width=True):
            st.session_state.history = []
            st.session_state.transcription = ""
            st.rerun()

In [ ]:
from pyngrok import ngrok, conf
import threading, os, time

conf.get_default().auth_token = "3EXbAhX6eMVMPfO6Og7iliM66Jj_67BvD1QbUD77p1Doa64WK"

os.system("pkill -f streamlit")
time.sleep(2)

def run():
    os.system("streamlit run app.py --server.port 8501 --server.headless true")

threading.Thread(target=run, daemon=True).start()
time.sleep(8)

# Force HTTPS
url = ngrok.connect(8501, bind_tls=True)
print(f"🌐 {url.public_url}?ngrok-skip-browser-warning=true")

In [ ]:
# Fermer tous les tunnels ngrok existants
from pyngrok import ngrok

ngrok.kill()  # tuer tous les tunnels actifs
print("✓ Tunnels fermés")

In [ ]:
os.system("pkill -f streamlit")
import time
time.sleep(2)

import threading
from pyngrok import ngrok, conf

conf.get_default().auth_token = "3EXbAhX6eMVMPfO6Og7iliM66Jj_67BvD1QbUD77p1Doa64WK"

def run():
    os.system("streamlit run app.py --server.port 8501 --server.headless true")

threading.Thread(target=run, daemon=True).start()
time.sleep(8)

url = ngrok.connect(8501)
print(f"🌐 {url.public_url}?ngrok-skip-browser-warning=true")

In [ ]:
from huggingface_hub import list_repo_files
files = list(list_repo_files("abdeldjalil-smaili/llama3-maqa-medical", token=HF_TOKEN))
print(files)

In [ ]:
# Cellule de debug TTS dans Colab
import os

# Test direct
os.system('habibi-tts_infer-cli --ref_audio "ref_voice.wav" --ref_text "نيفي راهو يسيل" --gen_text "راسي يوجعني" --dialect ALG --output_dir "./test_tts"')

# Voir ce qui a été généré
os.listdir("./test_tts") if os.path.exists("./test_tts") else "dossier non créé"

In [ ]:
# Réinstaller habibi-tts
!pip install habibi-tts -q

# Vérifier l'installation
!habibi-tts_infer-cli --help

In [ ]:
# Debug TTS dans Colab — simuler exactement ce que fait l'app
import os, time

REF_AUDIO = "ref_voice.wav"
REF_TEXT  = "نيفي راهو يسيل"

ts = int(time.time())
output_dir  = "./outputs"
output_file = f"atlas_{ts}.wav"
output_path = f"{output_dir}/{output_file}"

os.makedirs(output_dir, exist_ok=True)

cmd = (
    f'habibi-tts_infer-cli '
    f'--ref_audio "{REF_AUDIO}" '
    f'--ref_text "{REF_TEXT}" '
    f'--gen_text "راسي يوجعني" '
    f'--dialect ALG '
    f'--output_dir "{output_dir}" '
    f'--output_file "{output_file}"'
)
print(f"Commande : {cmd}")
ret = os.system(cmd)
print(f"Code retour : {ret}")
print(f"Fichier existe : {os.path.exists(output_path)}")
print(f"Contenu outputs : {os.listdir(output_dir)}")

In [ ]:
# Installer habibi-tts dans l'environnement Colab
!pip install habibi-tts -q
!pip uninstall torchcodec -y -q

# Vérifier
!habibi-tts_infer-cli --help 2>/dev/null | head -3

In [ ]:
!pip install streamlit transformers sentence-transformers faiss-gpu \
             datasets huggingface_hub librosa numpy torch \
             bitsandbytes accelerate -q
!pip install habibi-tts -q  # si pas déjà installé

In [ ]:
!pip install streamlit transformers sentence-transformers faiss-cpu \
             datasets huggingface_hub librosa numpy torch \
             bitsandbytes accelerate pyngrok -q

In [ ]:
# Vérifie qu'il est bien là
import os
print("ref_voice.wav présent :", os.path.exists("ref_voice.wav"))

In [ ]:
from google.colab import files
uploaded = files.upload()  # sélectionne ton ref_voice.wav

# Vérifier qu'il est bien là
import os
print("ref_voice.wav présent :", os.path.exists("ref_voice.wav"))

In [ ]:
import os

# Convertir new_test.mp4 → ref_voice.wav
os.system("ffmpeg -i new_test.mp4 -ar 22050 -ac 1 ref_voice.wav -y")

print("ref_voice.wav présent :", os.path.exists("ref_voice.wav"))

In [ ]:
!pip install transformers==4.46.0 -q

In [ ]:
!pip install tokenizers==0.20.3 -q
!pip install transformers==4.46.0 -q  # pour rester cohérent

In [ ]:
!pip install unsloth -q

In [ ]:
%%writefile app.py
import os
os.system("pip uninstall torchcodec -y -q")

import streamlit as st
import torch
import subprocess
import numpy as np
import faiss
from transformers import (
    WhisperProcessor, WhisperForConditionalGeneration,
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
)
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
from huggingface_hub import login
import librosa
import tempfile
import time

from huggingface_hub import login login()
REF_AUDIO = "ref_voice.wav"
REF_TEXT  = "نيفي راهو يسيل"

st.set_page_config(page_title="المساعد الطبي الجزائري", page_icon="🏥", layout="wide")

st.markdown("""
<style>
    .header {
        background: linear-gradient(135deg, #1a6b8a, #0d4d6b);
        color: white; padding: 20px; border-radius: 12px;
        text-align: center; margin-bottom: 20px;
    }
    .header h1 { font-size: 2rem; margin: 0; }
    .header p  { font-size: 1rem; margin: 5px 0 0 0; opacity: 0.85; }
    .chat-container {
        background: white; border-radius: 12px; padding: 20px;
        min-height: 400px; max-height: 600px; overflow-y: auto;
        box-shadow: 0 2px 8px rgba(0,0,0,0.08); margin-bottom: 16px;
    }
    .msg-user {
        background: #1a6b8a; color: white; padding: 10px 16px;
        border-radius: 18px 18px 4px 18px; margin: 8px 0;
        max-width: 75%; margin-left: auto; text-align: right; direction: rtl;
    }
    .msg-atlas {
        background: #e8f4f8; color: #1a1a1a; padding: 10px 16px;
        border-radius: 18px 18px 18px 4px; margin: 8px 0;
        max-width: 75%; border-left: 4px solid #1a6b8a; direction: rtl;
    }
    .msg-llama {
        background: #e8f8ee; color: #1a1a1a; padding: 10px 16px;
        border-radius: 18px 18px 18px 4px; margin: 8px 0;
        max-width: 75%; border-left: 4px solid #2ecc71; direction: rtl;
    }
    .msg-maqa {
        background: #f3e8ff; color: #1a1a1a; padding: 10px 16px;
        border-radius: 18px 18px 18px 4px; margin: 8px 0;
        max-width: 75%; border-left: 4px solid #8e44ad; direction: rtl;
    }
    .transcription-box {
        background: #fff8e1; border: 1px solid #ffc107;
        border-radius: 8px; padding: 12px; margin: 10px 0;
        direction: rtl; font-size: 1.1rem;
    }
    .last-question-box {
        background: #e3f2fd; border: 1px solid #1a6b8a;
        border-radius: 8px; padding: 12px; margin: 10px 0;
        direction: rtl; font-size: 1.1rem;
    }
</style>
""", unsafe_allow_html=True)

st.markdown("""
<div class="header">
    <h1>🏥 المساعد الطبي الجزائري</h1>
    <p>نظام ذكاء اصطناعي طبي بالدارجة الجزائرية</p>
</div>
""", unsafe_allow_html=True)

if "history" not in st.session_state:
    st.session_state.history = []
if "transcription" not in st.session_state:
    st.session_state.transcription = ""


@st.cache_resource
def load_all_models():
    login(token=HF_TOKEN)

    # ── Whisper ───────────────────────────────────────────────────────────────
    whisper_processor = WhisperProcessor.from_pretrained("openai/whisper-small")
    whisper_model = WhisperForConditionalGeneration.from_pretrained(
        "abdeldjalil-smaili/whisper-algerian-medical-final",
        subfolder="checkpoint-1000", token=HF_TOKEN
    ).to("cuda" if torch.cuda.is_available() else "cpu")
    whisper_model.eval()

    # ── RAG (FAISS) ───────────────────────────────────────────────────────────
    embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
    dataset  = load_dataset("abdeldjalil-smaili/algerian-medical-llm-v3", token=HF_TOKEN, split="train")

    def get_q(s):
        try:    return s['text'].split("### Input:")[1].split("### Response:")[0].strip()
        except: return s['text'][:200]

    def get_a(s):
        try:    return s['text'].split("### Response:")[1].strip()
        except: return ""

    questions  = [get_q(s) for s in dataset]
    answers    = [get_a(s) for s in dataset]
    embeddings = embedder.encode(questions, batch_size=256,
                                 show_progress_bar=False, convert_to_numpy=True)
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings.astype(np.float32))

    # ── 4-bit config pour Atlas-Chat et LLaMA Darja ───────────────────────────
    bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )

    # ── Modèle 1 : Atlas-Chat 9B (zero-shot, 4-bit) ───────────────────────────
    atlas_tok = AutoTokenizer.from_pretrained("MBZUAI-Paris/Atlas-Chat-9B", token=HF_TOKEN)
    atlas_mdl = AutoModelForCausalLM.from_pretrained(
        "MBZUAI-Paris/Atlas-Chat-9B", quantization_config=bnb,
        device_map="auto", token=HF_TOKEN
    )
    atlas_mdl.eval()

    # ── Modèle 2 : LLaMA Darja fine-tuné (4-bit) ─────────────────────────────
    llama_darja_tok = AutoTokenizer.from_pretrained(
        "meta-llama/Llama-3.2-3B-Instruct", token=HF_TOKEN)
    llama_darja_mdl = AutoModelForCausalLM.from_pretrained(
        "abdeldjalil-smaili/llama3b-algerian-medical", quantization_config=bnb,
        device_map="auto", token=HF_TOKEN
    )
    llama_darja_mdl.eval()

    # ── Modèle 3 : LLaMA MAQA — fp16 natif sans quantization ─────────────────
    llama_maqa_tok = AutoTokenizer.from_pretrained(
        "meta-llama/Llama-3.2-3B-Instruct", token=HF_TOKEN
    )
    llama_maqa_mdl = AutoModelForCausalLM.from_pretrained(
        "abdeldjalil-smaili/llama3-maqa-medical",
        torch_dtype=torch.float16,
        device_map="auto",
        token=HF_TOKEN
    )
    llama_maqa_mdl.eval()

    return (whisper_processor, whisper_model,
            embedder, index, questions, answers,
            atlas_tok, atlas_mdl,
            llama_darja_tok, llama_darja_mdl,
            llama_maqa_tok, llama_maqa_mdl)


# ── Transcription Whisper ─────────────────────────────────────────────────────
def transcribe(audio_path, processor, model):
    audio, _ = librosa.load(audio_path, sr=16000)
    inputs = processor(audio, sampling_rate=16000, return_tensors="pt").input_features
    inputs = inputs.to("cuda" if torch.cuda.is_available() else "cpu")
    with torch.no_grad():
        ids = model.generate(inputs, language="ar")
    return processor.batch_decode(ids, skip_special_tokens=True)[0]


# ── RAG context ───────────────────────────────────────────────────────────────
def get_rag_context(question, embedder, index, questions, answers, k=3, threshold=8.0):
    q_emb = embedder.encode([question], convert_to_numpy=True)
    distances, indices = index.search(q_emb.astype(np.float32), k)
    contexts = []
    for dist, idx in zip(distances[0], indices[0]):
        if dist < threshold:
            contexts.append(f"Q: {questions[idx]}\nR: {answers[idx]}")
    return "\n\n".join(contexts)


# ── Inférence Atlas-Chat & LLaMA Darja ───────────────────────────────────────
def generate_response_darja(question, model, tokenizer, embedder, index, questions, answers):
    context = get_rag_context(question, embedder, index, questions, answers)
    content = f"""أنت مساعد طبي يتكلم الدارجة الجزائرية فقط.
{f"استخدم هاد المعلومات باش تجاوب:{chr(10)}{context}{chr(10)}{chr(10)}" if context else ""}
قواعد مهمة:
- أجب بجملة واحدة أو جملتين فقط
- استخدم كلام بسيط مفهوم
- ما تذكرش فحوصات معقدة إلا كان السؤال خطير
- ركز على النصيحة العملية البسيطة

السؤال: {question}
الجواب بالدارجة الجزائرية:"""

    messages = [{"role": "user", "content": content}]
    prompt   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs   = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    inputs   = {k: v.to("cuda" if torch.cuda.is_available() else "cpu") for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens     = 60,
            temperature        = 0.2,
            do_sample          = True,
            pad_token_id       = tokenizer.eos_token_id,
            repetition_penalty = 1.5,
        )
    input_len = inputs["input_ids"].shape[1]
    return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()


# ── Inférence LLaMA MAQA — fidèle au notebook ────────────────────────────────
MAQA_SYSTEM_PROMPT = (
    "أنت مساعد طبي متخصص. أجب على الأسئلة الطبية "
    "بدقة وباللغة العربية الفصحى."
)

def generate_response_maqa(question, model, tokenizer):
    messages = [
        {"role": "system", "content": MAQA_SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    inputs = {k: v.to("cuda" if torch.cuda.is_available() else "cpu") for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens     = 200,
            temperature        = 0.7,
            top_p              = 0.9,
            repetition_penalty = 1.1,
            do_sample          = True,
            pad_token_id       = tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded.split("assistant")[-1].strip()


# ── TTS sur CPU (VRAM occupée par les 3 LLMs) ────────────────────────────────
def synthesize_tts(text, output_dir, unique_name):
    os.makedirs(output_dir, exist_ok=True)

    tmp_out    = f"{output_dir}/infer_cli_example.wav"
    final_path = f"{output_dir}/{unique_name}.wav"

    # Supprimer les anciens fichiers
    for f in [tmp_out, final_path]:
        if os.path.exists(f):
            os.remove(f)

    result = subprocess.run(
        ['habibi-tts_infer-cli',
         '--ref_audio', REF_AUDIO,
         '--ref_text',  REF_TEXT,
         '--gen_text',  text,
         '--dialect',   'ALG',
         '--output_dir', output_dir,
         '--device',    'cpu'],      # ← CPU car VRAM saturée par les LLMs
        capture_output=True, text=True
    )

    # Attendre que le fichier soit créé (max 120s)
    deadline = time.time() + 120
    while time.time() < deadline:
        if os.path.exists(tmp_out) and os.path.getsize(tmp_out) > 0:
            os.rename(tmp_out, final_path)
            return final_path
        time.sleep(1)

    return final_path  # retourne le chemin même si timeout


# ── Chargement des modèles ────────────────────────────────────────────────────
with st.spinner("⏳ جاري تحميل النماذج..."):
    (w_proc, w_mdl,
     embedder, index, questions, answers,
     atlas_tok, atlas_mdl,
     llama_darja_tok, llama_darja_mdl,
     llama_maqa_tok, llama_maqa_mdl) = load_all_models()
st.success("✅ النماذج جاهزة!")


# ── UI ────────────────────────────────────────────────────────────────────────
col1, col2 = st.columns([2, 1])

with col1:
    st.markdown("### 💬 المحادثة")

    chat_html = '<div class="chat-container">'
    if not st.session_state.history:
        chat_html += '<p style="text-align:center;color:#999;margin-top:150px;">اسأل سؤالك الطبي بالدارجة الجزائرية 🎤</p>'
    for msg in st.session_state.history:
        chat_html += '<div style="text-align:right;font-size:0.75rem;opacity:0.7;margin-top:10px;">👤 أنت</div>'
        chat_html += f'<div class="msg-user">{msg["question"]}</div>'

        chat_html += '<div style="font-size:0.75rem;opacity:0.7;">🤖 Atlas-Chat (Zero-Shot + RAG)</div>'
        chat_html += f'<div class="msg-atlas">{msg["atlas"]}</div>'

        chat_html += '<div style="font-size:0.75rem;opacity:0.7;">🟢 LLaMA Darja (Fine-Tuné + RAG)</div>'
        chat_html += f'<div class="msg-llama">{msg["llama"]}</div>'

        chat_html += '<div style="font-size:0.75rem;opacity:0.7;">🟣 LLaMA MAQA (Fine-Tuné MSA)</div>'
        chat_html += f'<div class="msg-maqa">{msg["maqa"]}</div>'

        chat_html += '<hr style="border:none;border-top:1px solid #eee;margin:12px 0">'
    chat_html += '</div>'
    st.markdown(chat_html, unsafe_allow_html=True)

    if st.session_state.transcription:
        st.markdown(f"""
        <div class="last-question-box">
            📝 <b>آخر سؤال مفرَّغ:</b> {st.session_state.transcription}
        </div>
        """, unsafe_allow_html=True)

with col2:
    st.markdown("### 🎤 تسجيل السؤال")

    audio_input = st.audio_input("🎤 سجل سؤالك هنا")
    transcription_placeholder = st.empty()

    if audio_input is not None:
        audio_bytes = audio_input.read()
        st.audio(audio_bytes, format='audio/wav')

        if st.button("🚀 ابدأ التحليل", type="primary", use_container_width=True):

            with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
                tmp.write(audio_bytes)
                tmp_path = tmp.name

            wav_path = tmp_path.replace(".wav", "_16k.wav")
            subprocess.run(
                ['ffmpeg', '-i', tmp_path, '-ar', '16000', '-ac', '1', wav_path, '-y'],
                capture_output=True
            )

            with st.spinner("🎤 جاري التفريغ..."):
                question = transcribe(wav_path, w_proc, w_mdl)
                st.session_state.transcription = question

            transcription_placeholder.markdown(f"""
            <div class="transcription-box">
                🎤 <b>سؤالك:</b> {question}
            </div>
            """, unsafe_allow_html=True)

            with st.spinner("🤖 Atlas-Chat يجاوب..."):
                response_atlas = generate_response_darja(
                    question, atlas_mdl, atlas_tok, embedder, index, questions, answers
                )

            with st.spinner("🟢 LLaMA Darja يجاوب..."):
                response_llama = generate_response_darja(
                    question, llama_darja_mdl, llama_darja_tok, embedder, index, questions, answers
                )

            with st.spinner("🟣 LLaMA MAQA يجاوب..."):
                response_maqa = generate_response_maqa(
                    question, llama_maqa_mdl, llama_maqa_tok
                )

            # TTS séquentiels sur CPU (~50s chacun)
            with st.spinner("🔊 Atlas-Chat — توليد صوتي... (~50ث)"):
                atlas_audio = synthesize_tts(response_atlas, "./outputs_atlas", "atlas_response")

            with st.spinner("🔊 LLaMA Darja — توليد صوتي... (~50ث)"):
                llama_audio = synthesize_tts(response_llama, "./outputs_llama", "llama_response")

            with st.spinner("🔊 LLaMA MAQA — توليد صوتي... (~50ث)"):
                maqa_audio  = synthesize_tts(response_maqa,  "./outputs_maqa",  "maqa_response")

            st.session_state.history.append({
                "question"   : question,
                "atlas"      : response_atlas,
                "llama"      : response_llama,
                "maqa"       : response_maqa,
                "atlas_audio": atlas_audio,
                "llama_audio": llama_audio,
                "maqa_audio" : maqa_audio,
            })
            st.rerun()

    if st.session_state.history:
        last = st.session_state.history[-1]
        st.markdown("### 🔊 الردود الصوتية")

        st.markdown("**🤖 Atlas-Chat:**")
        if os.path.exists(last["atlas_audio"]):
            st.audio(last["atlas_audio"])
        else:
            st.warning("Atlas-Chat: fichier audio non trouvé")

        st.markdown("**🟢 LLaMA Darja:**")
        if os.path.exists(last["llama_audio"]):
            st.audio(last["llama_audio"])
        else:
            st.warning("LLaMA Darja: fichier audio non trouvé")

        st.markdown("**🟣 LLaMA MAQA:**")
        if os.path.exists(last["maqa_audio"]):
            st.audio(last["maqa_audio"])
        else:
            st.warning("LLaMA MAQA: fichier audio non trouvé")

        st.markdown("---")
        if st.button("🗑️ مسح المحادثة", use_container_width=True):
            st.session_state.history = []
            st.session_state.transcription = ""
            st.rerun()


In [ ]:
import os
os.system("pip uninstall torchcodec -y -q")

import streamlit as st
import torch
import subprocess
import numpy as np
import faiss
from transformers import (
    WhisperProcessor, WhisperForConditionalGeneration,
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
)
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
from huggingface_hub import login
import librosa
import tempfile
import time

from huggingface_hub import login login()
REF_AUDIO = "ref_voice.wav"
REF_TEXT  = "نيفي راهو يسيل"

st.set_page_config(page_title="المساعد الطبي الجزائري", page_icon="🏥", layout="wide")

st.markdown("""
<style>
    .header {
        background: linear-gradient(135deg, #1a6b8a, #0d4d6b);
        color: white; padding: 20px; border-radius: 12px;
        text-align: center; margin-bottom: 20px;
    }
    .header h1 { font-size: 2rem; margin: 0; }
    .header p  { font-size: 1rem; margin: 5px 0 0 0; opacity: 0.85; }
    .chat-container {
        background: white; border-radius: 12px; padding: 20px;
        min-height: 400px; max-height: 600px; overflow-y: auto;
        box-shadow: 0 2px 8px rgba(0,0,0,0.08); margin-bottom: 16px;
    }
    .msg-user {
        background: #1a6b8a; color: white; padding: 10px 16px;
        border-radius: 18px 18px 4px 18px; margin: 8px 0;
        max-width: 75%; margin-left: auto; text-align: right; direction: rtl;
    }
    .msg-atlas {
        background: #e8f4f8; color: #1a1a1a; padding: 10px 16px;
        border-radius: 18px 18px 18px 4px; margin: 8px 0;
        max-width: 75%; border-left: 4px solid #1a6b8a; direction: rtl;
    }
    .msg-llama {
        background: #e8f8ee; color: #1a1a1a; padding: 10px 16px;
        border-radius: 18px 18px 18px 4px; margin: 8px 0;
        max-width: 75%; border-left: 4px solid #2ecc71; direction: rtl;
    }
    .msg-maqa {
        background: #f3e8ff; color: #1a1a1a; padding: 10px 16px;
        border-radius: 18px 18px 18px 4px; margin: 8px 0;
        max-width: 75%; border-left: 4px solid #8e44ad; direction: rtl;
    }
    .transcription-box {
        background: #fff8e1; border: 1px solid #ffc107;
        border-radius: 8px; padding: 12px; margin: 10px 0;
        direction: rtl; font-size: 1.1rem;
    }
    .last-question-box {
        background: #e3f2fd; border: 1px solid #1a6b8a;
        border-radius: 8px; padding: 12px; margin: 10px 0;
        direction: rtl; font-size: 1.1rem;
    }
</style>
""", unsafe_allow_html=True)

st.markdown("""
<div class="header">
    <h1>🏥 المساعد الطبي الجزائري</h1>
    <p>نظام ذكاء اصطناعي طبي بالدارجة الجزائرية</p>
</div>
""", unsafe_allow_html=True)

if "history" not in st.session_state:
    st.session_state.history = []
if "transcription" not in st.session_state:
    st.session_state.transcription = ""


@st.cache_resource
def load_all_models():
    login(token=HF_TOKEN)

    # ── Whisper ───────────────────────────────────────────────────────────────
    whisper_processor = WhisperProcessor.from_pretrained("openai/whisper-small")
    whisper_model = WhisperForConditionalGeneration.from_pretrained(
        "abdeldjalil-smaili/whisper-algerian-medical-final",
        subfolder="checkpoint-1000", token=HF_TOKEN
    ).to("cuda" if torch.cuda.is_available() else "cpu")
    whisper_model.eval()

    # ── RAG (FAISS) ───────────────────────────────────────────────────────────
    embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
    dataset  = load_dataset("abdeldjalil-smaili/algerian-medical-llm-v3", token=HF_TOKEN, split="train")

    def get_q(s):
        try:    return s['text'].split("### Input:")[1].split("### Response:")[0].strip()
        except: return s['text'][:200]

    def get_a(s):
        try:    return s['text'].split("### Response:")[1].strip()
        except: return ""

    questions  = [get_q(s) for s in dataset]
    answers    = [get_a(s) for s in dataset]
    embeddings = embedder.encode(questions, batch_size=256,
                                 show_progress_bar=False, convert_to_numpy=True)
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings.astype(np.float32))

    # ── 4-bit config pour Atlas-Chat et LLaMA Darja ───────────────────────────
    bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )

    # ── Modèle 1 : Atlas-Chat 9B (zero-shot, 4-bit) ───────────────────────────
    atlas_tok = AutoTokenizer.from_pretrained("MBZUAI-Paris/Atlas-Chat-9B", token=HF_TOKEN)
    atlas_mdl = AutoModelForCausalLM.from_pretrained(
        "MBZUAI-Paris/Atlas-Chat-9B", quantization_config=bnb,
        device_map="auto", token=HF_TOKEN
    )
    atlas_mdl.eval()

    # ── Modèle 2 : LLaMA Darja fine-tuné (4-bit) ─────────────────────────────
    llama_darja_tok = AutoTokenizer.from_pretrained(
        "meta-llama/Llama-3.2-3B-Instruct", token=HF_TOKEN)
    llama_darja_mdl = AutoModelForCausalLM.from_pretrained(
        "abdeldjalil-smaili/llama3b-algerian-medical", quantization_config=bnb,
        device_map="auto", token=HF_TOKEN
    )
    llama_darja_mdl.eval()

    # ── Modèle 3 : LLaMA MAQA — bfloat16 natif ───────────────────────────────
    llama_maqa_tok = AutoTokenizer.from_pretrained(
        "meta-llama/Llama-3.2-3B-Instruct", token=HF_TOKEN
    )
    llama_maqa_tok.pad_token = llama_maqa_tok.eos_token
    llama_maqa_mdl = AutoModelForCausalLM.from_pretrained(
        "abdeldjalil-smaili/llama3-maqa-medical",
        torch_dtype=torch.bfloat16,
        device_map="auto",
        token=HF_TOKEN
    )
    llama_maqa_mdl.eval()

    return (whisper_processor, whisper_model,
            embedder, index, questions, answers,
            atlas_tok, atlas_mdl,
            llama_darja_tok, llama_darja_mdl,
            llama_maqa_tok, llama_maqa_mdl)


# ── Transcription Whisper ─────────────────────────────────────────────────────
def transcribe(audio_path, processor, model):
    audio, _ = librosa.load(audio_path, sr=16000)
    inputs = processor(audio, sampling_rate=16000, return_tensors="pt").input_features
    inputs = inputs.to("cuda" if torch.cuda.is_available() else "cpu")
    with torch.no_grad():
        ids = model.generate(inputs, language="ar")
    return processor.batch_decode(ids, skip_special_tokens=True)[0]


# ── RAG context ───────────────────────────────────────────────────────────────
def get_rag_context(question, embedder, index, questions, answers, k=3, threshold=8.0):
    q_emb = embedder.encode([question], convert_to_numpy=True)
    distances, indices = index.search(q_emb.astype(np.float32), k)
    contexts = []
    for dist, idx in zip(distances[0], indices[0]):
        if dist < threshold:
            contexts.append(f"Q: {questions[idx]}\nR: {answers[idx]}")
    return "\n\n".join(contexts)


# ── Inférence Atlas-Chat & LLaMA Darja ───────────────────────────────────────
def generate_response_darja(question, model, tokenizer, embedder, index, questions, answers):
    context = get_rag_context(question, embedder, index, questions, answers)
    content = f"""أنت مساعد طبي يتكلم الدارجة الجزائرية فقط.
{f"استخدم هاد المعلومات باش تجاوب:{chr(10)}{context}{chr(10)}{chr(10)}" if context else ""}
قواعد مهمة:
- أجب بجملة واحدة أو جملتين فقط
- استخدم كلام بسيط مفهوم
- ما تذكرش فحوصات معقدة إلا كان السؤال خطير
- ركز على النصيحة العملية البسيطة

السؤال: {question}
الجواب بالدارجة الجزائرية:"""

    messages = [{"role": "user", "content": content}]
    prompt   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs   = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    inputs   = {k: v.to("cuda" if torch.cuda.is_available() else "cpu") for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens     = 60,
            temperature        = 0.2,
            do_sample          = True,
            pad_token_id       = tokenizer.eos_token_id,
            repetition_penalty = 1.5,
        )
    input_len = inputs["input_ids"].shape[1]
    return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()


# ── Inférence LLaMA MAQA ─────────────────────────────────────────────────────
MAQA_SYSTEM_PROMPT = (
    "أنت مساعد طبي متخصص. أجب على الأسئلة الطبية "
    "بدقة وباللغة العربية الفصحى."
)

def generate_response_maqa(question, model, tokenizer):
    messages = [
        {"role": "system", "content": MAQA_SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    inputs = {k: v.to("cuda" if torch.cuda.is_available() else "cpu") for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens     = 200,
            temperature        = 0.7,
            top_p              = 0.9,
            repetition_penalty = 1.1,
            do_sample          = True,
            pad_token_id       = tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded.split("assistant")[-1].strip()


# ── TTS ───────────────────────────────────────────────────────────────────────
def synthesize_tts(text, output_dir, unique_name):
    os.makedirs(output_dir, exist_ok=True)

    tmp_out    = f"{output_dir}/infer_cli_example.wav"
    final_path = f"{output_dir}/{unique_name}.wav"

    for f in [tmp_out, final_path]:
        if os.path.exists(f):
            os.remove(f)

    # Désinstaller torchcodec avant chaque appel TTS (sentence-transformers le réinstalle)
    os.system("pip uninstall torchcodec -y -q")

    subprocess.run(
        ['habibi-tts_infer-cli',
         '--ref_audio', REF_AUDIO,
         '--ref_text',  REF_TEXT,
         '--gen_text',  text,
         '--dialect',   'ALG',
         '--output_dir', output_dir,
         '--device',    'cuda'],
        capture_output=True, text=True
    )

    deadline = time.time() + 120
    while time.time() < deadline:
        if os.path.exists(tmp_out) and os.path.getsize(tmp_out) > 0:
            os.rename(tmp_out, final_path)
            return final_path
        time.sleep(1)

    return final_path


# ── Chargement des modèles ────────────────────────────────────────────────────
with st.spinner("⏳ جاري تحميل النماذج..."):
    (w_proc, w_mdl,
     embedder, index, questions, answers,
     atlas_tok, atlas_mdl,
     llama_darja_tok, llama_darja_mdl,
     llama_maqa_tok, llama_maqa_mdl) = load_all_models()
st.success("✅ النماذج جاهزة!")


# ── UI ────────────────────────────────────────────────────────────────────────
col1, col2 = st.columns([2, 1])

with col1:
    st.markdown("### 💬 المحادثة")

    chat_html = '<div class="chat-container">'
    if not st.session_state.history:
        chat_html += '<p style="text-align:center;color:#999;margin-top:150px;">اسأل سؤالك الطبي بالدارجة الجزائرية 🎤</p>'
    for msg in st.session_state.history:
        chat_html += '<div style="text-align:right;font-size:0.75rem;opacity:0.7;margin-top:10px;">👤 أنت</div>'
        chat_html += f'<div class="msg-user">{msg["question"]}</div>'

        chat_html += '<div style="font-size:0.75rem;opacity:0.7;">🤖 Atlas-Chat (Zero-Shot + RAG)</div>'
        chat_html += f'<div class="msg-atlas">{msg["atlas"]}</div>'

        chat_html += '<div style="font-size:0.75rem;opacity:0.7;">🟢 LLaMA Darja (Fine-Tuné + RAG)</div>'
        chat_html += f'<div class="msg-llama">{msg["llama"]}</div>'

        chat_html += '<div style="font-size:0.75rem;opacity:0.7;">🟣 LLaMA MAQA (Fine-Tuné MSA)</div>'
        chat_html += f'<div class="msg-maqa">{msg["maqa"]}</div>'

        chat_html += '<hr style="border:none;border-top:1px solid #eee;margin:12px 0">'
    chat_html += '</div>'
    st.markdown(chat_html, unsafe_allow_html=True)

    if st.session_state.transcription:
        st.markdown(f"""
        <div class="last-question-box">
            📝 <b>آخر سؤال مفرَّغ:</b> {st.session_state.transcription}
        </div>
        """, unsafe_allow_html=True)

with col2:
    st.markdown("### 🎤 تسجيل السؤال")

    audio_input = st.audio_input("🎤 سجل سؤالك هنا")
    transcription_placeholder = st.empty()

    if audio_input is not None:
        audio_bytes = audio_input.read()
        st.audio(audio_bytes, format='audio/wav')

        if st.button("🚀 ابدأ التحليل", type="primary", use_container_width=True):

            with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
                tmp.write(audio_bytes)
                tmp_path = tmp.name

            wav_path = tmp_path.replace(".wav", "_16k.wav")
            subprocess.run(
                ['ffmpeg', '-i', tmp_path, '-ar', '16000', '-ac', '1', wav_path, '-y'],
                capture_output=True
            )

            with st.spinner("🎤 جاري التفريغ..."):
                question = transcribe(wav_path, w_proc, w_mdl)
                st.session_state.transcription = question

            transcription_placeholder.markdown(f"""
            <div class="transcription-box">
                🎤 <b>سؤالك:</b> {question}
            </div>
            """, unsafe_allow_html=True)

            with st.spinner("🤖 Atlas-Chat يجاوب..."):
                response_atlas = generate_response_darja(
                    question, atlas_mdl, atlas_tok, embedder, index, questions, answers
                )

            with st.spinner("🟢 LLaMA Darja يجاوب..."):
                response_llama = generate_response_darja(
                    question, llama_darja_mdl, llama_darja_tok, embedder, index, questions, answers
                )

            with st.spinner("🟣 LLaMA MAQA يجاوب..."):
                response_maqa = generate_response_maqa(
                    question, llama_maqa_mdl, llama_maqa_tok
                )

            with st.spinner("🔊 Atlas-Chat — توليد صوتي..."):
                atlas_audio = synthesize_tts(response_atlas, "./outputs_atlas", "atlas_response")

            with st.spinner("🔊 LLaMA Darja — توليد صوتي..."):
                llama_audio = synthesize_tts(response_llama, "./outputs_llama", "llama_response")

            with st.spinner("🔊 LLaMA MAQA — توليد صوتي..."):
                maqa_audio  = synthesize_tts(response_maqa,  "./outputs_maqa",  "maqa_response")

            st.session_state.history.append({
                "question"   : question,
                "atlas"      : response_atlas,
                "llama"      : response_llama,
                "maqa"       : response_maqa,
                "atlas_audio": atlas_audio,
                "llama_audio": llama_audio,
                "maqa_audio" : maqa_audio,
            })
            st.rerun()

    if st.session_state.history:
        last = st.session_state.history[-1]
        st.markdown("### 🔊 الردود الصوتية")

        st.markdown("**🤖 Atlas-Chat:**")
        if os.path.exists(last["atlas_audio"]):
            st.audio(last["atlas_audio"])
        else:
            st.warning("Atlas-Chat: fichier audio non trouvé")

        st.markdown("**🟢 LLaMA Darja:**")
        if os.path.exists(last["llama_audio"]):
            st.audio(last["llama_audio"])
        else:
            st.warning("LLaMA Darja: fichier audio non trouvé")

        st.markdown("**🟣 LLaMA MAQA:**")
        if os.path.exists(last["maqa_audio"]):
            st.audio(last["maqa_audio"])
        else:
            st.warning("LLaMA MAQA: fichier audio non trouvé")

        st.markdown("---")
        if st.button("🗑️ مسح المحادثة", use_container_width=True):
            st.session_state.history = []
            st.session_state.transcription = ""
            st.rerun()

In [ ]:
import torch
free, total = torch.cuda.mem_get_info()
print(f"VRAM libre : {free/1024**3:.1f} GB / {total/1024**3:.1f} GB")

In [ ]:
!pip install torchaudio --upgrade -q

In [ ]:
!pip uninstall torchaudio -y -q
!pip install torchaudio==2.11.0 --index-url https://download.pytorch.org/whl/cu128 -q

In [ ]:
%%writefile app.py
import os
os.system("pip uninstall torchcodec -y -q")

import streamlit as st
import torch
import subprocess
import numpy as np
import faiss
from transformers import (
    WhisperProcessor, WhisperForConditionalGeneration,
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
)
from unsloth import FastLanguageModel
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
from huggingface_hub import login
import librosa
import tempfile
import time

from huggingface_hub import login login()
REF_AUDIO = "ref_voice.wav"
REF_TEXT  = "نيفي راهو يسيل"

st.set_page_config(page_title="المساعد الطبي الجزائري", page_icon="🏥", layout="wide")

st.markdown("""
<style>
    .header {
        background: linear-gradient(135deg, #1a6b8a, #0d4d6b);
        color: white; padding: 20px; border-radius: 12px;
        text-align: center; margin-bottom: 20px;
    }
    .header h1 { font-size: 2rem; margin: 0; }
    .header p  { font-size: 1rem; margin: 5px 0 0 0; opacity: 0.85; }
    .chat-container {
        background: white; border-radius: 12px; padding: 20px;
        min-height: 400px; max-height: 600px; overflow-y: auto;
        box-shadow: 0 2px 8px rgba(0,0,0,0.08); margin-bottom: 16px;
    }
    .msg-user {
        background: #1a6b8a; color: white; padding: 10px 16px;
        border-radius: 18px 18px 4px 18px; margin: 8px 0;
        max-width: 75%; margin-left: auto; text-align: right; direction: rtl;
    }
    .msg-atlas {
        background: #e8f4f8; color: #1a1a1a; padding: 10px 16px;
        border-radius: 18px 18px 18px 4px; margin: 8px 0;
        max-width: 75%; border-left: 4px solid #1a6b8a; direction: rtl;
    }
    .msg-llama {
        background: #e8f8ee; color: #1a1a1a; padding: 10px 16px;
        border-radius: 18px 18px 18px 4px; margin: 8px 0;
        max-width: 75%; border-left: 4px solid #2ecc71; direction: rtl;
    }
    .msg-maqa {
        background: #f3e8ff; color: #1a1a1a; padding: 10px 16px;
        border-radius: 18px 18px 18px 4px; margin: 8px 0;
        max-width: 75%; border-left: 4px solid #8e44ad; direction: rtl;
    }
    .transcription-box {
        background: #fff8e1; border: 1px solid #ffc107;
        border-radius: 8px; padding: 12px; margin: 10px 0;
        direction: rtl; font-size: 1.1rem;
    }
    .last-question-box {
        background: #e3f2fd; border: 1px solid #1a6b8a;
        border-radius: 8px; padding: 12px; margin: 10px 0;
        direction: rtl; font-size: 1.1rem;
    }
</style>
""", unsafe_allow_html=True)

st.markdown("""
<div class="header">
    <h1>🏥 المساعد الطبي الجزائري</h1>
    <p>نظام ذكاء اصطناعي طبي بالدارجة الجزائرية</p>
</div>
""", unsafe_allow_html=True)

if "history" not in st.session_state:
    st.session_state.history = []
if "transcription" not in st.session_state:
    st.session_state.transcription = ""


@st.cache_resource
def load_all_models():
    login(token=HF_TOKEN)

    # ── Whisper ───────────────────────────────────────────────────────────────
    whisper_processor = WhisperProcessor.from_pretrained("openai/whisper-small")
    whisper_model = WhisperForConditionalGeneration.from_pretrained(
        "abdeldjalil-smaili/whisper-algerian-medical-final",
        subfolder="checkpoint-1000", token=HF_TOKEN
    ).to("cuda" if torch.cuda.is_available() else "cpu")
    whisper_model.eval()

    # ── RAG (FAISS) ───────────────────────────────────────────────────────────
    embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
    dataset  = load_dataset("abdeldjalil-smaili/algerian-medical-llm-v3", token=HF_TOKEN, split="train")

    def get_q(s):
        try:    return s['text'].split("### Input:")[1].split("### Response:")[0].strip()
        except: return s['text'][:200]

    def get_a(s):
        try:    return s['text'].split("### Response:")[1].strip()
        except: return ""

    questions  = [get_q(s) for s in dataset]
    answers    = [get_a(s) for s in dataset]
    embeddings = embedder.encode(questions, batch_size=256,
                                 show_progress_bar=False, convert_to_numpy=True)
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings.astype(np.float32))

    # ── 4-bit config pour Atlas-Chat et LLaMA Darja ───────────────────────────
    bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )

    # ── Modèle 1 : Atlas-Chat 9B (zero-shot, 4-bit) ───────────────────────────
    atlas_tok = AutoTokenizer.from_pretrained("MBZUAI-Paris/Atlas-Chat-9B", token=HF_TOKEN)
    atlas_mdl = AutoModelForCausalLM.from_pretrained(
        "MBZUAI-Paris/Atlas-Chat-9B", quantization_config=bnb,
        device_map="auto", token=HF_TOKEN
    )
    atlas_mdl.eval()

    # ── Modèle 2 : LLaMA Darja fine-tuné (4-bit) ─────────────────────────────
    llama_darja_tok = AutoTokenizer.from_pretrained(
        "meta-llama/Llama-3.2-3B-Instruct", token=HF_TOKEN)
    llama_darja_mdl = AutoModelForCausalLM.from_pretrained(
        "abdeldjalil-smaili/llama3b-algerian-medical", quantization_config=bnb,
        device_map="auto", token=HF_TOKEN
    )
    llama_darja_mdl.eval()

    # ── Modèle 3 : LLaMA MAQA — chargé via Unsloth + for_inference ───────────
    # Identique au notebook : load_in_4bit=True + FastLanguageModel.for_inference()
    llama_maqa_mdl, llama_maqa_tok = FastLanguageModel.from_pretrained(
        model_name     = "abdeldjalil-smaili/llama3-maqa-medical",
        max_seq_length = 2048,
        dtype          = None,
        load_in_4bit   = True,
        token          = HF_TOKEN,
    )
    FastLanguageModel.for_inference(llama_maqa_mdl)

    return (whisper_processor, whisper_model,
            embedder, index, questions, answers,
            atlas_tok, atlas_mdl,
            llama_darja_tok, llama_darja_mdl,
            llama_maqa_tok, llama_maqa_mdl)


# ── Transcription Whisper ─────────────────────────────────────────────────────
def transcribe(audio_path, processor, model):
    audio, _ = librosa.load(audio_path, sr=16000)
    inputs = processor(audio, sampling_rate=16000, return_tensors="pt").input_features
    inputs = inputs.to("cuda" if torch.cuda.is_available() else "cpu")
    with torch.no_grad():
        ids = model.generate(inputs, language="ar")
    return processor.batch_decode(ids, skip_special_tokens=True)[0]


# ── RAG context ───────────────────────────────────────────────────────────────
def get_rag_context(question, embedder, index, questions, answers, k=3, threshold=8.0):
    q_emb = embedder.encode([question], convert_to_numpy=True)
    distances, indices = index.search(q_emb.astype(np.float32), k)
    contexts = []
    for dist, idx in zip(distances[0], indices[0]):
        if dist < threshold:
            contexts.append(f"Q: {questions[idx]}\nR: {answers[idx]}")
    return "\n\n".join(contexts)


# ── Inférence Atlas-Chat & LLaMA Darja ───────────────────────────────────────
def generate_response_darja(question, model, tokenizer, embedder, index, questions, answers):
    context = get_rag_context(question, embedder, index, questions, answers)
    content = f"""أنت مساعد طبي يتكلم الدارجة الجزائرية فقط.
{f"استخدم هاد المعلومات باش تجاوب:{chr(10)}{context}{chr(10)}{chr(10)}" if context else ""}
قواعد مهمة:
- أجب بجملة واحدة أو جملتين فقط
- استخدم كلام بسيط مفهوم
- ما تذكرش فحوصات معقدة إلا كان السؤال خطير
- ركز على النصيحة العملية البسيطة

السؤال: {question}
الجواب بالدارجة الجزائرية:"""

    messages = [{"role": "user", "content": content}]
    prompt   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs   = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    inputs   = {k: v.to("cuda" if torch.cuda.is_available() else "cpu") for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens     = 60,
            temperature        = 0.2,
            do_sample          = True,
            pad_token_id       = tokenizer.eos_token_id,
            repetition_penalty = 1.5,
        )
    input_len = inputs["input_ids"].shape[1]
    return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()


# ── Inférence LLaMA MAQA — identique au notebook ─────────────────────────────
MAQA_SYSTEM_PROMPT = (
    "أنت مساعد طبي متخصص. أجب على الأسئلة الطبية "
    "بدقة وباللغة العربية الفصحى."
)

def generate_response_maqa(question, model, tokenizer):
    """
    Identique au notebook :
    - FastLanguageModel.for_inference(model) déjà appelé au chargement
    - même system prompt, mêmes paramètres de génération
    - extraction via .split("assistant")[-1].strip()
    """
    messages = [
        {"role": "system", "content": MAQA_SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens     = 200,
            temperature        = 0.7,
            top_p              = 0.9,
            repetition_penalty = 1.1,
            do_sample          = True,
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded.split("assistant")[-1].strip()


# ── TTS sur CPU (VRAM occupée par les 3 LLMs) ────────────────────────────────
def synthesize_tts(text, output_dir, unique_name):
    os.makedirs(output_dir, exist_ok=True)

    tmp_out    = f"{output_dir}/infer_cli_example.wav"
    final_path = f"{output_dir}/{unique_name}.wav"

    for f in [tmp_out, final_path]:
        if os.path.exists(f):
            os.remove(f)

    subprocess.run(
        ['habibi-tts_infer-cli',
         '--ref_audio', REF_AUDIO,
         '--ref_text',  REF_TEXT,
         '--gen_text',  text,
         '--dialect',   'ALG',
         '--output_dir', output_dir,
         '--device',    'cpu'],
        capture_output=True, text=True
    )

    deadline = time.time() + 120
    while time.time() < deadline:
        if os.path.exists(tmp_out) and os.path.getsize(tmp_out) > 0:
            os.rename(tmp_out, final_path)
            return final_path
        time.sleep(1)

    return final_path


# ── Chargement des modèles ────────────────────────────────────────────────────
with st.spinner("⏳ جاري تحميل النماذج..."):
    (w_proc, w_mdl,
     embedder, index, questions, answers,
     atlas_tok, atlas_mdl,
     llama_darja_tok, llama_darja_mdl,
     llama_maqa_tok, llama_maqa_mdl) = load_all_models()
st.success("✅ النماذج جاهزة!")


# ── UI ────────────────────────────────────────────────────────────────────────
col1, col2 = st.columns([2, 1])

with col1:
    st.markdown("### 💬 المحادثة")

    chat_html = '<div class="chat-container">'
    if not st.session_state.history:
        chat_html += '<p style="text-align:center;color:#999;margin-top:150px;">اسأل سؤالك الطبي بالدارجة الجزائرية 🎤</p>'
    for msg in st.session_state.history:
        chat_html += '<div style="text-align:right;font-size:0.75rem;opacity:0.7;margin-top:10px;">👤 أنت</div>'
        chat_html += f'<div class="msg-user">{msg["question"]}</div>'

        chat_html += '<div style="font-size:0.75rem;opacity:0.7;">🤖 Atlas-Chat (Zero-Shot + RAG)</div>'
        chat_html += f'<div class="msg-atlas">{msg["atlas"]}</div>'

        chat_html += '<div style="font-size:0.75rem;opacity:0.7;">🟢 LLaMA Darja (Fine-Tuné + RAG)</div>'
        chat_html += f'<div class="msg-llama">{msg["llama"]}</div>'

        chat_html += '<div style="font-size:0.75rem;opacity:0.7;">🟣 LLaMA MAQA (Fine-Tuné MSA)</div>'
        chat_html += f'<div class="msg-maqa">{msg["maqa"]}</div>'

        chat_html += '<hr style="border:none;border-top:1px solid #eee;margin:12px 0">'
    chat_html += '</div>'
    st.markdown(chat_html, unsafe_allow_html=True)

    if st.session_state.transcription:
        st.markdown(f"""
        <div class="last-question-box">
            📝 <b>آخر سؤال مفرَّغ:</b> {st.session_state.transcription}
        </div>
        """, unsafe_allow_html=True)

with col2:
    st.markdown("### 🎤 تسجيل السؤال")

    audio_input = st.audio_input("🎤 سجل سؤالك هنا")
    transcription_placeholder = st.empty()

    if audio_input is not None:
        audio_bytes = audio_input.read()
        st.audio(audio_bytes, format='audio/wav')

        if st.button("🚀 ابدأ التحليل", type="primary", use_container_width=True):

            with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
                tmp.write(audio_bytes)
                tmp_path = tmp.name

            wav_path = tmp_path.replace(".wav", "_16k.wav")
            subprocess.run(
                ['ffmpeg', '-i', tmp_path, '-ar', '16000', '-ac', '1', wav_path, '-y'],
                capture_output=True
            )

            with st.spinner("🎤 جاري التفريغ..."):
                question = transcribe(wav_path, w_proc, w_mdl)
                st.session_state.transcription = question

            transcription_placeholder.markdown(f"""
            <div class="transcription-box">
                🎤 <b>سؤالك:</b> {question}
            </div>
            """, unsafe_allow_html=True)

            with st.spinner("🤖 Atlas-Chat يجاوب..."):
                response_atlas = generate_response_darja(
                    question, atlas_mdl, atlas_tok, embedder, index, questions, answers
                )

            with st.spinner("🟢 LLaMA Darja يجاوب..."):
                response_llama = generate_response_darja(
                    question, llama_darja_mdl, llama_darja_tok, embedder, index, questions, answers
                )

            with st.spinner("🟣 LLaMA MAQA يجاوب..."):
                response_maqa = generate_response_maqa(
                    question, llama_maqa_mdl, llama_maqa_tok
                )

            with st.spinner("🔊 Atlas-Chat — توليد صوتي... (~50ث)"):
                atlas_audio = synthesize_tts(response_atlas, "./outputs_atlas", "atlas_response")

            with st.spinner("🔊 LLaMA Darja — توليد صوتي... (~50ث)"):
                llama_audio = synthesize_tts(response_llama, "./outputs_llama", "llama_response")

            with st.spinner("🔊 LLaMA MAQA — توليد صوتي... (~50ث)"):
                maqa_audio  = synthesize_tts(response_maqa,  "./outputs_maqa",  "maqa_response")

            st.session_state.history.append({
                "question"   : question,
                "atlas"      : response_atlas,
                "llama"      : response_llama,
                "maqa"       : response_maqa,
                "atlas_audio": atlas_audio,
                "llama_audio": llama_audio,
                "maqa_audio" : maqa_audio,
            })
            st.rerun()

    if st.session_state.history:
        last = st.session_state.history[-1]
        st.markdown("### 🔊 الردود الصوتية")

        st.markdown("**🤖 Atlas-Chat:**")
        if os.path.exists(last["atlas_audio"]):
            st.audio(last["atlas_audio"])
        else:
            st.warning("Atlas-Chat: fichier audio non trouvé")

        st.markdown("**🟢 LLaMA Darja:**")
        if os.path.exists(last["llama_audio"]):
            st.audio(last["llama_audio"])
        else:
            st.warning("LLaMA Darja: fichier audio non trouvé")

        st.markdown("**🟣 LLaMA MAQA:**")
        if os.path.exists(last["maqa_audio"]):
            st.audio(last["maqa_audio"])
        else:
            st.warning("LLaMA MAQA: fichier audio non trouvé")

        st.markdown("---")
        if st.button("🗑️ مسح المحادثة", use_container_width=True):
            st.session_state.history = []
            st.session_state.transcription = ""
            st.rerun()

In [ ]:
!pip install "transformers>=4.51.3,<=4.57.6" -q

In [ ]:
%%writefile app.py
import os
os.system("pip uninstall torchcodec -y -q")

import streamlit as st
import torch
import subprocess
import numpy as np
import faiss
from transformers import (
    WhisperProcessor, WhisperForConditionalGeneration,
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
)
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
from huggingface_hub import login
import librosa
import tempfile
import time

from huggingface_hub import login 
login()
REF_AUDIO = "ref_voice.wav"
REF_TEXT  = "نيفي راهو يسيل"

st.set_page_config(page_title="المساعد الطبي الجزائري", page_icon="🏥", layout="wide")

st.markdown("""
<style>
    .header {
        background: linear-gradient(135deg, #1a6b8a, #0d4d6b);
        color: white; padding: 20px; border-radius: 12px;
        text-align: center; margin-bottom: 20px;
    }
    .header h1 { font-size: 2rem; margin: 0; }
    .header p  { font-size: 1rem; margin: 5px 0 0 0; opacity: 0.85; }
    .chat-container {
        background: white; border-radius: 12px; padding: 20px;
        min-height: 400px; max-height: 600px; overflow-y: auto;
        box-shadow: 0 2px 8px rgba(0,0,0,0.08); margin-bottom: 16px;
    }
    .msg-user {
        background: #1a6b8a; color: white; padding: 10px 16px;
        border-radius: 18px 18px 4px 18px; margin: 8px 0;
        max-width: 75%; margin-left: auto; text-align: right; direction: rtl;
    }
    .msg-atlas {
        background: #e8f4f8; color: #1a1a1a; padding: 10px 16px;
        border-radius: 18px 18px 18px 4px; margin: 8px 0;
        max-width: 75%; border-left: 4px solid #1a6b8a; direction: rtl;
    }
    .msg-llama {
        background: #e8f8ee; color: #1a1a1a; padding: 10px 16px;
        border-radius: 18px 18px 18px 4px; margin: 8px 0;
        max-width: 75%; border-left: 4px solid #2ecc71; direction: rtl;
    }
    .msg-maqa {
        background: #f3e8ff; color: #1a1a1a; padding: 10px 16px;
        border-radius: 18px 18px 18px 4px; margin: 8px 0;
        max-width: 75%; border-left: 4px solid #8e44ad; direction: rtl;
    }
    .transcription-box {
        background: #fff8e1; border: 1px solid #ffc107;
        border-radius: 8px; padding: 12px; margin: 10px 0;
        direction: rtl; font-size: 1.1rem;
    }
    .last-question-box {
        background: #e3f2fd; border: 1px solid #1a6b8a;
        border-radius: 8px; padding: 12px; margin: 10px 0;
        direction: rtl; font-size: 1.1rem;
    }
</style>
""", unsafe_allow_html=True)

st.markdown("""
<div class="header">
    <h1>🏥 المساعد الطبي الجزائري</h1>
    <p>نظام ذكاء اصطناعي طبي بالدارجة الجزائرية</p>
</div>
""", unsafe_allow_html=True)

if "history" not in st.session_state:
    st.session_state.history = []
if "transcription" not in st.session_state:
    st.session_state.transcription = ""


@st.cache_resource
def load_all_models():
    login(token=HF_TOKEN)

    # ── Whisper ───────────────────────────────────────────────────────────────
    whisper_processor = WhisperProcessor.from_pretrained("openai/whisper-small")
    whisper_model = WhisperForConditionalGeneration.from_pretrained(
        "abdeldjalil-smaili/whisper-algerian-medical-final",
        subfolder="checkpoint-1000", token=HF_TOKEN
    ).to("cuda" if torch.cuda.is_available() else "cpu")
    whisper_model.eval()

    # ── RAG (FAISS) ───────────────────────────────────────────────────────────
    embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
    dataset  = load_dataset("abdeldjalil-smaili/algerian-medical-llm-v3", token=HF_TOKEN, split="train")

    def get_q(s):
        try:    return s['text'].split("### Input:")[1].split("### Response:")[0].strip()
        except: return s['text'][:200]

    def get_a(s):
        try:    return s['text'].split("### Response:")[1].strip()
        except: return ""

    questions  = [get_q(s) for s in dataset]
    answers    = [get_a(s) for s in dataset]
    embeddings = embedder.encode(questions, batch_size=256,
                                 show_progress_bar=False, convert_to_numpy=True)
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings.astype(np.float32))

    # ── 4-bit config pour Atlas-Chat et LLaMA Darja ───────────────────────────
    bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )

    # ── Modèle 1 : Atlas-Chat 9B (zero-shot, 4-bit) ───────────────────────────
    atlas_tok = AutoTokenizer.from_pretrained("MBZUAI-Paris/Atlas-Chat-9B", token=HF_TOKEN)
    atlas_mdl = AutoModelForCausalLM.from_pretrained(
        "MBZUAI-Paris/Atlas-Chat-9B", quantization_config=bnb,
        device_map="auto", token=HF_TOKEN
    )
    atlas_mdl.eval()

    # ── Modèle 2 : LLaMA Darja fine-tuné (4-bit) ─────────────────────────────
    llama_darja_tok = AutoTokenizer.from_pretrained(
        "meta-llama/Llama-3.2-3B-Instruct", token=HF_TOKEN)
    llama_darja_mdl = AutoModelForCausalLM.from_pretrained(
        "abdeldjalil-smaili/llama3b-algerian-medical", quantization_config=bnb,
        device_map="auto", token=HF_TOKEN
    )
    llama_darja_mdl.eval()

    # ── Modèle 3 : LLaMA MAQA — fp16 natif (94GB VRAM disponible sur G4) ─────
    # Sans Unsloth (incompatible torch 2.11)
    # fp16 pur = précision identique au notebook (merged_16bit)
    llama_maqa_tok = AutoTokenizer.from_pretrained(
        "meta-llama/Llama-3.2-3B-Instruct", token=HF_TOKEN
    )
    llama_maqa_tok.pad_token = llama_maqa_tok.eos_token
    llama_maqa_mdl = AutoModelForCausalLM.from_pretrained(
        "abdeldjalil-smaili/llama3-maqa-medical",
        torch_dtype=torch.float16,
        device_map="auto",
        token=HF_TOKEN
    )
    llama_maqa_mdl.eval()

    return (whisper_processor, whisper_model,
            embedder, index, questions, answers,
            atlas_tok, atlas_mdl,
            llama_darja_tok, llama_darja_mdl,
            llama_maqa_tok, llama_maqa_mdl)


# ── Transcription Whisper ─────────────────────────────────────────────────────
def transcribe(audio_path, processor, model):
    audio, _ = librosa.load(audio_path, sr=16000)
    inputs = processor(audio, sampling_rate=16000, return_tensors="pt").input_features
    inputs = inputs.to("cuda" if torch.cuda.is_available() else "cpu")
    with torch.no_grad():
        ids = model.generate(inputs, language="ar")
    return processor.batch_decode(ids, skip_special_tokens=True)[0]


# ── RAG context ───────────────────────────────────────────────────────────────
def get_rag_context(question, embedder, index, questions, answers, k=3, threshold=8.0):
    q_emb = embedder.encode([question], convert_to_numpy=True)
    distances, indices = index.search(q_emb.astype(np.float32), k)
    contexts = []
    for dist, idx in zip(distances[0], indices[0]):
        if dist < threshold:
            contexts.append(f"Q: {questions[idx]}\nR: {answers[idx]}")
    return "\n\n".join(contexts)


# ── Inférence Atlas-Chat & LLaMA Darja ───────────────────────────────────────
def generate_response_darja(question, model, tokenizer, embedder, index, questions, answers):
    context = get_rag_context(question, embedder, index, questions, answers)
    content = f"""أنت مساعد طبي يتكلم الدارجة الجزائرية فقط.
{f"استخدم هاد المعلومات باش تجاوب:{chr(10)}{context}{chr(10)}{chr(10)}" if context else ""}
قواعد مهمة:
- أجب بجملة واحدة أو جملتين فقط
- استخدم كلام بسيط مفهوم
- ما تذكرش فحوصات معقدة إلا كان السؤال خطير
- ركز على النصيحة العملية البسيطة

السؤال: {question}
الجواب بالدارجة الجزائرية:"""

    messages = [{"role": "user", "content": content}]
    prompt   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs   = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    inputs   = {k: v.to("cuda" if torch.cuda.is_available() else "cpu") for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens     = 60,
            temperature        = 0.2,
            do_sample          = True,
            pad_token_id       = tokenizer.eos_token_id,
            repetition_penalty = 1.5,
        )
    input_len = inputs["input_ids"].shape[1]
    return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()


# ── Inférence LLaMA MAQA — identique au notebook ─────────────────────────────
MAQA_SYSTEM_PROMPT = (
    "أنت مساعد طبي متخصص. أجب على الأسئلة الطبية "
    "بدقة وباللغة العربية الفصحى."
)

def generate_response_maqa(question, model, tokenizer):
    """
    Identique au notebook :
    - system prompt MSA
    - max_new_tokens=200, temperature=0.7, top_p=0.9, repetition_penalty=1.1
    - extraction via .split("assistant")[-1].strip()
    - fp16 natif sur GPU G4 (94GB VRAM) = précision maximale
    """
    messages = [
        {"role": "system", "content": MAQA_SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    inputs = {k: v.to("cuda" if torch.cuda.is_available() else "cpu") for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens     = 200,
            temperature        = 0.7,
            top_p              = 0.9,
            repetition_penalty = 1.1,
            do_sample          = True,
            pad_token_id       = tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded.split("assistant")[-1].strip()


# ── TTS sur CPU ───────────────────────────────────────────────────────────────
def synthesize_tts(text, output_dir, unique_name):
    os.makedirs(output_dir, exist_ok=True)

    tmp_out    = f"{output_dir}/infer_cli_example.wav"
    final_path = f"{output_dir}/{unique_name}.wav"

    for f in [tmp_out, final_path]:
        if os.path.exists(f):
            os.remove(f)

    subprocess.run(
        ['habibi-tts_infer-cli',
         '--ref_audio', REF_AUDIO,
         '--ref_text',  REF_TEXT,
         '--gen_text',  text,
         '--dialect',   'ALG',
         '--output_dir', output_dir,
         '--device',    'cpu'],
        capture_output=True, text=True
    )

    deadline = time.time() + 120
    while time.time() < deadline:
        if os.path.exists(tmp_out) and os.path.getsize(tmp_out) > 0:
            os.rename(tmp_out, final_path)
            return final_path
        time.sleep(1)

    return final_path


# ── Chargement des modèles ────────────────────────────────────────────────────
with st.spinner("⏳ جاري تحميل النماذج..."):
    (w_proc, w_mdl,
     embedder, index, questions, answers,
     atlas_tok, atlas_mdl,
     llama_darja_tok, llama_darja_mdl,
     llama_maqa_tok, llama_maqa_mdl) = load_all_models()
st.success("✅ النماذج جاهزة!")


# ── UI ────────────────────────────────────────────────────────────────────────
col1, col2 = st.columns([2, 1])

with col1:
    st.markdown("### 💬 المحادثة")

    chat_html = '<div class="chat-container">'
    if not st.session_state.history:
        chat_html += '<p style="text-align:center;color:#999;margin-top:150px;">اسأل سؤالك الطبي بالدارجة الجزائرية 🎤</p>'
    for msg in st.session_state.history:
        chat_html += '<div style="text-align:right;font-size:0.75rem;opacity:0.7;margin-top:10px;">👤 أنت</div>'
        chat_html += f'<div class="msg-user">{msg["question"]}</div>'

        chat_html += '<div style="font-size:0.75rem;opacity:0.7;">🤖 Atlas-Chat (Zero-Shot + RAG)</div>'
        chat_html += f'<div class="msg-atlas">{msg["atlas"]}</div>'

        chat_html += '<div style="font-size:0.75rem;opacity:0.7;">🟢 LLaMA Darja (Fine-Tuné + RAG)</div>'
        chat_html += f'<div class="msg-llama">{msg["llama"]}</div>'

        chat_html += '<div style="font-size:0.75rem;opacity:0.7;">🟣 LLaMA MAQA (Fine-Tuné MSA)</div>'
        chat_html += f'<div class="msg-maqa">{msg["maqa"]}</div>'

        chat_html += '<hr style="border:none;border-top:1px solid #eee;margin:12px 0">'
    chat_html += '</div>'
    st.markdown(chat_html, unsafe_allow_html=True)

    if st.session_state.transcription:
        st.markdown(f"""
        <div class="last-question-box">
            📝 <b>آخر سؤال مفرَّغ:</b> {st.session_state.transcription}
        </div>
        """, unsafe_allow_html=True)

with col2:
    st.markdown("### 🎤 تسجيل السؤال")

    audio_input = st.audio_input("🎤 سجل سؤالك هنا")
    transcription_placeholder = st.empty()

    if audio_input is not None:
        audio_bytes = audio_input.read()
        st.audio(audio_bytes, format='audio/wav')

        if st.button("🚀 ابدأ التحليل", type="primary", use_container_width=True):

            with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
                tmp.write(audio_bytes)
                tmp_path = tmp.name

            wav_path = tmp_path.replace(".wav", "_16k.wav")
            subprocess.run(
                ['ffmpeg', '-i', tmp_path, '-ar', '16000', '-ac', '1', wav_path, '-y'],
                capture_output=True
            )

            with st.spinner("🎤 جاري التفريغ..."):
                question = transcribe(wav_path, w_proc, w_mdl)
                st.session_state.transcription = question

            transcription_placeholder.markdown(f"""
            <div class="transcription-box">
                🎤 <b>سؤالك:</b> {question}
            </div>
            """, unsafe_allow_html=True)

            with st.spinner("🤖 Atlas-Chat يجاوب..."):
                response_atlas = generate_response_darja(
                    question, atlas_mdl, atlas_tok, embedder, index, questions, answers
                )

            with st.spinner("🟢 LLaMA Darja يجاوب..."):
                response_llama = generate_response_darja(
                    question, llama_darja_mdl, llama_darja_tok, embedder, index, questions, answers
                )

            with st.spinner("🟣 LLaMA MAQA يجاوب..."):
                response_maqa = generate_response_maqa(
                    question, llama_maqa_mdl, llama_maqa_tok
                )

            with st.spinner("🔊 Atlas-Chat — توليد صوتي..."):
                atlas_audio = synthesize_tts(response_atlas, "./outputs_atlas", "atlas_response")

            with st.spinner("🔊 LLaMA Darja — توليد صوتي..."):
                llama_audio = synthesize_tts(response_llama, "./outputs_llama", "llama_response")

            with st.spinner("🔊 LLaMA MAQA — توليد صوتي..."):
                maqa_audio  = synthesize_tts(response_maqa,  "./outputs_maqa",  "maqa_response")

            st.session_state.history.append({
                "question"   : question,
                "atlas"      : response_atlas,
                "llama"      : response_llama,
                "maqa"       : response_maqa,
                "atlas_audio": atlas_audio,
                "llama_audio": llama_audio,
                "maqa_audio" : maqa_audio,
            })
            st.rerun()

    if st.session_state.history:
        last = st.session_state.history[-1]
        st.markdown("### 🔊 الردود الصوتية")

        st.markdown("**🤖 Atlas-Chat:**")
        if os.path.exists(last["atlas_audio"]):
            st.audio(last["atlas_audio"])
        else:
            st.warning("Atlas-Chat: fichier audio non trouvé")

        st.markdown("**🟢 LLaMA Darja:**")
        if os.path.exists(last["llama_audio"]):
            st.audio(last["llama_audio"])
        else:
            st.warning("LLaMA Darja: fichier audio non trouvé")

        st.markdown("**🟣 LLaMA MAQA:**")
        if os.path.exists(last["maqa_audio"]):
            st.audio(last["maqa_audio"])
        else:
            st.warning("LLaMA MAQA: fichier audio non trouvé")

        st.markdown("---")
        if st.button("🗑️ مسح المحادثة", use_container_width=True):
            st.session_state.history = []
            st.session_state.transcription = ""
            st.rerun()


In [ ]:
!pkill -f streamlit

from pyngrok import ngrok
ngrok.kill()
import subprocess, time

ngrok.set_auth_token("3EXbAhX6eMVMPfO6Og7iliM66Jj_67BvD1QbUD77p1Doa64WK")  # ← ton token depuis dashboard.ngrok.com

proc = subprocess.Popen(["streamlit", "run", "app.py",
                         "--server.port", "8501",
                         "--server.headless", "true"])
time.sleep(8)

public_url = ngrok.connect(8501)
print("🌐 URL :", public_url)

In [ ]:
!pip install transformers==4.51.3 unsloth --upgrade -q

In [ ]:
!pip install "unsloth_zoo>=2026.2.1" --upgrade -q
!pip install "unsloth @ git+https://github.com/unslothai/unsloth.git" --upgrade -q

In [ ]:
with open("app.py", "r") as f:
    content = f.read()

old = '''    llama_maqa_mdl = AutoModelForCausalLM.from_pretrained(
        "abdeldjalil-smaili/llama3-maqa-medical",
        torch_dtype=torch.float16,
        device_map="auto",
        token=HF_TOKEN
    )'''

new = '''    llama_maqa_mdl = AutoModelForCausalLM.from_pretrained(
        "abdeldjalil-smaili/llama3-maqa-medical",
        torch_dtype=torch.bfloat16,
        device_map="auto",
        token=HF_TOKEN
    )'''

content = content.replace(old, new)
with open("app.py", "w") as f:
    f.write(content)

print("✓ Corrigé — bfloat16 au lieu de float16")

In [ ]:
!pkill -f streamlit

from pyngrok import ngrok
ngrok.kill()
import subprocess, time

ngrok.set_auth_token("3EXbAhX6eMVMPfO6Og7iliM66Jj_67BvD1QbUD77p1Doa64WK")

proc = subprocess.Popen(["streamlit", "run", "app.py",
                         "--server.port", "8501",
                         "--server.headless", "true"])
time.sleep(8)

public_url = ngrok.connect(8501)
print("🌐 URL :", public_url)

In [ ]:
import subprocess, os, time

os.makedirs("./test_audio", exist_ok=True)

start = time.time()
result = subprocess.run(
    ['habibi-tts_infer-cli',
     '--ref_audio', 'ref_voice.wav',
     '--ref_text', 'نيفي راهو يسيل',
     '--gen_text', 'التحكم بالضغط المرتفع يتطلب إتباع النصائح التالية',
     '--dialect', 'ALG',
     '--output_dir', './test_audio',
     '--device', 'cpu'],
    capture_output=True, text=True
)
print(f"Temps : {time.time()-start:.1f}s")
print("returncode:", result.returncode)
print("STDERR:", result.stderr[-500:])

for root, dirs, files in os.walk("./test_audio"):
    for f in files:
        full = os.path.join(root, f)
        print(full, "—", os.path.getsize(full), "bytes")

In [ ]:
!pip uninstall torchaudio -y -q
!pip install torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cpu -q

In [ ]:
import subprocess, os, time

os.makedirs("./test_audio2", exist_ok=True)
start = time.time()
result = subprocess.run(
    ['habibi-tts_infer-cli',
     '--ref_audio', 'ref_voice.wav',
     '--ref_text', 'نيفي راهو يسيل',
     '--gen_text', 'مرحبا',
     '--dialect', 'ALG',
     '--output_dir', './test_audio2',
     '--device', 'cpu'],
    capture_output=True, text=True
)
print(f"Temps : {time.time()-start:.1f}s")
print("returncode:", result.returncode)
print("STDERR:", result.stderr[-300:])

for root, dirs, files in os.walk("./test_audio2"):
    for f in files:
        full = os.path.join(root, f)
        print(full, "—", os.path.getsize(full), "bytes")

In [ ]:
!pip uninstall torchaudio torchvision -y -q
!pip install torchaudio==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cpu -q

In [ ]:
import subprocess, os, time

os.makedirs("./test_audio3", exist_ok=True)
start = time.time()
result = subprocess.run(
    ['habibi-tts_infer-cli',
     '--ref_audio', 'ref_voice.wav',
     '--ref_text', 'نيفي راهو يسيل',
     '--gen_text', 'مرحبا',
     '--dialect', 'ALG',
     '--output_dir', './test_audio3',
     '--device', 'cpu'],
    capture_output=True, text=True
)
print(f"Temps : {time.time()-start:.1f}s")
print("returncode:", result.returncode)
print("STDERR:", result.stderr[-300:])

for root, dirs, files in os.walk("./test_audio3"):
    for f in files:
        full = os.path.join(root, f)
        print(full, "—", os.path.getsize(full), "bytes")

In [ ]:
!pip uninstall torchaudio torchvision -y -q
!pip install torchaudio torchvision --index-url https://download.pytorch.org/whl/cu128 -q

In [ ]:
import subprocess, os, time

os.makedirs("./test_audio4", exist_ok=True)
start = time.time()
result = subprocess.run(
    ['habibi-tts_infer-cli',
     '--ref_audio', 'ref_voice.wav',
     '--ref_text', 'نيفي راهو يسيل',
     '--gen_text', 'مرحبا',
     '--dialect', 'ALG',
     '--output_dir', './test_audio4',
     '--device', 'cpu'],
    capture_output=True, text=True
)
print(f"Temps : {time.time()-start:.1f}s")
print("returncode:", result.returncode)
print("STDERR:", result.stderr[-500:])

for root, dirs, files in os.walk("./test_audio4"):
    for f in files:
        full = os.path.join(root, f)
        print(full, "—", os.path.getsize(full), "bytes")

In [ ]:
with open("app.py", "r") as f:
    content = f.read()

content = content.replace(
    'os.system("pip uninstall torchcodec -y -q")\n',
    ''
)

with open("app.py", "w") as f:
    f.write(content)

print("✓ Ligne torchcodec supprimée")

In [ ]:
with open("app.py", "r") as f:
    content = f.read()

content = content.replace(
    "         '--device',    'cpu'],",
    "         '--device',    'cuda'],"
)

with open("app.py", "w") as f:
    f.write(content)

print("✓ CPU → CUDA pour TTS")

In [ ]:
!pip install torchcodec --index-url https://download.pytorch.org/whl/cu128 -q

In [ ]:
!pkill -f streamlit
from pyngrok import ngrok
ngrok.kill()
import subprocess, time

ngrok.set_auth_token("3EXbAhX6eMVMPfO6Og7iliM66Jj_67BvD1QbUD77p1Doa64WK")
proc = subprocess.Popen(["streamlit", "run", "app.py",
                         "--server.port", "8501",
                         "--server.headless", "true"])
time.sleep(8)
public_url = ngrok.connect(8501)
print("🌐 URL :", public_url)

In [ ]:
import subprocess, os, time

os.makedirs("./test_final", exist_ok=True)
start = time.time()
result = subprocess.run(
    ['habibi-tts_infer-cli',
     '--ref_audio', 'ref_voice.wav',
     '--ref_text', 'نيفي راهو يسيل',
     '--gen_text', 'مرحبا كيف حالك',
     '--dialect', 'ALG',
     '--output_dir', './test_final',
     '--device', 'cuda'],
    capture_output=True, text=True
)
print(f"Temps : {time.time()-start:.1f}s")
print("returncode:", result.returncode)
print("STDERR:", result.stderr[-300:])

for root, dirs, files in os.walk("./test_final"):
    for f in files:
        full = os.path.join(root, f)
        print(full, "—", os.path.getsize(full), "bytes")

In [ ]:
!pip uninstall torchaudio torchvision -y -q
!pip install torchaudio torchvision --index-url https://download.pytorch.org/whl/cu128 -q
!pip install torchcodec --index-url https://download.pytorch.org/whl/cu128 -q

In [ ]:
import subprocess, os, time

os.makedirs("./test_final2", exist_ok=True)
result = subprocess.run(
    ['habibi-tts_infer-cli',
     '--ref_audio', 'ref_voice.wav',
     '--ref_text', 'نيفي راهو يسيل',
     '--gen_text', 'مرحبا',
     '--dialect', 'ALG',
     '--output_dir', './test_final2',
     '--device', 'cuda'],
    capture_output=True, text=True
)
print("returncode:", result.returncode)
print("STDERR:", result.stderr[-300:])
for root, dirs, files in os.walk("./test_final2"):
    for f in files:
        full = os.path.join(root, f)
        print(full, "—", os.path.getsize(full), "bytes")

In [ ]:
with open("app.py", "r") as f:
    content = f.read()

old = '''%%writefile app.py
import os'''

new = '''%%writefile app.py
import os
os.system("pip uninstall torchaudio torchvision -y -q")
os.system("pip install torchaudio torchvision --index-url https://download.pytorch.org/whl/cu128 -q")
os.system("pip install torchcodec --index-url https://download.pytorch.org/whl/cu128 -q")'''

content = content.replace(old, new)
with open("app.py", "w") as f:
    f.write(content)

print("✓ Installations TTS ajoutées au début de app.py")

In [ ]:
!pkill -f streamlit
from pyngrok import ngrok
ngrok.kill()
import subprocess, time

ngrok.set_auth_token("3EXbAhX6eMVMPfO6Og7iliM66Jj_67BvD1QbUD77p1Doa64WK")
proc = subprocess.Popen(["streamlit", "run", "app.py",
                         "--server.port", "8501",
                         "--server.headless", "true"])
time.sleep(30)  # attendre les installations pip au démarrage

public_url = ngrok.connect(8501)
print("🌐 URL :", public_url)

In [ ]:
with open("app.py", "r") as f:
    content = f.read()

# 1. Remplacer la ligne torchcodec par les bonnes installations
content = content.replace(
    'os.system("pip uninstall torchcodec -y -q")',
    '''os.system("pip uninstall torchaudio torchvision -y -q")
os.system("pip install torchaudio torchvision --index-url https://download.pytorch.org/whl/cu128 -q")
os.system("pip install torchcodec --index-url https://download.pytorch.org/whl/cu128 -q")'''
)

with open("app.py", "w") as f:
    f.write(content)

# Vérifier
with open("app.py") as f:
    print(f.read()[:400])

In [ ]:
!pkill -f streamlit
from pyngrok import ngrok
ngrok.kill()
import subprocess, time

ngrok.set_auth_token("3EXbAhX6eMVMPfO6Og7iliM66Jj_67BvD1QbUD77p1Doa64WK")

proc = subprocess.Popen(["streamlit", "run", "app.py",
                         "--server.port", "8501",
                         "--server.headless", "true"])

print("⏳ Attente des pip install...")
time.sleep(120)  # 2 minutes pour les installations

public_url = ngrok.connect(8501)
print("🌐 URL :", public_url)

In [ ]:
import subprocess

def run(cmd):
    subprocess.run(cmd, shell=True, check=False)

run("pip uninstall torchaudio torchvision torchcodec -y -q")
run("pip install torchaudio torchvision --index-url https://download.pytorch.org/whl/cu128 -q")
run("pip install torchcodec --index-url https://download.pytorch.org/whl/cu128 -q")
run("pip install streamlit transformers==4.51.3 sentence-transformers faiss-cpu datasets huggingface_hub librosa numpy bitsandbytes accelerate pyngrok habibi-tts -q")
run("apt-get install -y ffmpeg -q")

print("✅ Installation terminée — REDÉMARRE LE RUNTIME MAINTENANT")
print("Runtime → Restart runtime (ou Ctrl+M .)")

In [ ]:
import subprocess, os

os.makedirs("./test_tts_check", exist_ok=True)
result = subprocess.run(
    ['habibi-tts_infer-cli',
     '--ref_audio', 'ref_voice.wav',
     '--ref_text', 'نيفي راهو يسيل',
     '--gen_text', 'مرحبا',
     '--dialect', 'ALG',
     '--output_dir', './test_tts_check',
     '--device', 'cuda'],
    capture_output=True, text=True
)
print("returncode:", result.returncode)
if result.returncode == 0:
    print("✅ TTS OK — prêt à lancer l'app")
else:
    print("❌", result.stderr[-300:])

In [ ]:
import torch
print("torch:", torch.__version__)

import subprocess
r = subprocess.run(['pip', 'show', 'torchaudio'], capture_output=True, text=True)
print(r.stdout)

In [ ]:
import subprocess

subprocess.run("pip uninstall torchaudio torchvision torchcodec -y -q", shell=True)
subprocess.run("pip install torchaudio==2.8.0 torchvision==0.23.0 --index-url https://download.pytorch.org/whl/cu128 -q", shell=True)
subprocess.run("pip install torchcodec --index-url https://download.pytorch.org/whl/cu128 -q", shell=True)

print("✅ Done — redémarre le runtime maintenant")

In [ ]:
import subprocess, os

os.makedirs("./test_tts2", exist_ok=True)
result = subprocess.run(
    ['habibi-tts_infer-cli',
     '--ref_audio', 'ref_voice.wav',
     '--ref_text', 'نيفي راهو يسيل',
     '--gen_text', 'مرحبا',
     '--dialect', 'ALG',
     '--output_dir', './test_tts2',
     '--device', 'cuda'],
    capture_output=True, text=True
)
print("returncode:", result.returncode)
print("STDERR:", result.stderr[-300:])
for root, dirs, files in os.walk("./test_tts2"):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
import os, subprocess

# Convertir new_test.mp4 → ref_voice.wav (tu l'as déjà uploadé avant)
if os.path.exists("new_test.mp4"):
    subprocess.run("ffmpeg -i new_test.mp4 -ar 22050 -ac 1 ref_voice.wav -y", shell=True)
    print("✅ ref_voice.wav créé depuis new_test.mp4")
else:
    # Si new_test.mp4 n'est plus là, uploader manuellement
    from google.colab import files
    print("Upload ton fichier audio de référence (wav ou mp4)...")
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    subprocess.run(f"ffmpeg -i '{fname}' -ar 22050 -ac 1 ref_voice.wav -y", shell=True)
    print("✅ ref_voice.wav créé")

print("Taille:", os.path.getsize("ref_voice.wav"), "bytes")

In [ ]:
result = subprocess.run(
    ['habibi-tts_infer-cli',
     '--ref_audio', 'ref_voice.wav',
     '--ref_text', 'نيفي راهو يسيل',
     '--gen_text', 'مرحبا',
     '--dialect', 'ALG',
     '--output_dir', './test_tts2',
     '--device', 'cuda'],
    capture_output=True, text=True
)
print("returncode:", result.returncode)
print("STDERR:", result.stderr[-200:])

In [ ]:
import subprocess, os

def run(cmd):
    subprocess.run(cmd, shell=True, check=False)

# ── 1. Installations ──────────────────────────────────────────────
run("pip uninstall torchaudio torchvision torchcodec -y -q")
run("pip install torchaudio==2.8.0 torchvision==0.23.0 --index-url https://download.pytorch.org/whl/cu128 -q")
run("pip install torchcodec --index-url https://download.pytorch.org/whl/cu128 -q")
run("pip install streamlit transformers==4.51.3 sentence-transformers faiss-cpu datasets huggingface_hub librosa numpy bitsandbytes accelerate pyngrok habibi-tts -q")
run("apt-get install -y ffmpeg -q")
print("✅ Installations terminées")

# ── 2. ref_voice.wav ─────────────────────────────────────────────
if os.path.exists("new_test.mp4"):
    run("ffmpeg -i new_test.mp4 -ar 22050 -ac 1 ref_voice.wav -y")
    print("✅ ref_voice.wav créé depuis new_test.mp4")
else:
    from google.colab import files
    print("📁 Upload ton fichier audio de référence...")
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    run(f"ffmpeg -i '{fname}' -ar 22050 -ac 1 ref_voice.wav -y")
    print("✅ ref_voice.wav créé depuis", fname)

print("Taille ref_voice.wav :", os.path.getsize("ref_voice.wav"), "bytes")
print("\n⚠️  REDÉMARRE LE RUNTIME MAINTENANT (Runtime → Restart session)")

In [ ]:
import subprocess, os

def run(cmd):
    subprocess.run(cmd, shell=True, check=False)

# ── 1. Installations ──────────────────────────────────────────────
run("pip uninstall torchaudio torchvision torchcodec -y -q")
run("pip install torchaudio==2.8.0 torchvision==0.23.0 --index-url https://download.pytorch.org/whl/cu128 -q")
run("pip install streamlit transformers==4.51.3 sentence-transformers faiss-cpu datasets huggingface_hub librosa numpy bitsandbytes accelerate pyngrok habibi-tts -q")
run("apt-get install -y ffmpeg -q")

# ── 2. Désinstaller torchcodec APRÈS tout (sentence-transformers le réinstalle) ──
run("pip uninstall torchcodec -y -q")
print("✅ Installations terminées — torchcodec supprimé")

# ── 3. ref_voice.wav ─────────────────────────────────────────────
if os.path.exists("ref_voice.wav"):
    print("✅ ref_voice.wav déjà présent —", os.path.getsize("ref_voice.wav"), "bytes")
elif os.path.exists("new_test.mp4"):
    run("ffmpeg -i new_test.mp4 -ar 22050 -ac 1 ref_voice.wav -y")
    print("✅ ref_voice.wav créé depuis new_test.mp4")
else:
    from google.colab import files
    print("📁 Upload ton fichier audio...")
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    run(f"ffmpeg -i '{fname}' -ar 22050 -ac 1 ref_voice.wav -y")
    print("✅ ref_voice.wav créé")

print("Taille ref_voice.wav :", os.path.getsize("ref_voice.wav"), "bytes")
print("\n⚠️  REDÉMARRE LE RUNTIME (Runtime → Restart session)")

In [ ]:
import subprocess, os, time
from pyngrok import ngrok

# ── Vérification TTS ─────────────────────────────────────────────
os.makedirs("./test_tts_check", exist_ok=True)
result = subprocess.run(
    ['habibi-tts_infer-cli',
     '--ref_audio', 'ref_voice.wav',
     '--ref_text', 'نيفي راهو يسيل',
     '--gen_text', 'مرحبا',
     '--dialect', 'ALG',
     '--output_dir', './test_tts_check',
     '--device', 'cuda'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("✅ TTS OK")
else:
    print("❌ TTS échoue :", result.stderr[-200:])
    raise Exception("TTS ne fonctionne pas — ne pas continuer")

In [ ]:
%%writefile app.py
# colle tout le contenu du fichier ici
import os
os.system("pip uninstall torchcodec -y -q")
import streamlit as st
import torch
import subprocess
import numpy as np
import faiss
from transformers import (
    WhisperProcessor, WhisperForConditionalGeneration,
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
)
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
from huggingface_hub import login
import librosa
import tempfile
import time

from huggingface_hub import login login()
REF_AUDIO = "ref_voice.wav"
REF_TEXT  = "نيفي راهو يسيل"

st.set_page_config(page_title="المساعد الطبي الجزائري", page_icon="🏥", layout="wide")

st.markdown("""
<style>
    .header {
        background: linear-gradient(135deg, #1a6b8a, #0d4d6b);
        color: white; padding: 20px; border-radius: 12px;
        text-align: center; margin-bottom: 20px;
    }
    .header h1 { font-size: 2rem; margin: 0; }
    .header p  { font-size: 1rem; margin: 5px 0 0 0; opacity: 0.85; }
    .chat-container {
        background: white; border-radius: 12px; padding: 20px;
        min-height: 400px; max-height: 600px; overflow-y: auto;
        box-shadow: 0 2px 8px rgba(0,0,0,0.08); margin-bottom: 16px;
    }
    .msg-user {
        background: #1a6b8a; color: white; padding: 10px 16px;
        border-radius: 18px 18px 4px 18px; margin: 8px 0;
        max-width: 75%; margin-left: auto; text-align: right; direction: rtl;
    }
    .msg-atlas {
        background: #e8f4f8; color: #1a1a1a; padding: 10px 16px;
        border-radius: 18px 18px 18px 4px; margin: 8px 0;
        max-width: 75%; border-left: 4px solid #1a6b8a; direction: rtl;
    }
    .msg-llama {
        background: #e8f8ee; color: #1a1a1a; padding: 10px 16px;
        border-radius: 18px 18px 18px 4px; margin: 8px 0;
        max-width: 75%; border-left: 4px solid #2ecc71; direction: rtl;
    }
    .transcription-box {
        background: #fff8e1; border: 1px solid #ffc107;
        border-radius: 8px; padding: 12px; margin: 10px 0;
        direction: rtl; font-size: 1.1rem;
    }
    .last-question-box {
        background: #e3f2fd; border: 1px solid #1a6b8a;
        border-radius: 8px; padding: 12px; margin: 10px 0;
        direction: rtl; font-size: 1.1rem;
    }
</style>
""", unsafe_allow_html=True)

st.markdown("""
<div class="header">
    <h1>🏥 المساعد الطبي الجزائري</h1>
    <p>نظام ذكاء اصطناعي طبي بالدارجة الجزائرية</p>
</div>
""", unsafe_allow_html=True)

if "history" not in st.session_state:
    st.session_state.history = []
if "transcription" not in st.session_state:
    st.session_state.transcription = ""


@st.cache_resource
def load_all_models():
    login(token=HF_TOKEN)

    # ── Whisper ───────────────────────────────────────────────────────────────
    whisper_processor = WhisperProcessor.from_pretrained("openai/whisper-small")
    whisper_model = WhisperForConditionalGeneration.from_pretrained(
        "abdeldjalil-smaili/whisper-algerian-medical-final",
        subfolder="checkpoint-1000", token=HF_TOKEN
    ).to("cuda" if torch.cuda.is_available() else "cpu")
    whisper_model.eval()

    # ── RAG (FAISS) ───────────────────────────────────────────────────────────
    embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
    dataset  = load_dataset("abdeldjalil-smaili/algerian-medical-llm-v3", token=HF_TOKEN, split="train")

    def get_q(s):
        try:    return s['text'].split("### Input:")[1].split("### Response:")[0].strip()
        except: return s['text'][:200]

    def get_a(s):
        try:    return s['text'].split("### Response:")[1].strip()
        except: return ""

    questions  = [get_q(s) for s in dataset]
    answers    = [get_a(s) for s in dataset]
    embeddings = embedder.encode(questions, batch_size=256,
                                 show_progress_bar=False, convert_to_numpy=True)
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings.astype(np.float32))

    # ── 4-bit config ──────────────────────────────────────────────────────────
    bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )

    # ── Modèle 1 : Atlas-Chat 9B (zero-shot, 4-bit) ───────────────────────────
    atlas_tok = AutoTokenizer.from_pretrained("MBZUAI-Paris/Atlas-Chat-9B", token=HF_TOKEN)
    atlas_mdl = AutoModelForCausalLM.from_pretrained(
        "MBZUAI-Paris/Atlas-Chat-9B", quantization_config=bnb,
        device_map="auto", token=HF_TOKEN
    )
    atlas_mdl.eval()

    # ── Modèle 2 : LLaMA Darja fine-tuné (4-bit) ─────────────────────────────
    llama_darja_tok = AutoTokenizer.from_pretrained(
        "meta-llama/Llama-3.2-3B-Instruct", token=HF_TOKEN)
    llama_darja_mdl = AutoModelForCausalLM.from_pretrained(
        "abdeldjalil-smaili/llama3b-algerian-medical", quantization_config=bnb,
        device_map="auto", token=HF_TOKEN
    )
    llama_darja_mdl.eval()

    return (whisper_processor, whisper_model,
            embedder, index, questions, answers,
            atlas_tok, atlas_mdl,
            llama_darja_tok, llama_darja_mdl)


# ── Transcription Whisper ─────────────────────────────────────────────────────
def transcribe(audio_path, processor, model):
    audio, _ = librosa.load(audio_path, sr=16000)
    inputs = processor(audio, sampling_rate=16000, return_tensors="pt").input_features
    inputs = inputs.to("cuda" if torch.cuda.is_available() else "cpu")
    with torch.no_grad():
        ids = model.generate(inputs, language="ar")
    return processor.batch_decode(ids, skip_special_tokens=True)[0]


# ── RAG context ───────────────────────────────────────────────────────────────
def get_rag_context(question, embedder, index, questions, answers, k=3, threshold=8.0):
    q_emb = embedder.encode([question], convert_to_numpy=True)
    distances, indices = index.search(q_emb.astype(np.float32), k)
    contexts = []
    for dist, idx in zip(distances[0], indices[0]):
        if dist < threshold:
            contexts.append(f"Q: {questions[idx]}\nR: {answers[idx]}")
    return "\n\n".join(contexts)


# ── Inférence Atlas-Chat & LLaMA Darja ───────────────────────────────────────
def generate_response_darja(question, model, tokenizer, embedder, index, questions, answers):
    context = get_rag_context(question, embedder, index, questions, answers)
    content = f"""أنت مساعد طبي يتكلم الدارجة الجزائرية فقط.
{f"استخدم هاد المعلومات باش تجاوب:{chr(10)}{context}{chr(10)}{chr(10)}" if context else ""}
قواعد مهمة:
- أجب بجملة واحدة أو جملتين فقط
- استخدم كلام بسيط مفهوم
- ما تذكرش فحوصات معقدة إلا كان السؤال خطير
- ركز على النصيحة العملية البسيطة

السؤال: {question}
الجواب بالدارجة الجزائرية:"""

    messages = [{"role": "user", "content": content}]
    prompt   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs   = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    inputs   = {k: v.to("cuda" if torch.cuda.is_available() else "cpu") for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens     = 60,
            temperature        = 0.2,
            do_sample          = True,
            pad_token_id       = tokenizer.eos_token_id,
            repetition_penalty = 1.5,
        )
    input_len = inputs["input_ids"].shape[1]
    return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()


# ── TTS ───────────────────────────────────────────────────────────────────────
def synthesize_tts(text, output_dir, unique_name):
    os.makedirs(output_dir, exist_ok=True)

    tmp_out    = f"{output_dir}/infer_cli_example.wav"
    final_path = f"{output_dir}/{unique_name}.wav"

    for f in [tmp_out, final_path]:
        if os.path.exists(f):
            os.remove(f)

    subprocess.run(
        ['habibi-tts_infer-cli',
         '--ref_audio', REF_AUDIO,
         '--ref_text',  REF_TEXT,
         '--gen_text',  text,
         '--dialect',   'ALG',
         '--output_dir', output_dir,
         '--device',    'cuda'],
        capture_output=True, text=True
    )

    deadline = time.time() + 120
    while time.time() < deadline:
        if os.path.exists(tmp_out) and os.path.getsize(tmp_out) > 0:
            os.rename(tmp_out, final_path)
            return final_path
        time.sleep(1)

    return final_path


# ── Chargement des modèles ────────────────────────────────────────────────────
with st.spinner("⏳ جاري تحميل النماذج..."):
    (w_proc, w_mdl,
     embedder, index, questions, answers,
     atlas_tok, atlas_mdl,
     llama_darja_tok, llama_darja_mdl) = load_all_models()
st.success("✅ النماذج جاهزة!")


# ── UI ────────────────────────────────────────────────────────────────────────
col1, col2 = st.columns([2, 1])

with col1:
    st.markdown("### 💬 المحادثة")

    chat_html = '<div class="chat-container">'
    if not st.session_state.history:
        chat_html += '<p style="text-align:center;color:#999;margin-top:150px;">اسأل سؤالك الطبي بالدارجة الجزائرية 🎤</p>'
    for msg in st.session_state.history:
        chat_html += '<div style="text-align:right;font-size:0.75rem;opacity:0.7;margin-top:10px;">👤 أنت</div>'
        chat_html += f'<div class="msg-user">{msg["question"]}</div>'

        chat_html += '<div style="font-size:0.75rem;opacity:0.7;">🤖 Atlas-Chat (Zero-Shot + RAG)</div>'
        chat_html += f'<div class="msg-atlas">{msg["atlas"]}</div>'

        chat_html += '<div style="font-size:0.75rem;opacity:0.7;">🟢 LLaMA Darja (Fine-Tuné + RAG)</div>'
        chat_html += f'<div class="msg-llama">{msg["llama"]}</div>'

        chat_html += '<hr style="border:none;border-top:1px solid #eee;margin:12px 0">'
    chat_html += '</div>'
    st.markdown(chat_html, unsafe_allow_html=True)

    if st.session_state.transcription:
        st.markdown(f"""
        <div class="last-question-box">
            📝 <b>آخر سؤال مفرَّغ:</b> {st.session_state.transcription}
        </div>
        """, unsafe_allow_html=True)

with col2:
    st.markdown("### 🎤 تسجيل السؤال")

    audio_input = st.audio_input("🎤 سجل سؤالك هنا")
    transcription_placeholder = st.empty()

    if audio_input is not None:
        audio_bytes = audio_input.read()
        st.audio(audio_bytes, format='audio/wav')

        if st.button("🚀 ابدأ التحليل", type="primary", use_container_width=True):

            with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
                tmp.write(audio_bytes)
                tmp_path = tmp.name

            wav_path = tmp_path.replace(".wav", "_16k.wav")
            subprocess.run(
                ['ffmpeg', '-i', tmp_path, '-ar', '16000', '-ac', '1', wav_path, '-y'],
                capture_output=True
            )

            with st.spinner("🎤 جاري التفريغ..."):
                question = transcribe(wav_path, w_proc, w_mdl)
                st.session_state.transcription = question

            transcription_placeholder.markdown(f"""
            <div class="transcription-box">
                🎤 <b>سؤالك:</b> {question}
            </div>
            """, unsafe_allow_html=True)

            with st.spinner("🤖 Atlas-Chat يجاوب..."):
                response_atlas = generate_response_darja(
                    question, atlas_mdl, atlas_tok, embedder, index, questions, answers
                )

            with st.spinner("🟢 LLaMA Darja يجاوب..."):
                response_llama = generate_response_darja(
                    question, llama_darja_mdl, llama_darja_tok, embedder, index, questions, answers
                )

            with st.spinner("🔊 Atlas-Chat — توليد صوتي..."):
                atlas_audio = synthesize_tts(response_atlas, "./outputs_atlas", "atlas_response")

            with st.spinner("🔊 LLaMA Darja — توليد صوتي..."):
                llama_audio = synthesize_tts(response_llama, "./outputs_llama", "llama_response")

            st.session_state.history.append({
                "question"   : question,
                "atlas"      : response_atlas,
                "llama"      : response_llama,
                "atlas_audio": atlas_audio,
                "llama_audio": llama_audio,
            })
            st.rerun()

    if st.session_state.history:
        last = st.session_state.history[-1]
        st.markdown("### 🔊 الردود الصوتية")

        st.markdown("**🤖 Atlas-Chat:**")
        if os.path.exists(last["atlas_audio"]):
            st.audio(last["atlas_audio"])
        else:
            st.warning("Atlas-Chat: fichier audio non trouvé")

        st.markdown("**🟢 LLaMA Darja:**")
        if os.path.exists(last["llama_audio"]):
            st.audio(last["llama_audio"])
        else:
            st.warning("LLaMA Darja: fichier audio non trouvé")

        st.markdown("---")
        if st.button("🗑️ مسح المحادثة", use_container_width=True):
            st.session_state.history = []
            st.session_state.transcription = ""
            st.rerun()

In [ ]:
import subprocess, time, os, threading

os.system("pkill -f streamlit")
os.system("pkill -f cloudflared")
time.sleep(3)

# Installer cloudflared
os.system("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared")
os.system("chmod +x /usr/local/bin/cloudflared")

# Lancer Streamlit
proc = subprocess.Popen(["streamlit", "run", "app.py",
                         "--server.port", "8501",
                         "--server.headless", "true"])
time.sleep(15)

# Lancer cloudflared et capturer l'URL
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)

# Attendre et extraire l'URL
url = None
for _ in range(30):
    line = tunnel_proc.stderr.readline().decode("utf-8", errors="ignore")
    if "trycloudflare.com" in line:
        url = line.strip().split()[-1]
        break
    time.sleep(1)

print("🌐 URL :", url)

In [ ]:
import subprocess, time

tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)

url = None
for _ in range(60):
    line = tunnel_proc.stderr.readline().decode("utf-8", errors="ignore")
    print(line.strip())  # afficher toutes les lignes pour voir l'URL complète
    if "trycloudflare.com" in line and "http" in line:
        # extraire l'URL complète
        import re
        match = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
        if match:
            url = match.group(0)
            break
    time.sleep(1)

print("\n🌐 URL complète :", url)

In [ ]:
import subprocess, time, os, re

os.system("pkill -f streamlit")
os.system("pkill -f cloudflared")
time.sleep(3)

proc = subprocess.Popen(["streamlit", "run", "app.py",
                         "--server.port", "8501",
                         "--server.headless", "true"])
time.sleep(15)

tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)

url = None
for _ in range(60):
    line = tunnel_proc.stderr.readline().decode("utf-8", errors="ignore")
    match = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
    if match:
        url = match.group(0)
        break
    time.sleep(1)

print("🌐 URL :", url)

In [ ]:
%%writefile app.py
# colle tout le contenu du fichier ici
import os
os.system("pip uninstall torchcodec -y -q")
import streamlit as st
import torch
import subprocess
import numpy as np
import faiss
from transformers import (
    WhisperProcessor, WhisperForConditionalGeneration,
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
)
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
from huggingface_hub import login
import librosa
import tempfile
import time

from huggingface_hub import login login()
REF_AUDIO = "ref_voice.wav"
REF_TEXT  = "نيفي راهو يسيل"

st.set_page_config(page_title="المساعد الطبي الجزائري", page_icon="🏥", layout="wide")

st.markdown("""
<style>
    .header {
        background: linear-gradient(135deg, #1a6b8a, #0d4d6b);
        color: white; padding: 20px; border-radius: 12px;
        text-align: center; margin-bottom: 20px;
    }
    .header h1 { font-size: 2rem; margin: 0; }
    .header p  { font-size: 1rem; margin: 5px 0 0 0; opacity: 0.85; }
    .chat-container {
        background: white; border-radius: 12px; padding: 20px;
        min-height: 400px; max-height: 600px; overflow-y: auto;
        box-shadow: 0 2px 8px rgba(0,0,0,0.08); margin-bottom: 16px;
    }
    .msg-user {
        background: #1a6b8a; color: white; padding: 10px 16px;
        border-radius: 18px 18px 4px 18px; margin: 8px 0;
        max-width: 75%; margin-left: auto; text-align: right; direction: rtl;
    }
    .msg-atlas {
        background: #e8f4f8; color: #1a1a1a; padding: 10px 16px;
        border-radius: 18px 18px 18px 4px; margin: 8px 0;
        max-width: 75%; border-left: 4px solid #1a6b8a; direction: rtl;
    }
    .msg-maqa {
        background: #f3e8ff; color: #1a1a1a; padding: 10px 16px;
        border-radius: 18px 18px 18px 4px; margin: 8px 0;
        max-width: 75%; border-left: 4px solid #8e44ad; direction: rtl;
    }
    .transcription-box {
        background: #fff8e1; border: 1px solid #ffc107;
        border-radius: 8px; padding: 12px; margin: 10px 0;
        direction: rtl; font-size: 1.1rem;
    }
    .last-question-box {
        background: #e3f2fd; border: 1px solid #1a6b8a;
        border-radius: 8px; padding: 12px; margin: 10px 0;
        direction: rtl; font-size: 1.1rem;
    }
</style>
""", unsafe_allow_html=True)

st.markdown("""
<div class="header">
    <h1>🏥 المساعد الطبي الجزائري</h1>
    <p>نظام ذكاء اصطناعي طبي بالدارجة الجزائرية</p>
</div>
""", unsafe_allow_html=True)

if "history" not in st.session_state:
    st.session_state.history = []
if "transcription" not in st.session_state:
    st.session_state.transcription = ""


@st.cache_resource
def load_all_models():
    login(token=HF_TOKEN)

    # ── Whisper ───────────────────────────────────────────────────────────────
    whisper_processor = WhisperProcessor.from_pretrained("openai/whisper-small")
    whisper_model = WhisperForConditionalGeneration.from_pretrained(
        "abdeldjalil-smaili/whisper-algerian-medical-final",
        subfolder="checkpoint-1000", token=HF_TOKEN
    ).to("cuda" if torch.cuda.is_available() else "cpu")
    whisper_model.eval()

    # ── RAG (FAISS) ───────────────────────────────────────────────────────────
    embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
    dataset  = load_dataset("abdeldjalil-smaili/algerian-medical-llm-v3", token=HF_TOKEN, split="train")

    def get_q(s):
        try:    return s['text'].split("### Input:")[1].split("### Response:")[0].strip()
        except: return s['text'][:200]

    def get_a(s):
        try:    return s['text'].split("### Response:")[1].strip()
        except: return ""

    questions  = [get_q(s) for s in dataset]
    answers    = [get_a(s) for s in dataset]
    embeddings = embedder.encode(questions, batch_size=256,
                                 show_progress_bar=False, convert_to_numpy=True)
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings.astype(np.float32))

    # ── 4-bit config pour Atlas-Chat ──────────────────────────────────────────
    bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )

    # ── Modèle 1 : Atlas-Chat 9B (zero-shot, 4-bit) ───────────────────────────
    atlas_tok = AutoTokenizer.from_pretrained("MBZUAI-Paris/Atlas-Chat-9B", token=HF_TOKEN)
    atlas_mdl = AutoModelForCausalLM.from_pretrained(
        "MBZUAI-Paris/Atlas-Chat-9B", quantization_config=bnb,
        device_map="auto", token=HF_TOKEN
    )
    atlas_mdl.eval()

    # ── Modèle 2 : LLaMA MAQA — bfloat16 natif ───────────────────────────────
    llama_maqa_tok = AutoTokenizer.from_pretrained(
        "meta-llama/Llama-3.2-3B-Instruct", token=HF_TOKEN
    )
    llama_maqa_tok.pad_token = llama_maqa_tok.eos_token
    llama_maqa_mdl = AutoModelForCausalLM.from_pretrained(
        "abdeldjalil-smaili/llama3-maqa-medical",
        torch_dtype=torch.bfloat16,
        device_map="auto",
        token=HF_TOKEN
    )
    llama_maqa_mdl.eval()

    return (whisper_processor, whisper_model,
            embedder, index, questions, answers,
            atlas_tok, atlas_mdl,
            llama_maqa_tok, llama_maqa_mdl)


# ── Transcription Whisper ─────────────────────────────────────────────────────
def transcribe(audio_path, processor, model):
    audio, _ = librosa.load(audio_path, sr=16000)
    inputs = processor(audio, sampling_rate=16000, return_tensors="pt").input_features
    inputs = inputs.to("cuda" if torch.cuda.is_available() else "cpu")
    with torch.no_grad():
        ids = model.generate(inputs, language="ar")
    return processor.batch_decode(ids, skip_special_tokens=True)[0]


# ── RAG context ───────────────────────────────────────────────────────────────
def get_rag_context(question, embedder, index, questions, answers, k=3, threshold=8.0):
    q_emb = embedder.encode([question], convert_to_numpy=True)
    distances, indices = index.search(q_emb.astype(np.float32), k)
    contexts = []
    for dist, idx in zip(distances[0], indices[0]):
        if dist < threshold:
            contexts.append(f"Q: {questions[idx]}\nR: {answers[idx]}")
    return "\n\n".join(contexts)


# ── Inférence Atlas-Chat (prompt Darja + RAG) ─────────────────────────────────
def generate_response_atlas(question, model, tokenizer, embedder, index, questions, answers):
    context = get_rag_context(question, embedder, index, questions, answers)
    content = f"""أنت مساعد طبي يتكلم الدارجة الجزائرية فقط.
{f"استخدم هاد المعلومات باش تجاوب:{chr(10)}{context}{chr(10)}{chr(10)}" if context else ""}
قواعد مهمة:
- أجب بجملة واحدة أو جملتين فقط
- استخدم كلام بسيط مفهوم
- ما تذكرش فحوصات معقدة إلا كان السؤال خطير
- ركز على النصيحة العملية البسيطة

السؤال: {question}
الجواب بالدارجة الجزائرية:"""

    messages = [{"role": "user", "content": content}]
    prompt   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs   = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    inputs   = {k: v.to("cuda" if torch.cuda.is_available() else "cpu") for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens     = 60,
            temperature        = 0.2,
            do_sample          = True,
            pad_token_id       = tokenizer.eos_token_id,
            repetition_penalty = 1.5,
        )
    input_len = inputs["input_ids"].shape[1]
    return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()


# ── Inférence LLaMA ─────────────────────────────────────────────────────
MAQA_SYSTEM_PROMPT = (
    "أنت مساعد طبي متخصص. أجب على الأسئلة الطبية "
    "بدقة وباللغة العربية الفصحى."
)

def generate_response_maqa(question, model, tokenizer):
    messages = [
        {"role": "system", "content": MAQA_SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    inputs = {k: v.to("cuda" if torch.cuda.is_available() else "cpu") for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens     = 200,
            temperature        = 0.7,
            top_p              = 0.9,
            repetition_penalty = 1.1,
            do_sample          = True,
            pad_token_id       = tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded.split("assistant")[-1].strip()


# ── TTS ───────────────────────────────────────────────────────────────────────
def synthesize_tts(text, output_dir, unique_name):
    os.makedirs(output_dir, exist_ok=True)

    tmp_out    = f"{output_dir}/infer_cli_example.wav"
    final_path = f"{output_dir}/{unique_name}.wav"

    for f in [tmp_out, final_path]:
        if os.path.exists(f):
            os.remove(f)

    os.system("pip uninstall torchcodec -y -q")

    subprocess.run(
        ['habibi-tts_infer-cli',
         '--ref_audio', REF_AUDIO,
         '--ref_text',  REF_TEXT,
         '--gen_text',  text,
         '--dialect',   'ALG',
         '--output_dir', output_dir,
         '--device',    'cuda'],
        capture_output=True, text=True
    )

    deadline = time.time() + 120
    while time.time() < deadline:
        if os.path.exists(tmp_out) and os.path.getsize(tmp_out) > 0:
            os.rename(tmp_out, final_path)
            return final_path
        time.sleep(1)

    return final_path


# ── Chargement des modèles ────────────────────────────────────────────────────
with st.spinner("⏳ جاري تحميل النماذج..."):
    (w_proc, w_mdl,
     embedder, index, questions, answers,
     atlas_tok, atlas_mdl,
     llama_maqa_tok, llama_maqa_mdl) = load_all_models()
st.success("✅ النماذج جاهزة!")


# ── UI ────────────────────────────────────────────────────────────────────────
col1, col2 = st.columns([2, 1])

with col1:
    st.markdown("### 💬 المحادثة")

    chat_html = '<div class="chat-container">'
    if not st.session_state.history:
        chat_html += '<p style="text-align:center;color:#999;margin-top:150px;">اسأل سؤالك الطبي بالدارجة الجزائرية 🎤</p>'
    for msg in st.session_state.history:
        chat_html += '<div style="text-align:right;font-size:0.75rem;opacity:0.7;margin-top:10px;">👤 أنت</div>'
        chat_html += f'<div class="msg-user">{msg["question"]}</div>'

        chat_html += '<div style="font-size:0.75rem;opacity:0.7;">🤖 Atlas-Chat (Zero-Shot + RAG)</div>'
        chat_html += f'<div class="msg-atlas">{msg["atlas"]}</div>'

        chat_html += '<div style="font-size:0.75rem;opacity:0.7;">🟢 LLaMA Dardja (Fine-Tuné)</div>'
        chat_html += f'<div class="msg-maqa">{msg["maqa"]}</div>'

        chat_html += '<hr style="border:none;border-top:1px solid #eee;margin:12px 0">'
    chat_html += '</div>'
    st.markdown(chat_html, unsafe_allow_html=True)

    if st.session_state.transcription:
        st.markdown(f"""
        <div class="last-question-box">
            📝 <b>آخر سؤال مفرَّغ:</b> {st.session_state.transcription}
        </div>
        """, unsafe_allow_html=True)

with col2:
    st.markdown("### 🎤 تسجيل السؤال")

    audio_input = st.audio_input("🎤 سجل سؤالك هنا")
    transcription_placeholder = st.empty()

    if audio_input is not None:
        audio_bytes = audio_input.read()
        st.audio(audio_bytes, format='audio/wav')

        if st.button("🚀 ابدأ التحليل", type="primary", use_container_width=True):

            with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
                tmp.write(audio_bytes)
                tmp_path = tmp.name

            wav_path = tmp_path.replace(".wav", "_16k.wav")
            subprocess.run(
                ['ffmpeg', '-i', tmp_path, '-ar', '16000', '-ac', '1', wav_path, '-y'],
                capture_output=True
            )

            with st.spinner("🎤 جاري التفريغ..."):
                question = transcribe(wav_path, w_proc, w_mdl)
                st.session_state.transcription = question

            transcription_placeholder.markdown(f"""
            <div class="transcription-box">
                🎤 <b>سؤالك:</b> {question}
            </div>
            """, unsafe_allow_html=True)

            with st.spinner("🤖 Atlas-Chat يجاوب..."):
                response_atlas = generate_response_atlas(
                    question, atlas_mdl, atlas_tok, embedder, index, questions, answers
                )

            with st.spinner("🟢 LLaMA Darja يجاوب..."):
                response_maqa = generate_response_maqa(
                    question, llama_maqa_mdl, llama_maqa_tok
                )

            with st.spinner("🔊 Atlas-Chat — توليد صوتي..."):
                atlas_audio = synthesize_tts(response_atlas, "./outputs_atlas", "atlas_response")

            with st.spinner("🔊 🟢 LLaMA Darja — توليد صوتي..."):
                maqa_audio = synthesize_tts(response_maqa, "./outputs_maqa", "maqa_response")

            st.session_state.history.append({
                "question"   : question,
                "atlas"      : response_atlas,
                "maqa"       : response_maqa,
                "atlas_audio": atlas_audio,
                "maqa_audio" : maqa_audio,
            })
            st.rerun()

    if st.session_state.history:
        last = st.session_state.history[-1]
        st.markdown("### 🔊 الردود الصوتية")

        st.markdown("**🤖 Atlas-Chat:**")
        if os.path.exists(last["atlas_audio"]):
            st.audio(last["atlas_audio"])
        else:
            st.warning("Atlas-Chat: fichier audio non trouvé")

        st.markdown("**🟢 LLaMA Darja :**")
        if os.path.exists(last["maqa_audio"]):
            st.audio(last["maqa_audio"])
        else:
            st.warning("LLaMA : fichier audio non trouvé")

        st.markdown("---")
        if st.button("🗑️ مسح المحادثة", use_container_width=True):
            st.session_state.history = []
            st.session_state.transcription = ""
            st.rerun()